# Evaluación Espacial de Conservación (EOO, AOO), Biodiversidad Local, Estructura Genética y Amenazas de Uso del Suelo (MapBiomas Chile)

**Asignatura:** Biodiversidad y Conservación Biológica (2026)  
**Institución:** Facultad de Ciencias Forestales, Universidad de Concepción (Chile)  
**Docente titular:** Dr. Rodrigo Hasbún Zaror | **Ayudantía y desarrollo:** Mauricio Aguirre-Díaz ([`magdzraw/byc-2026`](https://github.com/magdzraw/byc-2026))  
**Especies configuradas:** Alerce (*Fitzroya cupressoides*), Araucaria (*Araucaria araucana*), Lleuque (*Prumnopitys andina*), Queule (*Gomortega keule*) y Ruil (*Nothofagus alessandrii*).

---

## Marco Normativo y Metodológico

Este cuaderno implementa un flujo reproducible en Python para estimar parámetros geográficos del **Criterio B** de amenaza de extinción e índices locales de diversidad de flora vascular acompañante, alineado con:

1. **Reglamento de Clasificación de Especies Silvestres de Chile (RCE — D.S. N° 29/2011 MMA):** Adopta las categorías y criterios de la UICN y las *Directrices para Emplear los Criterios de la Lista Roja de la UICN a Nivel Regional y Nacional (v4.0)*. En el modo por defecto (`RCE_CHILE`), la evaluación se restringe a registros en **territorio chileno (`CL`)** recortados por el límite nacional de alta resolución (Paso 1 regional). Para especies compartidas con Argentina (*Alerce*, *Araucaria*, *Lleuque*), la evaluación formal requiere considerar en un Paso 2 el eventual *efecto de rescate* desde poblaciones vecinas.
2. **Directrices de la Lista Roja Global de la UICN (v16, 2024):** Disponible seleccionando `ESCALA_EVALUACION = 'UICN_GLOBAL'` (incorpora registros de Chile y Argentina, `CL` + `AR`, y calcula el Polígono Convexo Mínimo global sin exclusión de disyunciones).
3. **Planes RECOGE (MMA-FAO-GEF) y literatura nacional:** Para especies con distribución sumamente restringida y fragmentada en rodales pequeños (como *Gomortega keule* en el **Plan RECOGE del Queule, Mayo 2022, pp. 6–7**, que reporta AOO en grilla de $1 \times 1\text{ km}$, o *Nothofagus alessandrii* en el **Plan RECOGE del Ruil, D.S. N° 42/2020**), el cuaderno calcula **dos grillas de AOO claramente diferenciadas**:
   * **AOO Oficial UICN / RCE ($2 \times 2\text{ km} = 4\text{ km}^2$):** Escala obligatoria para contrastar con los umbrales del Criterio B2 (UICN Sec. 4.10), optimizada por desplazamiento de origen para obtener el mínimo número de celdas ocupadas.
   * **AOO de Escala Fina / Literatura ($1 \times 1\text{ km} = 1\text{ km}^2$):** Capa complementaria de micro-ocupación comparable con los Planes RECOGE (no se compara con umbrales B2 para evitar sesgo de escala).

### Fuentes de Datos Utilizadas
| Recurso | Origen / API | Uso en el análisis |
| :--- | :--- | :--- |
| **Localidades de referencia** | `Coordenadas_codigo.csv` (delimitado por `;`) | Sitios de muestreo genético/terreno (con `Codigo`), validación y centros de la malla hexagonal de biodiversidad. |
| **Ocurrencias de la especie focal** | **GBIF API v1** (`/v1/species/match` + `/v1/occurrence/search`) | Cálculo de EOO y AOO tras curaduría taxonómica, espacial y biogeográfica. |
| **Flora vascular acompañante** | **GBIF API v1** (`phylumKey = 7707728`, *Tracheophyta*) | Cálculo de riqueza ($S$), Shannon ($H'$), Pielou ($J'$), Gini-Simpson ($1-D$), Chao1 y Cobertura ($\hat{C}$). |
| **Límites nacionales, comunales y línea de costa** | **GADM v4.1** (`CHL_0`, `CHL_1`, `CHL_3`; respaldo: **Natural Earth 1:10m**) | Máscara terrestre precisa (archipiélagos, fiordos, Chiloé, regiones y comunas) para recorte RCE Chile / Global y cruce territorial. |
| **Estructura genética (Módulo 11 opcional)** | Resultados de **STRUCTURE** (`.zip` de `Results/`, archivo `*_f` o `.csv`) | Cálculo de Evanno ($\Delta K$), consenso entre réplicas tipo CLUMPP, figura compuesta A+B+C, tortas SVG en Folium, IDW opcional y capas `.gpkg` para QGIS. |
| **Uso del suelo e incendios (Módulo 12)** | **MapBiomas Chile** — Colección 2.0 Cobertura/Uso del Suelo (1999–2024) y Colección 1.0 Fuego (2013–2025) | Diagnóstico de amenazas por localidad y celda AOO sin GEE (trayectoria 1999–2024, transiciones de bosque nativo, Subcriterio B(b)(iii), incendios y vulnerabilidad por clúster genético). |

## 1. Instalación e Importación de Librerías

Instalamos las dependencias espaciales, estadísticas, ráster y de visualización requeridas en Google Colab (`geopandas`, `shapely`, `pyogrio`, `rasterio`, `openpyxl`, `folium`, `mapclassify`, `matplotlib`, `scipy`, `requests`).

In [ ]:
# Instalación de librerías espaciales y científicas (en Colab; si ya están instaladas, pip las omite)
!pip install geopandas requests folium pyogrio rasterio openpyxl matplotlib mapclassify shapely scipy --quiet

import os
import re
import io
import math
import base64
import zipfile
import warnings
import unicodedata
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
from shapely.geometry import Polygon, Point, box
from shapely.ops import unary_union
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.patheffects import withStroke
import folium
try:
    import pyproj
    _pdir = pyproj.datadir.get_data_dir()
    if _pdir and os.path.isdir(_pdir):
        os.environ["PROJ_DATA"] = _pdir
        os.environ["PROJ_LIB"] = _pdir
except Exception:
    pass
try:
    import rasterio
    from rasterio.mask import mask as rio_mask
    RASTERIO_DISPONIBLE = True
except ImportError:
    RASTERIO_DISPONIBLE = False

# Sesión HTTP con reintentos automáticos ante cortes transitorios de red/SSL (GBIF / GADM)
SESSION_HTTP = requests.Session()
_retry_cfg = Retry(total=4, backoff_factor=0.6, status_forcelist=[429, 500, 502, 503, 504])
SESSION_HTTP.mount("https://", HTTPAdapter(max_retries=_retry_cfg))
SESSION_HTTP.mount("http://", HTTPAdapter(max_retries=_retry_cfg))

warnings.filterwarnings("ignore", category=UserWarning, module="pyogrio")
print("Librerías cargadas correctamente.")

## 2. Panel de Configuración del Estudio

Selecciona la **especie** y la **escala de evaluación**:
* **`RCE_CHILE` (por defecto):** Utiliza exclusivamente ocurrencias en **Chile (`CL`)** y recorta el EOO a la superficie terrestre chilena de alta resolución (GADM 4.1), siguiendo el Paso 1 del RCE-MMA.
* **`UICN_GLOBAL`:** Incluye ocurrencias nativas de **Chile y Argentina (`CL`, `AR`)** para evaluar el rango global de especies transandinas (*Alerce*, *Araucaria*, *Lleuque*).

> **Nota sobre la escala de AOO y literatura chilena:**  
> * **Estándar UICN / RCE (Criterio B2):** Siempre **$2000\text{ m}$ ($2 \times 2\text{ km} = 4\text{ km}^2$)** para todas las especies (UICN v16, Sec. 4.10).  
> * **Escala fina de literatura (complementaria):** En *Queule* ($1000\text{ m}$, Plan RECOGE Queule MMA-FAO-GEF 2022, pp. 6–7: EOO $4.392{,}5\text{ km}^2$ y AOO $86\text{ km}^2$ en grilla de $1\text{ km}^2$) y *Ruil* ($1000\text{ m}$, fragmentos de 1–60 ha en Cordillera de la Costa del Maule, Plan RECOGE Ruil D.S. 42/2020) se genera una segunda grilla fina para análisis ecológico comparativo.

In [ ]:
#@title Parámetros principales del análisis { display-mode: "form" }
ESPECIE_SELECCIONADA = "Lleuque"  #@param ["Alerce", "Araucaria", "Lleuque", "Queule", "Ruil"]
ESCALA_EVALUACION = "RCE_CHILE"  #@param ["RCE_CHILE", "UICN_GLOBAL"]
CSV_UNIFICADO = "Coordenadas_codigo.csv"  #@param {type:"string"}
INCLUIR_SITIOS_CSV_EN_UICN = True  #@param {type:"boolean"}
INCERTIDUMBRE_MAXIMA_M = 2000  #@param {type:"integer"}
PASOS_OPTIMIZACION_AOO = 10  #@param {type:"integer"}
RADIO_HEXAGONO_M = 2500  #@param {type:"integer"}
BUFFER_LOCAL_M = 7500  #@param {type:"integer"}
N_MIN_ESFUERZO = 15  #@param {type:"integer"}
MAX_REGISTROS_FLORA_PARCHE = 6000  #@param {type:"integer"}

# Países consultados en GBIF según escala normativa
PAISES_CONSULTA = ["CL"] if ESCALA_EVALUACION == "RCE_CHILE" else ["CL", "AR"]

# Taxonomía canónica (GBIF Backbone resuelve automáticamente sinónimos mediante usageKey)
TAXONOMIA_ESPECIES = {
    "Alerce":    "Fitzroya cupressoides",
    "Araucaria": "Araucaria araucana",
    "Lleuque":   "Prumnopitys andina",
    "Queule":    "Gomortega keule",
    "Ruil":      "Nothofagus alessandrii",
}

# Grilla oficial UICN B2 (obligatoria: 2x2 km = 4 km2)
GRID_AOO_UICN_M = 2000
AREA_CELDA_UICN_KM2 = (GRID_AOO_UICN_M / 1000.0) ** 2

# Grilla fina complementaria según literatura chilena / Planes RECOGE (None si literatura usa 2x2 km)
AOO_LITERATURA_CONFIG = {
    "Queule": {
        "grid_m": 1000,
        "referencia": "Plan RECOGE Queule (MMA-FAO-GEF, 2022, pp. 6-7; LEP-UdeC, 2019): EOO = 4.392,5 km², AOO (1x1 km) = 86 km²; RCE = EN (D.S. 151/2007)."
    },
    "Ruil": {
        "grid_m": 1000,
        "referencia": "Plan RECOGE Ruil (D.S. 42/2020 MMA; Bustamante & Castor, 1998): distribución restringida en Maule (~352 ha en >180 fragmentos de 1-60 ha); RCE = EN."
    },
    "Lleuque": {
        "grid_m": None,
        "referencia": "Fichas RCE-MMA / UICN: EOO nacional reportado ~36.962 km², AOO estándar en 2x2 km; RCE = VU."
    },
    "Araucaria": {
        "grid_m": None,
        "referencia": "RCE-MMA / CONAF / UICN: poblaciones disyuntas Costa (Nahuelbuta) y Andes; AOO estándar 2x2 km; RCE/UICN = EN (por Criterio A / subpoblaciones)."
    },
    "Alerce": {
        "grid_m": None,
        "referencia": "RCE-MMA / Lara et al. / UICN: poblaciones en Costa, Depresión Intermedia (ej. Lahuen Ñadi) y Andes; AOO estándar 2x2 km; RCE/UICN = EN (por Criterio A)."
    },
}

# Envolventes biogeográficas nativas para depurar registros cultivados/urbanos/erróneos en GBIF
# (calibradas con distribución documentada en literatura y Planes RECOGE)
ENVOLVENTES_SUBPOBLACIONES = {
    "Alerce": {
        "Costa / Cordillera Pelada - Chiloé": {
            "lat_min": -43.20, "lat_max": -39.80,
            "lon_min": -74.35, "lon_max": -73.15
        },
        "Depresión Intermedia (Llanquihue / Lahuen Ñadi)": {
            "lat_min": -41.75, "lat_max": -41.00,
            "lon_min": -73.15, "lon_max": -72.75
        },
        "Andes / Los Lagos - Palena (- Chubut)": {
            "lat_min": -43.60, "lat_max": -40.65,
            "lon_min": -72.75, "lon_max": -71.15
        }
    },
    "Araucaria": {
        "Costa / Cordillera de Nahuelbuta": {
            "lat_min": -38.65, "lat_max": -37.50,
            "lon_min": -73.45, "lon_max": -72.85
        },
        "Andes / Biobío - Los Ríos (- Neuquén)": {
            "lat_min": -40.35, "lat_max": -37.25,
            "lon_min": -72.15, "lon_max": -70.50
        }
    },
    "Lleuque": {
        "Costa / Cordillera de Nahuelbuta": {
            "lat_min": -38.15, "lat_max": -37.55,
            "lon_min": -73.25, "lon_max": -72.75
        },
        "Andes / Maule - Araucanía": {
            "lat_min": -39.30, "lat_max": -35.40,
            "lon_min": -72.00, "lon_max": -70.75
        }
    },
    "Queule": {
        # Plan RECOGE (p. 5): 35°47'S (Cauquenes) hasta 37°41'S (Nahuelbuta/Arauco)
        "Distribución Costera Continua (Maule - Nahuelbuta)": {
            "lat_min": -37.85, "lat_max": -35.70,
            "lon_min": -73.35, "lon_max": -72.60
        }
    },
    "Ruil": {
        # Cordillera de la Costa del Maule (Curepto ~35°06'S hasta Pelluhue/Chanco ~35°50'S)
        "Distribución Costera Continua (Cordillera de la Costa, Maule)": {
            "lat_min": -36.05, "lat_max": -35.00,
            "lon_min": -72.65, "lon_max": -72.00
        }
    }
}

def validar_envolvente_nativa(especie, lat, lon, env):
    '''Valida si una coordenada cae dentro del rango biogeográfico nativo.'''
    if not (env["lat_min"] <= lat <= env["lat_max"]):
        return False
    if not (env["lon_min"] <= lon):
        return False
    # Exclusión fina de registros cultivados en el Valle Central al sur del río Biobío para Queule
    if especie == "Queule":
        lon_limite = -72.95 if lat <= -37.0 else env["lon_max"]
        return lon <= lon_limite
    return lon <= env["lon_max"]

sufijo_escala = "rce_cl" if ESCALA_EVALUACION == "RCE_CHILE" else "uicn_global"
ARCHIVO_SALIDA = f"analisis_completo_{ESPECIE_SELECCIONADA.lower()}_{sufijo_escala}.gpkg"

print(f"Especie seleccionada : {ESPECIE_SELECCIONADA} ({TAXONOMIA_ESPECIES[ESPECIE_SELECCIONADA]})")
print(f"Escala de evaluación : {ESCALA_EVALUACION} | Países GBIF: {PAISES_CONSULTA}")
print(f"AOO oficial UICN/RCE : {GRID_AOO_UICN_M} m ({AREA_CELDA_UICN_KM2:.1f} km²/celda)")
lit_cfg = AOO_LITERATURA_CONFIG[ESPECIE_SELECCIONADA]
if lit_cfg["grid_m"]:
    print(f"AOO fino literatura  : {lit_cfg['grid_m']} m ({(lit_cfg['grid_m']/1000)**2:.2f} km²/celda)")
print(f"Referencia literatura: {lit_cfg['referencia']}")
print(f"Archivo GeoPackage   : {ARCHIVO_SALIDA}")

## 3. Carga de Localidades de Referencia y Sistemas de Referencia Espacial (CRS)

### ¿Por qué usamos dos proyecciones (LAEA *Equal-Area* y UTM Chile)?
1. **Cálculo métrico de áreas y grillas (LAEA *Equal-Area*):**  
   Las proyecciones UTM (`EPSG:32718` / `EPSG:32719`) son **conformes** (preservan ángulos, no áreas exactas) y además Chile continental en el centro-sur cruza el meridiano $-72^\circ$ (límite entre zonas 18S y 19S). Siguiendo el estándar cartográfico de la UICN (*GeoCAT / rCAT*), proyectamos los datos a una **proyección equivalente (*Lambert Azimuthal Equal-Area*, LAEA)** centrada en el centroide $(\phi_0, \lambda_0)$ de la especie:
   $$\text{PROJ: } \texttt{+proj=laea +lat\_0=}\phi_0\;\texttt{+lon\_0=}\lambda_0\;\texttt{+datum=WGS84 +units=m +no\_defs}$$
   Esta proyección garantiza **distorsión de área nula** en el cálculo de EOO, AOO y áreas hexagonales.
2. **Exportación cartográfica para QGIS (UTM WGS84):**  
   Al guardar el GeoPackage (`.gpkg`), las capas se reproyectan al huso UTM chileno correspondiente (`EPSG:32718` si $\bar{\lambda} \le -72^\circ$, o `EPSG:32719` en caso contrario) para compatibilidad inmediata con las capas del curso en QGIS, conservando intactos los atributos de área calculados en LAEA.

In [ ]:
REPO_GITHUB_RAW = "https://raw.githubusercontent.com/magdzraw/byc-2026/main/data"

def resolver_archivo_entrada(ruta_archivo):
    '''Busca el archivo localmente, en data/ o ../data/, y como respaldo lo descarga desde GitHub.'''
    if os.path.exists(ruta_archivo):
        return ruta_archivo
    nombre = os.path.basename(ruta_archivo)
    for cand in (os.path.join("data", nombre), os.path.join("..", "data", nombre)):
        if os.path.exists(cand):
            return cand
    url_raw = f"{REPO_GITHUB_RAW}/{nombre}"
    try:
        r = SESSION_HTTP.get(url_raw, timeout=20)
        if r.status_code == 200:
            with open(nombre, "wb") as f:
                f.write(r.content)
            print(f"Archivo de ejemplo '{nombre}' descargado automáticamente desde GitHub (magdzraw/byc-2026).")
            return nombre
    except Exception:
        pass
    raise FileNotFoundError(
        f"No se encontró '{ruta_archivo}' en el entorno local ni en el repositorio de GitHub. "
        f"En Colab, súbelo al panel de archivos o ajusta la ruta."
    )

def cargar_csv_localidades(ruta_csv):
    '''Carga el CSV de localidades manejando codificación UTF-8/Latin-1 y separador ';' o ','.'''
    ruta_csv = resolver_archivo_entrada(ruta_csv)
    for enc in ("utf-8-sig", "utf-8", "latin-1"):
        try:
            with open(ruta_csv, "r", encoding=enc) as fh:
                primera_linea = fh.readline()
            sep = ";" if primera_linea.count(";") >= primera_linea.count(",") else ","
            df = pd.read_csv(ruta_csv, sep=sep, encoding=enc)
            df.columns = [c.strip().lstrip("\ufeff") for c in df.columns]
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"No se pudo decodificar '{ruta_csv}'.")

df_total = cargar_csv_localidades(CSV_UNIFICADO)
df_sitios = df_total[df_total["Especie"].str.strip() == ESPECIE_SELECCIONADA].copy()

if df_sitios.empty:
    raise ValueError(f"No se encontraron localidades para '{ESPECIE_SELECCIONADA}' en {CSV_UNIFICADO}.")

df_sitios["Lat"] = pd.to_numeric(df_sitios["Lat"], errors="coerce")
df_sitios["Lon"] = pd.to_numeric(df_sitios["Lon"], errors="coerce")
df_sitios = df_sitios.dropna(subset=["Lat", "Lon"]).reset_index(drop=True)

# Definición de proyecciones
lat_centro = float(df_sitios["Lat"].mean())
lon_centro = float(df_sitios["Lon"].mean())

CRS_EQUAL_AREA = f"+proj=laea +lat_0={lat_centro:.4f} +lon_0={lon_centro:.4f} +datum=WGS84 +units=m +no_defs"
CRS_EXPORT_UTM = "EPSG:32718" if lon_centro <= -72.0 else "EPSG:32719"

cols_sitios = [c for c in ["ID", "Referencia", "Codigo", "Lat", "Lon"] if c in df_sitios.columns]
gdf_sitios_wgs84 = gpd.GeoDataFrame(
    df_sitios[cols_sitios],
    geometry=gpd.points_from_xy(df_sitios["Lon"], df_sitios["Lat"]),
    crs="EPSG:4326"
)
gdf_sitios_ea = gdf_sitios_wgs84.to_crs(CRS_EQUAL_AREA)

print(f"Localidades de referencia cargadas: {len(gdf_sitios_wgs84)}")
print(f"CRS cálculo métrico (Equal-Area) : LAEA centrado en ({lat_centro:.3f}°, {lon_centro:.3f}°)")
print(f"CRS exportación QGIS             : {CRS_EXPORT_UTM}")
display(df_sitios[cols_sitios])

## 4. Descarga y Curaduría de Ocurrencias de la Especie Focal (GBIF + Terreno)

Para garantizar trazabilidad y calidad de datos conforme a los estándares UICN/RCE:
1. **Resolución taxonómica automática (`taxonKey`):** Consultamos `/v1/species/match` del *GBIF Backbone Taxonomy* para obtener el `usageKey` aceptado. Esto incluye automáticamente todos los sinónimos reconocidos (por ejemplo, *Podocarpus andinus* para *Prumnopitys andina* o *Gomortega nitida* para *Gomortega keule*) sin riesgo de duplicados ni homónimos.
2. **Filtros de calidad espacial y biológica:**
   * `occurrenceStatus = 'PRESENT'` (excluye registros de ausencia explícita).
   * `hasCoordinate = 'true'` y `hasGeospatialIssue = 'false'`.
   * Exclusión de especímenes fósiles (`FOSSIL_SPECIMEN`) y registros declarados como cultivados, introducidos o manejados (`CULTIVATED`, `INTRODUCED`, `MANAGED`).
   * Filtro de precisión espacial: `coordinateUncertaintyInMeters <= INCERTIDUMBRE_MAXIMA_M` (o no reportada, documentándose su proporción).
   * Validación dentro de la envolvente biogeográfica nativa (descarta ejemplares ornamentales en parques urbanos o jardines botánicos).
3. **Integración con localidades de terreno (`Coordenadas_codigo.csv`):** Si `INCLUIR_SITIOS_CSV_EN_UICN = True`, los sitios confirmados de terreno/muestreo molecular se incorporan al conjunto de presencias validadas (identificados en el campo `fuente`).

In [ ]:
URL_GBIF_MATCH = "https://api.gbif.org/v1/species/match"
URL_GBIF_OCC = "https://api.gbif.org/v1/occurrence/search"

nombre_cientifico = TAXONOMIA_ESPECIES[ESPECIE_SELECCIONADA]
resp_match = SESSION_HTTP.get(URL_GBIF_MATCH, params={"name": nombre_cientifico}, timeout=15)
resp_match.raise_for_status()
match_info = resp_match.json()
TAXON_KEY = match_info.get("usageKey")
if not TAXON_KEY:
    raise RuntimeError(f"No se pudo resolver taxonKey en GBIF para '{nombre_cientifico}'.")

print(f"Resolución GBIF Backbone: {match_info.get('scientificName')} | taxonKey = {TAXON_KEY} | estado = {match_info.get('status')}")

envolventes_sp = ENVOLVENTES_SUBPOBLACIONES[ESPECIE_SELECCIONADA]
es_disyunta = len(envolventes_sp) > 1

# Paginación completa de ocurrencias en GBIF
registros_crudos = []
offset = 0
limit = 300
total_gbif = None
TOPE_SEGURIDAD_ESPECIE = 15000

while True:
    params = [
        ("taxonKey", str(TAXON_KEY)),
        ("hasCoordinate", "true"),
        ("hasGeospatialIssue", "false"),
        ("occurrenceStatus", "PRESENT"),
        ("limit", str(limit)),
        ("offset", str(offset)),
    ]
    for pais in PAISES_CONSULTA:
        params.append(("country", pais))

    resp = SESSION_HTTP.get(URL_GBIF_OCC, params=params, timeout=20)
    if resp.status_code != 200:
        print(f"Aviso HTTP {resp.status_code} en offset {offset}; deteniendo paginación.")
        break

    data = resp.json()
    if total_gbif is None:
        total_gbif = data.get("count", 0)
    lote = data.get("results", [])
    if not lote:
        break

    registros_crudos.extend(lote)
    offset += limit
    if data.get("endOfRecords", False) or offset >= total_gbif:
        break
    if offset >= TOPE_SEGURIDAD_ESPECIE:
        print(f"ADVERTENCIA: Se alcanzó el tope de {TOPE_SEGURIDAD_ESPECIE} registros (total GBIF = {total_gbif}).")
        break

# Curaduría de registros con conteo transparente de descartes
descartes = {"fosil": 0, "cultivado": 0, "incertidumbre": 0, "fuera_envolvente": 0}
registros_validos = []
sin_incertidumbre = 0

for r in registros_crudos:
    lat = r.get("decimalLatitude")
    lon = r.get("decimalLongitude")
    if lat is None or lon is None:
        continue

    if r.get("basisOfRecord") == "FOSSIL_SPECIMEN":
        descartes["fosil"] += 1
        continue

    estab = str(r.get("establishmentMeans", "")).upper()
    deg_estab = str(r.get("degreeOfEstablishment", "")).upper()
    if any(k in estab or k in deg_estab for k in ("CULTIVATED", "INTRODUCED", "MANAGED")):
        descartes["cultivado"] += 1
        continue

    incert = r.get("coordinateUncertaintyInMeters")
    if incert is None:
        sin_incertidumbre += 1
    elif incert > INCERTIDUMBRE_MAXIMA_M:
        descartes["incertidumbre"] += 1
        continue

    sub_asignada = None
    for sub_nombre, env in envolventes_sp.items():
        if validar_envolvente_nativa(ESPECIE_SELECCIONADA, lat, lon, env):
            sub_asignada = sub_nombre
            break

    if sub_asignada is None:
        descartes["fuera_envolvente"] += 1
        continue

    registros_validos.append({
        "gbif_id": str(r.get("key", "")),
        "species": r.get("species", nombre_cientifico),
        "subpoblacion": sub_asignada,
        "decimalLatitude": float(lat),
        "decimalLongitude": float(lon),
        "uncertainty_m": -1.0 if incert is None else float(incert),
        "year": r.get("year"),
        "countryCode": r.get("countryCode", "CL"),
        "fuente": "GBIF",
    })

df_sp = pd.DataFrame(registros_validos)

# Integrar localidades validadas del CSV (terreno / muestreo molecular)
if INCLUIR_SITIOS_CSV_EN_UICN:
    filas_csv = []
    for _, fila in df_sitios.iterrows():
        lat, lon = float(fila["Lat"]), float(fila["Lon"])
        sub_asig = None
        for s_nombre, env in envolventes_sp.items():
            if validar_envolvente_nativa(ESPECIE_SELECCIONADA, lat, lon, env):
                sub_asig = s_nombre
                break
        if sub_asig is None:
            sub_asig = list(envolventes_sp.keys())[0]
        filas_csv.append({
            "gbif_id": f"CSV_{fila.get('Codigo', fila.get('ID', ''))}",
            "species": nombre_cientifico,
            "subpoblacion": sub_asig,
            "decimalLatitude": lat,
            "decimalLongitude": lon,
            "uncertainty_m": 0.0,
            "year": 2026,
            "countryCode": "CL",
            "fuente": "Localidad_Referencia",
        })
    df_csv_pts = pd.DataFrame(filas_csv)
    df_sp = pd.concat([df_sp, df_csv_pts], ignore_index=True)

if df_sp.empty:
    raise RuntimeError("No quedaron ocurrencias válidas tras los filtros. Revisa las envolventes o el CSV.")

# Deduplicación espacial exacta (EOO y AOO dependen de sitios únicos de presencia)
n_antes_dedup = len(df_sp)
df_sp["lat_r"] = df_sp["decimalLatitude"].round(5)
df_sp["lon_r"] = df_sp["decimalLongitude"].round(5)
df_sp = df_sp.drop_duplicates(subset=["lat_r", "lon_r"]).drop(columns=["lat_r", "lon_r"]).reset_index(drop=True)

gdf_sp_wgs84 = gpd.GeoDataFrame(
    df_sp,
    geometry=gpd.points_from_xy(df_sp["decimalLongitude"], df_sp["decimalLatitude"]),
    crs="EPSG:4326"
)
gdf_sp_ea = gdf_sp_wgs84.to_crs(CRS_EQUAL_AREA)

print(f"\nResumen de curaduría GBIF ({PAISES_CONSULTA}):")
print(f"  Registros brutos descargados     : {len(registros_crudos)} (reportados por API: {total_gbif})")
print(f"  Descartados por fósil            : {descartes['fosil']}")
print(f"  Descartados por cultivado/intro. : {descartes['cultivado']}")
print(f"  Descartados por incert. >{INCERTIDUMBRE_MAXIMA_M}m : {descartes['incertidumbre']}")
print(f"  Descartados fuera de envolvente  : {descartes['fuera_envolvente']}")
print(f"  Válidos sin incert. reportada    : {sin_incertidumbre}")
print(f"  Sitios únicos finales (con CSV)  : {len(gdf_sp_ea)} (deduplicados desde {n_antes_dedup})")
print("\nDistribución por subpoblación biogeográfica:")
print(gdf_sp_ea["subpoblacion"].value_counts().to_string())

## 5. Máscara Territorial de Alta Resolución (GADM 4.1 / Natural Earth 1:10m)

En Chile centro-sur y austral (Chiloé, fiordos de Palena, Hornopirén, Chaitén y borde costero del Maule–Biobío), las capas mundiales de baja resolución (`1:50m` o `1:110m`) omiten islas y simplifican drásticamente la costa, deformando el área del EOO.

Aquí cargamos automáticamente el límite oficial de **alta resolución de GADM v4.1** (`gadm41_CHL_0.json.zip` para `RCE_CHILE`, sumando `gadm41_ARG_0.json.zip` si `ESCALA_EVALUACION == 'UICN_GLOBAL'`), con respaldo automático a **Natural Earth 1:10m** si el servidor GADM no respondiera.  
*(Si prefieres usar un shapefile local del INE/IDE Chile como `CHL_ADM1.shp`, basta con indicar su ruta en `RUTA_LIMITE_LOCAL`.)*

In [ ]:
RUTA_LIMITE_LOCAL = ""  # Opcional: ruta a CHL_ADM1.shp o similar si se desea usar capa local

def cargar_mascara_alta_resolucion(paises, bbox_wgs84, ruta_local=""):
    '''Descarga y recorta la máscara terrestre de alta resolución (GADM 4.1 o Natural Earth 10m).'''
    gdfs = []
    fuente_usada = ""

    if ruta_local and os.path.exists(ruta_local):
        g = gpd.read_file(ruta_local).to_crs("EPSG:4326")
        gdfs.append(g)
        fuente_usada = f"Archivo local ({ruta_local})"
    else:
        mapa_iso3 = {"CL": "CHL", "AR": "ARG"}
        cache_dir = "cache_limites_geo"
        os.makedirs(cache_dir, exist_ok=True)
        gadm_ok = True

        for p in paises:
            iso3 = mapa_iso3[p]
            zip_local = os.path.join(cache_dir, f"gadm41_{iso3}_0.json.zip")
            url_gadm = f"https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_{iso3}_0.json.zip"
            try:
                if not os.path.exists(zip_local):
                    r = SESSION_HTTP.get(url_gadm, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
                    r.raise_for_status()
                    with open(zip_local, "wb") as f:
                        f.write(r.content)
                with zipfile.ZipFile(zip_local, "r") as zf:
                    json_name = [n for n in zf.namelist() if n.endswith(".json")][0]
                    json_local = os.path.join(cache_dir, json_name)
                    if not os.path.exists(json_local):
                        zf.extract(json_name, cache_dir)
                gdfs.append(gpd.read_file(json_local).to_crs("EPSG:4326"))
            except Exception as e:
                print(f"Aviso: No se pudo cargar GADM 4.1 para {iso3} ({e}). Usando respaldo Natural Earth 10m...")
                gadm_ok = False
                gdfs = []
                break

        if gadm_ok and gdfs:
            fuente_usada = f"GADM v4.1 alta resolución ({', '.join(mapa_iso3[p] for p in paises)})"
        else:
            ne_zip = os.path.join(cache_dir, "ne_10m_admin_0_countries.zip")
            url_ne = "https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_0_countries.zip"
            if not os.path.exists(ne_zip):
                r = SESSION_HTTP.get(url_ne, timeout=45, headers={"User-Agent": "Mozilla/5.0"})
                r.raise_for_status()
                with open(ne_zip, "wb") as f:
                    f.write(r.content)
            ne_gdf = gpd.read_file(f"zip://{ne_zip}").to_crs("EPSG:4326")
            iso_list = [mapa_iso3[p] for p in paises]
            col_iso = "ADM0_A3" if "ADM0_A3" in ne_gdf.columns else "ISO_A3"
            gdfs.append(ne_gdf[ne_gdf[col_iso].isin(iso_list)].copy())
            fuente_usada = f"Natural Earth 1:10m ({', '.join(iso_list)})"

    gdf_paises = pd.concat(gdfs, ignore_index=True)
    gdf_paises["geometry"] = gdf_paises.geometry.make_valid()
    gdf_clip = gpd.clip(gdf_paises, bbox_wgs84)
    gdf_ea = gdf_clip.to_crs(CRS_EQUAL_AREA)
    mascara_geom = shapely.make_valid(gdf_ea.geometry.union_all())
    return mascara_geom, fuente_usada

bbox_estudio_ea = box(*gdf_sp_ea.total_bounds).buffer(30000)
bbox_estudio_wgs84 = gpd.GeoSeries([bbox_estudio_ea], crs=CRS_EQUAL_AREA).to_crs("EPSG:4326").iloc[0]

mascara_tierra_ea, FUENTE_MASCARA = cargar_mascara_alta_resolucion(
    PAISES_CONSULTA, bbox_estudio_wgs84, ruta_local=RUTA_LIMITE_LOCAL
)

# Validación espacial estricta contra el polígono terrestre (tolerancia costera de 1 km para deriva GPS)
if ESCALA_EVALUACION == "RCE_CHILE":
    mascara_tol = mascara_tierra_ea.buffer(1000)
    en_territorio = gdf_sp_ea.geometry.within(mascara_tol)
    n_fuera_frontera = int((~en_territorio).sum())
    if n_fuera_frontera > 0:
        print(f"Se depuraron {n_fuera_frontera} registros ubicados fuera del límite nacional de Chile (tolerancia costera 1 km).")
        gdf_sp_ea = gdf_sp_ea[en_territorio].reset_index(drop=True)
        gdf_sp_wgs84 = gdf_sp_ea.to_crs("EPSG:4326")

print(f"Máscara territorial lista: {FUENTE_MASCARA}")
print(f"Ocurrencias definitivas para cálculo EOO/AOO: {len(gdf_sp_ea)}")

## 6. Extensión de Presencia — EOO (Criterio B1 UICN / RCE)

### Definición y Fórmula
La **Extensión de Presencia (EOO)** es el área contenida dentro del límite imaginario continuo más corto que engloba todos los sitios conocidos, inferidos o proyectados de presencia actual de un taxón (UICN v16, Sec. 4.9). Se calcula como el área del **Polígono Convexo Mínimo (MCP, *convex hull*)**:

$$\text{EOO} = \frac{\text{Área}\left(\operatorname{Conv}(P)\right)}{10^6}\quad [\text{km}^2]$$

donde $P = \{p_1, p_2, \dots, p_n\}$ es el conjunto de ocurrencias nativas proyectadas en LAEA *Equal-Area*.

### Reglas Normativas Aplicadas
* **No exclusión de disyunciones en B1 (UICN Sec. 4.9):** Para el Criterio B1, la UICN desaconseja excluir disyunciones (como el Valle Central entre Costa y Andes), ya que el EOO cuantifica la **dispersión espacial del riesgo**. Por ello, el **EOO Oficial B1** corresponde al **MCP global** de todos los registros (recortado a territorio nacional terrestre en modo `RCE_CHILE`; reportando también el MCP bruto sin recorte marino).
* **Análisis biogeográfico complementario:** Para especies con distribución disyunta (*Alerce*, *Araucaria*, *Lleuque*), se calculan además los polígonos separados por núcleo (Costa, Depresión Intermedia, Andes) como capa informativa para conservación regional y cruce con estructura genética.
* **Umbrales geográficos preliminares del Criterio B1:**
  * $\text{EOO} < 100\text{ km}^2 \rightarrow$ umbral de **En Peligro Crítico (CR)**
  * $\text{EOO} < 5.000\text{ km}^2 \rightarrow$ umbral de **En Peligro (EN)**
  * $\text{EOO} < 20.000\text{ km}^2 \rightarrow$ umbral de **Vulnerable (VU)**
  * **¡Importante!** Cumplir el umbral geográfico B1 **no asigna por sí solo una categoría de amenaza**: la UICN/RCE exige cumplir además **al menos 2 de 3 subcriterios**: *(a)* fragmentación severa o número de localidades por amenaza ($\le 1, 5, 10$), *(b)* disminución continua, o *(c)* fluctuaciones extremas. Tampoco superar $20.000\text{ km}^2$ implica automáticamente *Preocupación Menor (LC)*, pues la especie puede calificar bajo otros criterios (p. ej., **Criterio A** por reducción poblacional, como ocurre en Alerce y Araucaria).

In [ ]:
def evaluar_umbral_b1(km2):
    '''Clasifica únicamente el umbral geográfico preliminar de EOO (Criterio B1).'''
    if km2 < 100:
        return "Umbral B1 de CR (<100 km²; requiere ≥2 subcriterios a/b/c)"
    elif km2 < 5000:
        return "Umbral B1 de EN (<5.000 km²; requiere ≥2 subcriterios a/b/c)"
    elif km2 < 20000:
        return "Umbral B1 de VU (<20.000 km²; requiere ≥2 subcriterios a/b/c)"
    else:
        return "Supera umbral B1 (≥20.000 km²; no activa B1, evaluar criterios A/B2/C/D)"

def construir_mcp(geometrias_gdf, mascara=None):
    '''Construye el Polígono Convexo Mínimo (MCP) y opcionalmente recorta por máscara terrestre.'''
    n = len(geometrias_gdf)
    if n >= 3:
        geom = geometrias_gdf.geometry.union_all().convex_hull
    elif n in (1, 2):
        geom = geometrias_gdf.geometry.buffer(2000).union_all()
    else:
        return Polygon()

    if mascara is not None:
        geom = shapely.make_valid(geom.intersection(mascara))
        if geom.geom_type == "GeometryCollection":
            polys = [g for g in geom.geoms if g.geom_type in ("Polygon", "MultiPolygon")]
            geom = unary_union(polys) if polys else Polygon()
    return geom

# 1. EOO MCP Bruto (sin recorte marino — referencia estricta UICN v16 Sec. 4.9)
eoo_mcp_bruto_geom = construir_mcp(gdf_sp_ea, mascara=None)
eoo_mcp_bruto_km2 = round(eoo_mcp_bruto_geom.area / 1e6, 2)

# 2. EOO MCP Recortado a máscara terrestre nacional / regional (estándar práctico fichas RCE Chile)
eoo_mcp_tierra_geom = construir_mcp(gdf_sp_ea, mascara=mascara_tierra_ea)
eoo_mcp_tierra_km2 = round(eoo_mcp_tierra_geom.area / 1e6, 2)

# EOO oficial según la escala elegida
if ESCALA_EVALUACION == "RCE_CHILE":
    eoo_oficial_geom = eoo_mcp_tierra_geom
    eoo_oficial_km2 = eoo_mcp_tierra_km2
    metodo_eoo = "MCP Nacional Chile recortado a tierra firme (RCE-MMA)"
else:
    eoo_oficial_geom = eoo_mcp_bruto_geom
    eoo_oficial_km2 = eoo_mcp_bruto_km2
    metodo_eoo = "MCP Global sin exclusión de disyunciones (UICN v16 Sec. 4.9)"

gdf_eoo_oficial_ea = gpd.GeoDataFrame([{
    "especie": ESPECIE_SELECCIONADA,
    "escala": ESCALA_EVALUACION,
    "metodo_eoo": metodo_eoo,
    "n_registros": len(gdf_sp_ea),
    "eoo_km2": eoo_oficial_km2,
    "eoo_mcp_bruto_km2": eoo_mcp_bruto_km2,
    "eoo_mcp_tierra_km2": eoo_mcp_tierra_km2,
    "umbral_preliminar_b1": evaluar_umbral_b1(eoo_oficial_km2),
    "geometry": eoo_oficial_geom,
}], crs=CRS_EQUAL_AREA)

# 3. Análisis biogeográfico complementario por núcleos/subpoblaciones (solo si la especie es disyunta)
gdf_eoo_subpob_ea = None
if es_disyunta:
    filas_sub = []
    for sub_nombre in envolventes_sp.keys():
        pts_sub = gdf_sp_ea[gdf_sp_ea["subpoblacion"] == sub_nombre]
        if len(pts_sub) > 0:
            geom_sub = construir_mcp(pts_sub, mascara=mascara_tierra_ea)
            if not geom_sub.is_empty:
                area_sub = round(geom_sub.area / 1e6, 2)
                filas_sub.append({
                    "especie": ESPECIE_SELECCIONADA,
                    "subpoblacion": sub_nombre,
                    "tipo": "Núcleo biogeográfico (complementario, no reemplaza B1)",
                    "n_registros": len(pts_sub),
                    "area_nucleo_km2": area_sub,
                    "geometry": geom_sub,
                })
    if filas_sub:
        gdf_eoo_subpob_ea = gpd.GeoDataFrame(filas_sub, crs=CRS_EQUAL_AREA)

print(f"EOO Oficial ({ESCALA_EVALUACION}) : {eoo_oficial_km2:,.2f} km²")
print(f"  ├─ MCP bruto (sin recorte marino) : {eoo_mcp_bruto_km2:,.2f} km²")
print(f"  ├─ MCP terrestre ({FUENTE_MASCARA}) : {eoo_mcp_tierra_km2:,.2f} km²")
print(f"  └─ Evaluación preliminar B1       : {evaluar_umbral_b1(eoo_oficial_km2)}")

if gdf_eoo_subpob_ea is not None:
    suma_nucleos = gdf_eoo_subpob_ea["area_nucleo_km2"].sum()
    matriz_intermedia = max(0.0, eoo_mcp_tierra_km2 - suma_nucleos)
    print("\nDesglose biogeográfico complementario por núcleos (no sustituye B1):")
    for _, r in gdf_eoo_subpob_ea.iterrows():
        print(f"  ├─ {r['subpoblacion']}: {r['area_nucleo_km2']:,.2f} km² ({r['n_registros']} registros)")
    print(f"  └─ Suma núcleos: {suma_nucleos:,.2f} km² | Matriz intermedia excluida: {matriz_intermedia:,.2f} km²")

## 7. Área de Ocupación — AOO (Criterio B2 UICN / RCE + Escala Fina de Literatura)

### Definición, Escala Estándar y Optimización de Grilla
El **Área de Ocupación (AOO)** es el área dentro del EOO efectivamente ocupada por el taxón (UICN v16, Sec. 4.10). Se estima superponiendo una cuadrícula regular de lado $g$ y contando las celdas ocupadas $N_{\text{celdas}}$:

$$\text{AOO}(g) = \min_{(\delta_x, \delta_y) \in [0, g)^2} \left|\left\{\left(\left\lfloor \frac{x_i - \delta_x}{g} \right\rfloor,\; \left\lfloor \frac{y_i - \delta_y}{g} \right\rfloor\right) : (x_i, y_i) \in P\right\}\right| \times \left(\frac{g}{1000}\right)^2\quad [\text{km}^2]$$

* **Grilla obligatoria para Criterio B2 ($g = 2000\text{ m}$, celda $= 4\text{ km}^2$):** La UICN (Sec. 4.10) y el RCE-MMA calibran los umbrales de B2 exclusivamente a esta escala:
  * $\text{AOO} < 10\text{ km}^2 \rightarrow$ umbral de **En Peligro Crítico (CR)**
  * $\text{AOO} < 500\text{ km}^2 \rightarrow$ umbral de **En Peligro (EN)**
  * $\text{AOO} < 2.000\text{ km}^2 \rightarrow$ umbral de **Vulnerable (VU)**
* **Minimización por desplazamiento de origen $(\delta_x, \delta_y)$:** Como el conteo de celdas depende de dónde se ubique el origen de la malla, evaluamos $10 \times 10 = 100$ desplazamientos sistemáticos y seleccionamos el **número mínimo de celdas ocupadas**, según indica explícitamente la UICN (Sec. 4.10).
* **Grilla fina complementaria de literatura / Plan RECOGE ($g = 1000\text{ m}$ para Queule y Ruil):** Permite comparar directamente contra estudios de micro-fragmentación (p. ej., **Plan RECOGE Queule 2022, p. 6**, que usa celdas de $1\text{ km}^2$), manteniéndola separada del Criterio B2 oficial.

In [ ]:
def evaluar_umbral_b2(km2):
    '''Clasifica únicamente el umbral geográfico preliminar de AOO en 2x2 km (Criterio B2).'''
    if km2 < 10:
        return "Umbral B2 de CR (<10 km²; requiere ≥2 subcriterios a/b/c)"
    elif km2 < 500:
        return "Umbral B2 de EN (<500 km²; requiere ≥2 subcriterios a/b/c)"
    elif km2 < 2000:
        return "Umbral B2 de VU (<2.000 km²; requiere ≥2 subcriterios a/b/c)"
    else:
        return "Supera umbral B2 (≥2.000 km²; no activa B2, evaluar criterios A/B1/C/D)"

def calcular_aoo_optimizado(gdf_pts_ea, grid_m, pasos=10):
    '''
    Calcula el AOO buscando el origen de grilla (dx, dy) que minimiza el número de celdas
    ocupadas (UICN v16 Sec. 4.10). Devuelve GeoDataFrame de celdas y estadísticas.
    '''
    xs = gdf_pts_ea.geometry.x.to_numpy()
    ys = gdf_pts_ea.geometry.y.to_numpy()
    area_celda_km2 = (grid_m / 1000.0) ** 2

    # Conteo base sin optimizar (origen 0, 0) para comparación didáctica
    celdas_base = len(set(zip(np.floor(xs / grid_m).astype(int), np.floor(ys / grid_m).astype(int))))

    mejor_n = None
    mejor_offset = (0.0, 0.0)
    mejores_celdas = None

    desplazamientos = np.linspace(0.0, grid_m, pasos, endpoint=False)
    for dx in desplazamientos:
        c_idx = np.floor((xs - dx) / grid_m).astype(int)
        for dy in desplazamientos:
            r_idx = np.floor((ys - dy) / grid_m).astype(int)
            unicas = set(zip(c_idx, r_idx))
            n_u = len(unicas)
            if mejor_n is None or n_u < mejor_n:
                mejor_n = n_u
                mejor_offset = (float(dx), float(dy))
                mejores_celdas = unicas

    dx_opt, dy_opt = mejor_offset
    poligonos = [
        box(
            c * grid_m + dx_opt,
            r * grid_m + dy_opt,
            (c + 1) * grid_m + dx_opt,
            (r + 1) * grid_m + dy_opt,
        )
        for (c, r) in sorted(mejores_celdas)
    ]

    aoo_km2 = round(mejor_n * area_celda_km2, 2)
    gdf_celdas = gpd.GeoDataFrame({
        "especie": ESPECIE_SELECCIONADA,
        "taxon": nombre_cientifico,
        "escala": ESCALA_EVALUACION,
        "celda_id": range(1, len(poligonos) + 1),
        "resolucion_m": grid_m,
        "area_celda_km2": area_celda_km2,
        "n_celdas_min": mejor_n,
        "n_celdas_origen0": celdas_base,
        "aoo_total_km2": aoo_km2,
        "geometry": poligonos,
    }, crs=CRS_EQUAL_AREA)

    return gdf_celdas, {
        "n_celdas_min": mejor_n,
        "n_celdas_origen0": celdas_base,
        "aoo_km2": aoo_km2,
        "area_celda_km2": area_celda_km2,
        "offset_m": mejor_offset,
    }

# 1. AOO Oficial UICN / RCE (2x2 km obligatorio)
gdf_aoo_uicn_ea, stats_aoo_uicn = calcular_aoo_optimizado(
    gdf_sp_ea, grid_m=GRID_AOO_UICN_M, pasos=PASOS_OPTIMIZACION_AOO
)
gdf_aoo_uicn_ea["umbral_preliminar_b2"] = evaluar_umbral_b2(stats_aoo_uicn["aoo_km2"])

print(f"AOO Oficial UICN/RCE (2x2 km): {stats_aoo_uicn['aoo_km2']:,.2f} km² "
      f"({stats_aoo_uicn['n_celdas_min']} celdas mínimas; vs. {stats_aoo_uicn['n_celdas_origen0']} en origen fijo 0,0)")
print(f"  └─ Evaluación preliminar B2 : {evaluar_umbral_b2(stats_aoo_uicn['aoo_km2'])}")

# 2. AOO Complementario de Escala Fina según Literatura / RECOGE (si aplica)
gdf_aoo_lit_ea = None
stats_aoo_lit = None
grid_lit_m = AOO_LITERATURA_CONFIG[ESPECIE_SELECCIONADA]["grid_m"]

if grid_lit_m is not None and grid_lit_m != GRID_AOO_UICN_M:
    gdf_aoo_lit_ea, stats_aoo_lit = calcular_aoo_optimizado(
        gdf_sp_ea, grid_m=grid_lit_m, pasos=PASOS_OPTIMIZACION_AOO
    )
    gdf_aoo_lit_ea["nota_escala"] = "Escala fina de literatura/RECOGE (no aplicar umbrales B2)"
    print(f"\nAOO Complementario Literatura ({grid_lit_m}x{grid_lit_m} m): {stats_aoo_lit['aoo_km2']:,.2f} km² "
          f"({stats_aoo_lit['n_celdas_min']} celdas mínimas; vs. {stats_aoo_lit['n_celdas_origen0']} en origen 0,0)")
    print(f"  └─ Referencia: {AOO_LITERATURA_CONFIG[ESPECIE_SELECCIONADA]['referencia']}")

## 8. Biodiversidad Local de Flora Vascular Acompañante (`Tracheophyta`) en Malla Hexagonal

### 8.1. ¿Cómo y con qué datos se construyen los hexágonos?

Para evaluar la diversidad de flora vascular en el entorno inmediato de las poblaciones estudiadas, construimos una **teselación hexagonal regular** en la proyección equivalente (**LAEA *Equal-Area***), garantizando que todas las unidades de muestreo tengan exactamente la misma superficie.

> **¿Por qué hexágonos y no cuadrados?** En ecología del paisaje, los hexágonos ofrecen **6 vecinos equidistantes** desde su centro (evitando la ambigüedad entre vecinos ortogonales y diagonales propia de las cuadrículas cuadradas) y una menor relación perímetro/área, reduciendo efectos de borde al agrupar ocurrencias puntuales.

#### Paso a paso de la construcción geométrica
1. **Datos base (centros espaciales):** Se utilizan las coordenadas de las **localidades de referencia** cargadas desde `Coordenadas_codigo.csv` (`gdf_sitios_ea`, proyectadas en metros en LAEA *Equal-Area*).
2. **Área de influencia local (`BUFFER_LOCAL_M = 7500 m`):** Alrededor de cada localidad se genera un círculo (*buffer*) de radio $7{,}5\text{ km}$ y todos los círculos se fusionan en una sola geometría (`area_estudio_local`) que delimita el paisaje circundante a los sitios de estudio.
3. **Geometría de cada hexágono regular (`RADIO_HEXAGONO_M = 2500 m`):**
   * Cada celda es un hexágono regular en orientación ***pointy-top*** (vértices superior e inferior en punta) con circunradio $r = \texttt{RADIO\_HEXAGONO\_M} = 2500\text{ m}$ (distancia del centro a cada vértice, igual a la longitud de cada lado).
   * **Área exacta de cada hexágono (`area_hex_km2`):**
     $$A_{\text{hex}} = \frac{3\sqrt{3}}{2}\, r^2$$
     Con $r = 2{,}5\text{ km}$, cada hexágono cubre exactamente $A_{\text{hex}} \approx 16{,}24\text{ km}^2$.
   * **Coordenadas de los 6 vértices** respecto al centro $(x_c, y_c)$ de cada celda (`offsets_x`, `offsets_y`):
     $$x_k = x_c + r \cos(\theta_k), \qquad y_k = y_c + r \sin(\theta_k), \qquad \theta_k = \frac{\pi}{6} + k\frac{\pi}{3} \quad (k = 0, 1, \dots, 5)$$
     El ángulo inicial $\pi/6 = 30^\circ$ rota la figura a orientación *pointy-top* (se usan 7 puntos en `np.linspace` para cerrar el polígono volviendo al vértice inicial).
   * **Espaciado entre centros para encaje tipo panal (sin huecos ni solapamientos):**
     * **Paso horizontal entre columnas (`dx`):** $dx = \sqrt{3}\, r \approx 4330{,}1\text{ m}$ (distancia entre lados opuestos del hexágono).
     * **Paso vertical entre filas (`dy`):** $dy = 1{,}5\, r = 3750\text{ m}$.
     * **Desfase alternado (`offset_x`):** en las filas impares (`fila % 2 == 1`), todos los centros se desplazan horizontalmente $dx / 2 = \frac{\sqrt{3}}{2} r$ para que las filas encajen exactamente entre sí.
4. **Selección de hexágonos del área de estudio:** Se recorre el rectángulo envolvente (`xmin, ymin, xmax, ymax`) de `area_estudio_local` y se conservan únicamente los hexágonos que **intersectan** dicho buffer (`poly.intersects(area_estudio_local)`).

---

### 8.2. Datos biológicos incorporados a cada hexágono (GBIF `Tracheophyta`)
* **Filtro taxonómico (`phylumKey = 7707728`):** Se consulta la API v1 de GBIF filtrando exclusivamente por el filo **Tracheophyta** (plantas vasculares: árboles, arbustos, hierbas y helechos). *(Nota técnica: GBIF exige la clave numérica `phylumKey`; el parámetro textual `phylum` es ignorado por la API y mezclaría fauna y hongos).*
* **Cobertura exacta y paginación:** Los hexágonos adyacentes se fusionan en parches conectados (con una tolerancia de $1\text{ m}$ para cerrar micro-brechas de punto flotante) y se consulta GBIF usando el *bounding box* real en WGS84 de cada parche con paginación completa (`offset += 300`), aplicando `occurrenceStatus = 'PRESENT'`, excluyendo fósiles, cultivados/introducidos e incertidumbre $> \texttt{INCERTIDUMBRE\_MAXIMA\_M}$, y deduplicando por `gbif_id`.
* **Asignación espacial:** Mediante una intersección punto-en-polígono (`gpd.sjoin(..., predicate="within")`), cada registro de flora vascular se vincula al `hex_id` que lo contiene.

---

### 8.3. Variables e Índices de Diversidad y Completitud Calculados por Hexágono

#### Variables base dentro de cada hexágono
| Símbolo | Columna / Código | Significado |
| :---: | :--- | :--- |
| $N$ | `n_registros` | Número total de registros de flora vascular dentro del hexágono ($N = \sum_{i=1}^{S_{\text{obs}}} n_i$). |
| $n_i$ | `conteos` | Número de registros correspondientes a la $i$-ésima especie dentro del hexágono. |
| $p_i$ | `p_i = conteos / N` | Proporción (frecuencia relativa) de registros de la especie $i$ respecto al total $N$ ($\sum_{i=1}^{S_{\text{obs}}} p_i = 1$). |
| $S_{\text{obs}}$ | `riqueza_s` | **Riqueza observada:** número de especies vasculares distintas registradas en el hexágono. |
| $f_1$ | `(conteos == 1).sum()` | Número de especies ***singletons*** (registradas exactamente $1$ vez en el hexágono, $n_i = 1$). |
| $f_2$ | `(conteos == 2).sum()` | Número de especies ***doubletons*** (registradas exactamente $2$ veces en el hexágono, $n_i = 2$). |
| `confiable` | `1` si $N \ge \texttt{N\_MIN\_ESFUERZO}$ | Bandera de esfuerzo mínimo (por defecto $N \ge 15$) para evitar interpretar índices calculados sobre muy pocos registros. |

#### Fórmulas e interpretación ecológica
1. **Índice de diversidad de Shannon-Wiener ($H'$, `shannon_h`):**
   $$H' = -\sum_{i=1}^{S_{\text{obs}}} p_i \ln(p_i)$$
   Combina riqueza y uniformidad (expresado en *nats* al usar logaritmo natural $\ln$). Vale $0$ cuando solo hay una especie y aumenta al crecer el número de especies y la equitatividad entre ellas (en comunidades vegetales suele oscilar entre $1{,}5$ y $3{,}5$).

2. **Índice de equitatividad de Pielou ($J'$, `pielou_j`):**
   $$J' = \begin{cases} \dfrac{H'}{\ln(S_{\text{obs}})} & \text{si } S_{\text{obs}} > 1 \\ \text{NaN (indeterminado)} & \text{si } S_{\text{obs}} \le 1 \end{cases}$$
   Normaliza $H'$ respecto a la diversidad máxima posible $\ln(S_{\text{obs}})$. Varía entre $0$ (dominancia extrema de una especie) y $1$ (todas las especies igual de representadas). Cuando $S_{\text{obs}} \le 1$, el cociente $0/0$ es indeterminado y se reporta como `NaN`.

3. **Índice de Gini-Simpson ($1 - D$, `simpson_1_d`):**
   $$1 - D = 1 - \sum_{i=1}^{S_{\text{obs}}} p_i^2$$
   Representa la probabilidad de que dos registros seleccionados al azar dentro del hexágono pertenezcan a especies diferentes. Varía entre $0$ y $1 - 1/S_{\text{obs}}$ y otorga mayor peso a las especies más abundantes/frecuentes que Shannon.

4. **Estimador no paramétrico de riqueza Chao1 ($\hat{S}_{\text{Chao1}}$, `chao1_s`):**
   $$\hat{S}_{\text{Chao1}} = S_{\text{obs}} + \frac{f_1 (f_1 - 1)}{2 (f_2 + 1)}$$
   Estima la riqueza total esperada (especies detectadas más una cota inferior de especies no detectadas) utilizando la proporción de especies raras ($f_1$ y $f_2$). Se emplea la forma corregida por sesgo ($f_2 + 1$ en el denominador), que permanece definida incluso si $f_2 = 0$.

5. **Cobertura de muestra de Chao & Jost (2012) ($\hat{C}$, `cobertura_c`):**
   $$\hat{C} = 1 - \frac{f_1}{N}\left[\frac{(N - 1)\, f_1}{(N - 1)\, f_1 + 2 f_2}\right]$$
   Estima la proporción ($0$ a $1$) de la comunidad representada por las especies ya observadas en el hexágono. Valores cercanos a $1$ indican un inventario más completo; valores bajos ($< 0{,}5$) advierten submuestreo.

> **Nota metodológica para la discusión:** Los datos de GBIF provienen de fuentes heterogéneas (herbarios y ciencia ciudadana) y **no** de parcelas estandarizadas de abundancia. Por ello, $H'$, $J'$ y $1-D$ deben interpretarse como **diversidad aparente basada en registros**, siempre analizada en conjunto con el esfuerzo muestral ($N$) y la cobertura de muestra ($\hat{C}$).

In [ ]:
# 1. Generación de malla hexagonal regular (orientación pointy-top) en LAEA Equal-Area
# Buffer circular de radio BUFFER_LOCAL_M (7500 m) alrededor de cada localidad de referencia
area_estudio_local = gdf_sitios_ea.geometry.buffer(BUFFER_LOCAL_M).union_all()

# Espaciado entre centros para teselación continua sin huecos ni solapamientos:
dx = np.sqrt(3) * RADIO_HEXAGONO_M  # Paso horizontal entre centros de una misma fila (~4330 m)
dy = 1.5 * RADIO_HEXAGONO_M         # Paso vertical entre filas consecutivas (3750 m)

# Límites del rectángulo envolvente alineados a la grilla
xmin, ymin, xmax, ymax = area_estudio_local.bounds
x_start = np.floor(xmin / dx) * dx
y_start = np.floor(ymin / dy) * dy

# Desplazamientos relativos de los 6 vértices (pi/6 = 30° deja vértices arriba/abajo: pointy-top)
angulos = np.linspace(np.pi / 6, 2 * np.pi + np.pi / 6, 7)
offsets_x = RADIO_HEXAGONO_M * np.cos(angulos)
offsets_y = RADIO_HEXAGONO_M * np.sin(angulos)

hex_geoms = []
y_act = y_start - dy
fila = 0
while y_act <= ymax + dy:
    # Desfase alternado de dx/2 en filas impares para encaje tipo panal
    offset_x = (dx / 2) if (fila % 2 == 1) else 0.0
    x_act = x_start - dx + offset_x
    while x_act <= xmax + dx:
        poly = Polygon([(x_act + ox, y_act + oy) for ox, oy in zip(offsets_x, offsets_y)])
        # Se conservan solo los hexágonos que intersectan el área de influencia de las localidades
        if poly.intersects(area_estudio_local):
            hex_geoms.append(poly)
        x_act += dx
    y_act += dy
    fila += 1

# Área exacta de cada hexágono regular: A_hex = (3*sqrt(3)/2) * r^2
area_hex_km2 = round((3 * np.sqrt(3) / 2) * ((RADIO_HEXAGONO_M / 1000.0) ** 2), 3)
gdf_hex_ea = gpd.GeoDataFrame(
    {"hex_id": range(1, len(hex_geoms) + 1), "area_hex_km2": area_hex_km2, "geometry": hex_geoms},
    crs=CRS_EQUAL_AREA
)

# 2. Parches conectados de hexágonos -> bounding boxes exactos en WGS84 para GBIF
# (buffer de 1 m fusiona bordes adyacentes evitando micro-brechas de punto flotante)
union_hex = gdf_hex_ea.geometry.buffer(1.0).union_all()
parches_ea = list(union_hex.geoms) if union_hex.geom_type == "MultiPolygon" else [union_hex]
gdf_parches_wgs84 = gpd.GeoDataFrame(geometry=parches_ea, crs=CRS_EQUAL_AREA).to_crs("EPSG:4326")

PHYLUM_KEY_TRACHEOPHYTA = 7707728
registros_flora = []
parches_truncados = 0

print(f"Malla generada: {len(gdf_hex_ea)} hexágonos ({area_hex_km2:.2f} km² c/u) agrupados en {len(gdf_parches_wgs84)} parches.")
print(f"Descargando flora vascular (Tracheophyta, phylumKey={PHYLUM_KEY_TRACHEOPHYTA}) con paginación completa...")

for p_idx, geom_p in enumerate(gdf_parches_wgs84.geometry, start=1):
    minx, miny, maxx, maxy = geom_p.bounds
    margen = 0.005
    lat_rng = f"{miny - margen:.5f},{maxy + margen:.5f}"
    lon_rng = f"{minx - margen:.5f},{maxx + margen:.5f}"

    offset = 0
    limit = 300
    total_parche = None

    while True:
        params = [
            ("decimalLatitude", lat_rng),
            ("decimalLongitude", lon_rng),
            ("phylumKey", str(PHYLUM_KEY_TRACHEOPHYTA)),
            ("hasCoordinate", "true"),
            ("hasGeospatialIssue", "false"),
            ("occurrenceStatus", "PRESENT"),
            ("limit", str(limit)),
            ("offset", str(offset)),
        ]
        if ESCALA_EVALUACION == "RCE_CHILE":
            params.append(("country", "CL"))

        try:
            resp = SESSION_HTTP.get(URL_GBIF_OCC, params=params, timeout=15)
            if resp.status_code != 200:
                break
            data = resp.json()
        except Exception:
            break

        if total_parche is None:
            total_parche = data.get("count", 0)
        lote = data.get("results", [])
        if not lote:
            break

        for r in lote:
            if r.get("basisOfRecord") == "FOSSIL_SPECIMEN":
                continue
            estab = str(r.get("establishmentMeans", "")).upper()
            deg_estab = str(r.get("degreeOfEstablishment", "")).upper()
            if any(k in estab or k in deg_estab for k in ("CULTIVATED", "INTRODUCED", "MANAGED")):
                continue
            incert = r.get("coordinateUncertaintyInMeters")
            if incert is not None and incert > INCERTIDUMBRE_MAXIMA_M:
                continue
            sp_name = r.get("species")
            if sp_name and r.get("decimalLatitude") is not None and r.get("decimalLongitude") is not None:
                registros_flora.append({
                    "gbif_id": str(r.get("key", "")),
                    "phylum": r.get("phylum", "Tracheophyta"),
                    "family": r.get("family", ""),
                    "species": sp_name,
                    "scientificName": r.get("scientificName", sp_name),
                    "lat": float(r["decimalLatitude"]),
                    "lon": float(r["decimalLongitude"]),
                })

        offset += limit
        if data.get("endOfRecords", False) or offset >= total_parche:
            break
        if offset >= MAX_REGISTROS_FLORA_PARCHE:
            parches_truncados += 1
            print(f"  Aviso: Parche {p_idx} alcanzó el tope de {MAX_REGISTROS_FLORA_PARCHE} registros (total disponible en GBIF: {total_parche}).")
            break

# 3. Deduplicación por gbif_id y cálculo de índices ecológicos
df_flora = pd.DataFrame(registros_flora)
if not df_flora.empty:
    df_flora = df_flora.drop_duplicates(subset=["gbif_id"]).reset_index(drop=True)
    gdf_flora_ea = gpd.GeoDataFrame(
        df_flora[["gbif_id", "phylum", "family", "species", "scientificName"]],
        geometry=gpd.points_from_xy(df_flora["lon"], df_flora["lat"]),
        crs="EPSG:4326"
    ).to_crs(CRS_EQUAL_AREA)

    # Restringir a puntos que efectivamente caen dentro de los hexágonos
    join_pts = gpd.sjoin(gdf_flora_ea, gdf_hex_ea[["hex_id", "geometry"]], how="inner", predicate="within")
    gdf_flora_dentro_ea = gdf_flora_ea.loc[join_pts.index.unique()].copy()

    def calcular_diversidad_hex(group):
        conteos = group["species"].value_counts()
        N = int(conteos.sum())
        S = int(len(conteos))
        p_i = conteos / N

        H = float(-np.sum(p_i * np.log(p_i))) if S > 1 else 0.0
        J = float(H / np.log(S)) if S > 1 else np.nan
        simpson = float(1.0 - np.sum(p_i ** 2))

        f1 = int((conteos == 1).sum())
        f2 = int((conteos == 2).sum())
        chao1 = float(S + (f1 * (f1 - 1)) / (2.0 * (f2 + 1))) if S > 0 else 0.0

        if N > 1 and f1 > 0:
            denom = (N - 1) * f1 + 2 * f2
            cobertura = float(1.0 - (f1 / N) * (((N - 1) * f1) / denom)) if denom > 0 else 0.0
        elif N > 0 and f1 == 0:
            cobertura = 1.0
        else:
            cobertura = np.nan

        return pd.Series({
            "n_registros": N,
            "riqueza_s": S,
            "shannon_h": round(H, 3),
            "pielou_j": round(J, 3) if pd.notna(J) else np.nan,
            "simpson_1_d": round(simpson, 3),
            "chao1_s": round(chao1, 2),
            "cobertura_c": round(cobertura, 3) if pd.notna(cobertura) else np.nan,
            "confiable": 1 if N >= N_MIN_ESFUERZO else 0,
            "sp_dominante": conteos.index[0],
        })

    if not join_pts.empty:
        metricas_hex = join_pts.groupby("hex_id").apply(calcular_diversidad_hex, include_groups=False).reset_index()
        gdf_hex_final_ea = gdf_hex_ea.merge(metricas_hex, on="hex_id", how="left")
        gdf_hex_final_ea = gdf_hex_final_ea.fillna({
            "n_registros": 0,
            "riqueza_s": 0,
            "shannon_h": 0.0,
            "simpson_1_d": 0.0,
            "chao1_s": 0.0,
            "confiable": 0,
            "sp_dominante": "Sin registros",
        })
        gdf_hex_final_ea["n_registros"] = gdf_hex_final_ea["n_registros"].astype(int)
        gdf_hex_final_ea["riqueza_s"] = gdf_hex_final_ea["riqueza_s"].astype(int)
        gdf_hex_final_ea["confiable"] = gdf_hex_final_ea["confiable"].astype(int)
    else:
        gdf_flora_dentro_ea = gpd.GeoDataFrame(columns=["gbif_id", "phylum", "family", "species", "scientificName", "geometry"], crs=CRS_EQUAL_AREA)
        gdf_hex_final_ea = gdf_hex_ea.assign(
            n_registros=0, riqueza_s=0, shannon_h=0.0, pielou_j=np.nan,
            simpson_1_d=0.0, chao1_s=0.0, cobertura_c=np.nan, confiable=0, sp_dominante="Sin registros"
        )
else:
    gdf_flora_dentro_ea = gpd.GeoDataFrame(columns=["gbif_id", "phylum", "family", "species", "scientificName", "geometry"], crs=CRS_EQUAL_AREA)
    gdf_hex_final_ea = gdf_hex_ea.assign(
        n_registros=0, riqueza_s=0, shannon_h=0.0, pielou_j=np.nan,
        simpson_1_d=0.0, chao1_s=0.0, cobertura_c=np.nan, confiable=0, sp_dominante="Sin registros"
    )

hex_con_datos = gdf_hex_final_ea[gdf_hex_final_ea["n_registros"] > 0]
hex_confiables = gdf_hex_final_ea[gdf_hex_final_ea["confiable"] == 1]

print(f"\nResumen de flora vascular acompañante (Tracheophyta):")
print(f"  Registros únicos dentro de hexágonos : {len(gdf_flora_dentro_ea)}")
print(f"  Especies vasculares registradas      : {gdf_flora_dentro_ea['species'].nunique() if not gdf_flora_dentro_ea.empty else 0}")
print(f"  Hexágonos con datos / confiables     : {len(hex_con_datos)} / {len(hex_confiables)} (umbral N ≥ {N_MIN_ESFUERZO})")
if not hex_confiables.empty:
    print(f"  Shannon (H') promedio (confiables)   : {hex_confiables['shannon_h'].mean():.3f}")
    print(f"  Pielou (J') promedio (confiables)    : {hex_confiables['pielou_j'].mean():.3f}")
    print(f"  Simpson (1-D) promedio (confiables)  : {hex_confiables['simpson_1_d'].mean():.3f}")
    print(f"  Cobertura (C) promedio (confiables)  : {hex_confiables['cobertura_c'].mean():.3f}")

## 9. Exportación a GeoPackage (`.gpkg` para QGIS) y Reporte Consolidado

Se eliminan versiones previas del archivo `.gpkg` para evitar capas huérfanas y se exportan todas las capas reproyectadas a **UTM Chile (`EPSG:32718` o `EPSG:32719`)**, conservando intactas las áreas calculadas en LAEA *Equal-Area*:

| Capa en el GeoPackage | Contenido |
| :--- | :--- |
| `iucn_eoo_oficial` | Polígono EOO oficial B1 (MCP nacional Chile en modo `RCE_CHILE`, o MCP global en `UICN_GLOBAL`). |
| `eoo_subpoblaciones_biogeograficas` | Polígonos separados por núcleo (Costa / Depresión Intermedia / Andes) para especies disyuntas. |
| `iucn_aoo_2x2km_oficial` | Grilla estándar UICN/RCE de $2 \times 2\text{ km}$ optimizada (mínimo de celdas) para Criterio B2. |
| `aoo_escala_fina_literatura` | Grilla complementaria ($1 \times 1\text{ km}$) para especies con referencia RECOGE (*Queule*, *Ruil*). |
| `ocurrencias_especie_validadas` | Registros curados de GBIF + localidades de terreno utilizadas en EOO y AOO. |
| `localidades_referencia` | Sitios del estudio (`ID`, `Referencia`, `Codigo`, `Lat`, `Lon`). |
| `hex_biodiversidad_local` | Malla hexagonal con $N$, $S_{\text{obs}}$, $H'$, $J'$, $1-D$, $S_{\text{Chao1}}$, $\hat{C}$ y especie vascular dominante. |
| `ocurrencias_flora_tracheophyta` | Registros de flora vascular acompañante (`phylumKey=7707728`) dentro de los hexágonos. |

In [ ]:
if os.path.exists(ARCHIVO_SALIDA):
    os.remove(ARCHIVO_SALIDA)

# Reproyección a UTM Chile para interoperabilidad con capas de QGIS
gdf_hex_final_ea.to_crs(CRS_EXPORT_UTM).to_file(ARCHIVO_SALIDA, layer="hex_biodiversidad_local", driver="GPKG")
gdf_sitios_ea.to_crs(CRS_EXPORT_UTM).to_file(ARCHIVO_SALIDA, layer="localidades_referencia", driver="GPKG")
gdf_eoo_oficial_ea.to_crs(CRS_EXPORT_UTM).to_file(ARCHIVO_SALIDA, layer="iucn_eoo_oficial", driver="GPKG")
if gdf_eoo_subpob_ea is not None:
    gdf_eoo_subpob_ea.to_crs(CRS_EXPORT_UTM).to_file(ARCHIVO_SALIDA, layer="eoo_subpoblaciones_biogeograficas", driver="GPKG")
gdf_aoo_uicn_ea.to_crs(CRS_EXPORT_UTM).to_file(ARCHIVO_SALIDA, layer="iucn_aoo_2x2km_oficial", driver="GPKG")
if gdf_aoo_lit_ea is not None:
    gdf_aoo_lit_ea.to_crs(CRS_EXPORT_UTM).to_file(ARCHIVO_SALIDA, layer="aoo_escala_fina_literatura", driver="GPKG")
gdf_sp_ea.to_crs(CRS_EXPORT_UTM).to_file(ARCHIVO_SALIDA, layer="ocurrencias_especie_validadas", driver="GPKG")
if not gdf_flora_dentro_ea.empty:
    gdf_flora_dentro_ea.to_crs(CRS_EXPORT_UTM).to_file(ARCHIVO_SALIDA, layer="ocurrencias_flora_tracheophyta", driver="GPKG")

# Tabla resumen lista para informes y presentaciones
resumen_rows = [
    ("Especie / Taxón", f"{ESPECIE_SELECCIONADA} ({nombre_cientifico})"),
    ("Escala de evaluación", f"{ESCALA_EVALUACION} ({', '.join(PAISES_CONSULTA)})"),
    ("Ocurrencias nativas únicas", f"{len(gdf_sp_ea)} (GBIF + CSV)" if INCLUIR_SITIOS_CSV_EN_UICN else f"{len(gdf_sp_ea)} (GBIF)"),
    ("EOO Oficial (Criterio B1)", f"{eoo_oficial_km2:,.2f} km² — {evaluar_umbral_b1(eoo_oficial_km2)}"),
    ("EOO MCP bruto (sin recorte marino)", f"{eoo_mcp_bruto_km2:,.2f} km²"),
    ("AOO Oficial 2×2 km (Criterio B2)", f"{stats_aoo_uicn['aoo_km2']:,.2f} km² ({stats_aoo_uicn['n_celdas_min']} celdas mín.) — {evaluar_umbral_b2(stats_aoo_uicn['aoo_km2'])}"),
]
if stats_aoo_lit is not None:
    resumen_rows.append((
        f"AOO Literatura ({grid_lit_m}×{grid_lit_m} m)",
        f"{stats_aoo_lit['aoo_km2']:,.2f} km² ({stats_aoo_lit['n_celdas_min']} celdas mín.; complementario, no B2)"
    ))
resumen_rows.extend([
    ("Referencia oficial / literatura", AOO_LITERATURA_CONFIG[ESPECIE_SELECCIONADA]["referencia"]),
    ("Flora vascular (Tracheophyta)", f"{len(gdf_flora_dentro_ea)} registros | {gdf_flora_dentro_ea['species'].nunique() if not gdf_flora_dentro_ea.empty else 0} especies en {len(hex_con_datos)} hexágonos"),
    ("GeoPackage exportado (CRS)", f"{ARCHIVO_SALIDA} ({CRS_EXPORT_UTM}; áreas calculadas en LAEA Equal-Area)"),
])

df_resumen = pd.DataFrame(resumen_rows, columns=["Parámetro", "Resultado"])
print(f"\n================ REPORTE CONSOLIDADO ({ESPECIE_SELECCIONADA} — {ESCALA_EVALUACION}) ================")
for k, v in resumen_rows:
    print(f"  {k:<36}: {v}")
if ESCALA_EVALUACION == "RCE_CHILE" and ESPECIE_SELECCIONADA in ("Alerce", "Araucaria", "Lleuque"):
    print("\n  [Nota RCE Paso 2]: Especie compartida con Argentina. Para clasificación regional definitiva,")
    print("  evaluar conectividad y eventual efecto de rescate desde poblaciones argentinas (UICN Regional v4.0).")

display(df_resumen)

## 10. Visualización Interactiva con Selector de Capas (`Folium`)

Utiliza el panel de capas (esquina superior derecha del mapa) para alternar los **mapas base** (compatibles con Google Colab sin bloqueo de tiles ni API key: **Mapa Claro CartoDB**, **Topográfico Esri** y **Satélite Esri**) y activar o desactivar independientemente:
* Malla hexagonal de diversidad de flora vascular ($H'$ de Shannon)
* EOO Oficial (B1) y núcleos biogeográficos complementarios
* AOO Oficial $2 \times 2\text{ km}$ (B2) y AOO fino de literatura ($1 \times 1\text{ km}$, cuando aplica)
* Ocurrencias validadas y localidades de referencia (`Codigo` y `Referencia`)

In [ ]:
centro_mapa = [lat_centro, lon_centro]
# tiles=None evita el tile por defecto de OpenStreetMap (bloqueado en iframes de Google Colab)
m = folium.Map(location=centro_mapa, zoom_start=8, tiles=None)

# Mapas base compatibles con Google Colab (sin API key ni bloqueo por Referer)
folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
    attr="&copy; OpenStreetMap contributors &copy; CARTO",
    name="Mapa Claro (CartoDB Positron)",
    show=True,
).add_to(m)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles &copy; Esri &mdash; Esri, DeLorme, NAVTEQ, USGS, NASA",
    name="Topográfico (Esri World Topo)",
    show=False,
).add_to(m)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles &copy; Esri &mdash; Source: Esri, Maxar, Earthstar Geographics",
    name="Satélite (Esri World Imagery)",
    show=False,
).add_to(m)

# 1. Malla hexagonal de biodiversidad (Shannon H')
gdf_hex_final_ea.to_crs("EPSG:4326").explore(
    m=m,
    name="Biodiversidad Flora Vascular (Shannon H')",
    column="shannon_h",
    cmap="viridis",
    tooltip=["hex_id", "n_registros", "riqueza_s", "shannon_h", "pielou_j", "simpson_1_d", "chao1_s", "cobertura_c", "confiable", "sp_dominante"],
    popup=True,
    legend=True,
    style_kwds={"fillOpacity": 0.55, "weight": 0.8},
)

# 2. EOO Oficial (B1)
gdf_eoo_oficial_ea.to_crs("EPSG:4326").explore(
    m=m,
    name=f"EOO Oficial B1 ({eoo_oficial_km2:,.0f} km²)",
    color="darkorange",
    style_kwds={"fill": True, "fillOpacity": 0.12, "weight": 2.5},
    tooltip=["especie", "escala", "eoo_km2", "umbral_preliminar_b1"],
)

# 3. Núcleos biogeográficos (si es disyunta)
if gdf_eoo_subpob_ea is not None:
    gdf_eoo_subpob_ea.to_crs("EPSG:4326").explore(
        m=m,
        name="Núcleos Biogeográficos (Complementario)",
        color="royalblue",
        style_kwds={"fill": True, "fillOpacity": 0.12, "weight": 2.0, "dashArray": "5, 5"},
        tooltip=["subpoblacion", "n_registros", "area_nucleo_km2"],
        show=False,
    )

# 4. AOO Oficial 2x2 km (B2)
gdf_aoo_uicn_ea.to_crs("EPSG:4326").explore(
    m=m,
    name=f"AOO Oficial 2×2 km ({stats_aoo_uicn['aoo_km2']:,.0f} km²)",
    color="crimson",
    style_kwds={"fillOpacity": 0.45, "weight": 1.0},
    tooltip=["celda_id", "resolucion_m", "aoo_total_km2", "umbral_preliminar_b2"],
)

# 5. AOO Escala Fina de Literatura (cuando aplica, ej. Queule / Ruil)
if gdf_aoo_lit_ea is not None:
    gdf_aoo_lit_ea.to_crs("EPSG:4326").explore(
        m=m,
        name=f"AOO Literatura {grid_lit_m}×{grid_lit_m} m ({stats_aoo_lit['aoo_km2']:,.0f} km²)",
        color="purple",
        style_kwds={"fillOpacity": 0.55, "weight": 1.0},
        tooltip=["celda_id", "resolucion_m", "aoo_total_km2", "nota_escala"],
        show=False,
    )

# 6. Ocurrencias validadas de la especie focal
gdf_sp_ea.to_crs("EPSG:4326").explore(
    m=m,
    name=f"Ocurrencias {ESPECIE_SELECCIONADA} ({len(gdf_sp_ea)})",
    color="forestgreen",
    marker_type="circle_marker",
    marker_kwds={"radius": 3.5, "fill_opacity": 0.8},
    tooltip=["species", "subpoblacion", "fuente", "year", "uncertainty_m"],
    show=False,
)

# 7. Localidades de referencia
tooltip_sitios = [c for c in ["Codigo", "Referencia", "ID"] if c in gdf_sitios_ea.columns]
gdf_sitios_ea.to_crs("EPSG:4326").explore(
    m=m,
    name="Localidades de Referencia",
    color="black",
    marker_type="circle_marker",
    marker_kwds={"radius": 6, "fill_opacity": 1.0},
    tooltip=tooltip_sitios,
)

folium.LayerControl(collapsed=False).add_to(m)
mapa_base_html = f"mapa_interactivo_{ESPECIE_SELECCIONADA.lower()}_{sufijo_escala}.html"
m.save(mapa_base_html)
print(f"Mapa interactivo guardado como archivo independiente: '{mapa_base_html}'")
display(m)

---

## 11. Módulo Opcional: Espacialización de Estructura Genética Poblacional (`STRUCTURE`)

Este módulo integra los resultados de agrupamiento bayesiano generados en **STRUCTURE** (Pritchard et al., 2000; Falush et al., 2003) directamente con las localidades de referencia (`Coordenadas_codigo.csv`), el EOO/AOO y los mapas interactivos y de publicación.

### 11.1. Marco Metodológico y Fórmulas

1. **Matriz de ancestría ($Q$):** Para un número asumido de grupos genéticos $K$, STRUCTURE estima la proporción de ancestría $q_{ik} \in [0, 1]$ de cada individuo $i$ en cada *cluster* $k$ ($\sum_{k=1}^{K} q_{ik} = 1$), y el promedio poblacional $\bar{Q}_{pk}$ para cada localidad $p \in \{1, \dots, P\}$.
2. **Selección del $K$ óptimo — Método de Evanno et al. (2005):**  
   A partir del archivo `.zip` con todas las corridas (`*_f`) de la carpeta `Results/` de STRUCTURE, sea $L_r(K) = \ln P(D)$ la verosimilitud estimada en la réplica $r$ para un valor de $K$, con media $\overline{L}(K)$ y desviación estándar muestral $s(L(K))$:
   $$L'(K) = \overline{L}(K) - \overline{L}(K - 1), \qquad |L''(K)| = \left|L'(K + 1) - L'(K)\right| = \left|\overline{L}(K + 1) - 2\overline{L}(K) + \overline{L}(K - 1)\right|$$
   $$\Delta K = \frac{|L''(K)|}{s\!\left(L(K)\right)}$$
   > **Nota:** $\Delta K$ no está definido para $K = 1$ ni para $K_{\max}$, y tiende a capturar el nivel jerárquico más alto de estructuración (frecuentemente $K = 2$). Por ello, este módulo permite **inspeccionar tanto $\Delta K$ como $\overline{L}(K) \pm s$** y elegir **manualmente** cualquier `K_SELECCIONADO` con sentido biológico/biogeográfico.
3. **Consenso entre réplicas (alineación tipo CLUMPP):** Para un mismo $K$, distintas iteraciones pueden permutar el número asignado a cada cluster (*label switching*). El modo `CONSENSO_CLUMPP` alinea automáticamente todas las réplicas contra la corrida de mayor $\ln P(D)$ mediante el algoritmo de asignación óptima húngara (`scipy.optimize.linear_sum_assignment`), promedia las matrices $Q$ alineadas y ordena los clusters geográficamente de **Norte a Sur** para que los colores fluyan de manera coherente en el mapa y el barplot.
4. **Métricas poblacionales derivadas:**
   * **Ancestría dominante:** $Q_{\max, p} = \max_{k} \bar{Q}_{pk}$. Se clasifica operacionalmente cada localidad como **Pura** ($Q_{\max} \ge 0{,}80$) o **Zona de mezcla / transición** ($Q_{\max} < 0{,}80$).
   * **Entropía de mezcla genética:** $H_{Q, p} = -\sum_{k=1}^{K} \bar{Q}_{pk} \ln(\bar{Q}_{pk})$ (vale $0$ si la población pertenece $100\%$ a un único cluster y aumenta con la mezcla).
5. **Discusión crítica sobre interpolación espacial IDW (*tipo `tess3r`*):**  
   Dado que los individuos comparten la coordenada de su **localidad de referencia** ($P = 8\text{–}12$ sitios), una interpolación continua por distancia inversa ponderada ($w_j = d_j^{-p}$) asume continuidad isotrópica y puede suavizar artificialmente barreras reales (como el Valle Central en especies disyuntas) o extrapolar fuera del muestreo. Por ello, **los diagramas de torta discretos por localidad son la representación principal recomendada**, mientras que el IDW se ofrece **apagado por defecto (`MOSTRAR_IDW_ANCESTRIA = False`)** con doble enmascaramiento (recortado a los núcleos/EOO terrestre y a un radio máximo alrededor de las localidades muestreadas) y opacidad modulada por la certeza $\frac{Q_{\max} - 1/K}{1 - 1/K}$.

In [ ]:
#@title Parámetros del Módulo 11: Estructura Genética (STRUCTURE) { display-mode: "form" }
ACTIVAR_MODULO_STRUCTURE = True  #@param {type:"boolean"}
ARCHIVO_STRUCTURE = "Results_lleuque.zip"  #@param {type:"string"}
K_SELECCIONADO = 0  #@param {type:"integer"}
MODO_REPLICAS = "CONSENSO_CLUMPP"  #@param ["CONSENSO_CLUMPP", "MEJOR_LNPD"]
ORDEN_BARPLOT = "NORTE_A_SUR"  #@param ["NORTE_A_SUR", "ID_ORIGINAL"]
MOSTRAR_IDW_ANCESTRIA = False  #@param {type:"boolean"}
RADIO_MASCARA_IDW_KM = 60  #@param {type:"integer"}
POTENCIA_IDW = 2.0  #@param {type:"number"}

# Paleta de alto contraste y accesible para daltonismo (los dos primeros tonos evocan púrpura/verde clásico)
PALETA_CLUSTERS = [
    "#4B0055",  # K1: Púrpura oscuro
    "#1B7837",  # K2: Verde bosque
    "#D95F02",  # K3: Naranjo quemado
    "#2166AC",  # K4: Azul profundo
    "#E6AB02",  # K5: Mostaza / dorado
    "#B2182B",  # K6: Carmesí
    "#01665E",  # K7: Verde petróleo
    "#7570B3",  # K8: Lavanda oscuro
    "#A65628",  # K9: Terracota
    "#666666",  # K10: Gris grafito
]

def parsear_archivo_structure_f(texto, nombre_archivo="run_f"):
    '''Parsea un archivo crudo *_f de STRUCTURE 2.3.x devolviendo metadatos, Q poblacional y Q individual.'''
    m_k = re.search(r"^\s*(\d+)\s+populations assumed", texto, re.MULTILINE)
    m_lnpd = re.search(r"Estimated Ln Prob of Data\s*=\s*([-\d\.]+)", texto)
    if not m_k or not m_lnpd:
        return None

    K = int(m_k.group(1))
    lnpd = float(m_lnpd.group(1))
    m_alpha = re.search(r"Mean value of alpha\s*=\s*([-\d\.]+)", texto)
    alpha = float(m_alpha.group(1)) if m_alpha else np.nan
    m_burn = re.search(r"^\s*(\d+)\s+Burn-in period", texto, re.MULTILINE)
    m_reps = re.search(r"^\s*(\d+)\s+Reps", texto, re.MULTILINE)
    fsts = {int(k): float(v) for k, v in re.findall(r"Mean value of Fst_(\d+)\s*=\s*([-\d\.]+)", texto)}

    # 1. Tabla poblacional (Proportion of membership of each pre-defined population)
    pop_rows = []
    in_pop = False
    for line in texto.splitlines():
        if "Given" in line and "Inferred Clusters" in line:
            in_pop = True
            continue
        if in_pop:
            if line.strip().startswith("---"):
                if pop_rows:
                    break
                continue
            m_pop = re.match(r"^\s*(\d+)\s*:\s*(.+)$", line)
            if m_pop:
                pop_id = int(m_pop.group(1))
                nums = re.findall(r"[-+]?\d*\.\d+|\d+", m_pop.group(2))
                if len(nums) >= K + 1:
                    q_vals = [float(x) for x in nums[:K]]
                    n_ind = int(float(nums[K]))
                    pop_rows.append({"Pop": pop_id, "n_ind": n_ind, **{f"K{i+1}": q_vals[i] for i in range(K)}})

    # 2. Tabla individual (Inferred ancestry of individuals)
    ind_rows = []
    in_ind = False
    for line in texto.splitlines():
        if "Inferred ancestry of individuals:" in line:
            in_ind = True
            continue
        if in_ind:
            s = line.strip()
            if not s:
                if ind_rows:
                    break
                continue
            if "Label" in s and "Pop:" in s:
                continue
            m_ind = re.match(r"^\s*(\d+)\s+(\S+)\s+\((\d+)\)\s+(\d+)\s*:\s*(.+)$", line)
            if m_ind:
                ind_id = int(m_ind.group(1))
                label = m_ind.group(2)
                pct_miss = int(m_ind.group(3))
                pop_id = int(m_ind.group(4))
                # Extrae únicamente los K valores principales (ignorando intervalos entre paréntesis si existieran)
                sin_paren = re.sub(r"\([^)]*\)", "", m_ind.group(5))
                nums = re.findall(r"[-+]?\d*\.\d+|\d+", sin_paren)
                if len(nums) >= K:
                    q_vals = [float(x) for x in nums[:K]]
                    ind_rows.append({
                        "ind_id": ind_id, "label": label, "pct_miss": pct_miss, "Pop": pop_id,
                        **{f"K{i+1}": q_vals[i] for i in range(K)}
                    })

    if not ind_rows and not pop_rows:
        return None

    df_ind = pd.DataFrame(ind_rows)
    df_pop = pd.DataFrame(pop_rows)
    if df_pop.empty and not df_ind.empty:
        k_cols = [f"K{i+1}" for i in range(K)]
        df_pop = df_ind.groupby("Pop", as_index=False)[k_cols].mean()
        df_pop["n_ind"] = df_ind.groupby("Pop").size().values

    return {
        "archivo": nombre_archivo,
        "K": K,
        "lnpd": lnpd,
        "alpha": alpha,
        "burnin": int(m_burn.group(1)) if m_burn else None,
        "reps": int(m_reps.group(1)) if m_reps else None,
        "fsts": fsts,
        "df_pop": df_pop,
        "df_ind": df_ind,
    }

def calcular_evanno(runs):
    '''Calcula la tabla de Evanno et al. (2005): Mean LnP(D), SD, L'(K), |L''(K)| y Delta K.'''
    df_runs = pd.DataFrame([{"K": r["K"], "lnpd": r["lnpd"], "archivo": r["archivo"]} for r in runs])
    res = df_runs.groupby("K", as_index=False).agg(
        n_reps=("lnpd", "count"),
        mean_lnpd=("lnpd", "mean"),
        sd_lnpd=("lnpd", lambda x: float(np.std(x, ddof=1)) if len(x) > 1 else np.nan),
        max_lnpd=("lnpd", "max"),
    ).sort_values("K").reset_index(drop=True)

    res["L_prime"] = np.nan
    res["abs_L_double_prime"] = np.nan
    res["delta_K"] = np.nan

    ks = res["K"].tolist()
    means = res["mean_lnpd"].tolist()
    sds = res["sd_lnpd"].tolist()

    for i in range(len(ks)):
        if i >= 1 and ks[i] - ks[i - 1] == 1:
            res.loc[i, "L_prime"] = means[i] - means[i - 1]
    for i in range(1, len(ks) - 1):
        if ks[i] - ks[i - 1] == 1 and ks[i + 1] - ks[i] == 1:
            l_dd = abs(means[i + 1] - 2.0 * means[i] + means[i - 1])
            res.loc[i, "abs_L_double_prime"] = l_dd
            if pd.notna(sds[i]) and sds[i] > 0:
                res.loc[i, "delta_K"] = l_dd / sds[i]

    return res

def alinear_consenso_clumpp(runs_k, K, modo="CONSENSO_CLUMPP"):
    '''Alinea réplicas de un mismo K mediante asignación húngara óptima (tipo CLUMPP) y calcula consenso.'''
    k_cols = [f"K{i+1}" for i in range(K)]
    runs_ord = sorted(runs_k, key=lambda r: r["lnpd"], reverse=True)
    ref = runs_ord[0]

    if modo == "MEJOR_LNPD" or len(runs_ord) == 1 or K == 1:
        return ref["df_pop"].copy(), ref["df_ind"].copy(), 1.0, ref

    Q_ref_ind = ref["df_ind"][k_cols].to_numpy(dtype=float)
    Q_ref_pop = ref["df_pop"][k_cols].to_numpy(dtype=float)

    ind_acum = Q_ref_ind.copy()
    pop_acum = Q_ref_pop.copy()
    correlaciones = []

    for r in runs_ord[1:]:
        Q_ind = r["df_ind"][k_cols].to_numpy(dtype=float)
        Q_pop = r["df_pop"][k_cols].to_numpy(dtype=float)
        if Q_ind.shape != Q_ref_ind.shape:
            continue
        # Matriz de costo L1 entre clusters de referencia y de la réplica
        cost = np.zeros((K, K), dtype=float)
        for a in range(K):
            for b in range(K):
                cost[a, b] = np.mean(np.abs(Q_ref_ind[:, a] - Q_ind[:, b]))
        _, col_perm = linear_sum_assignment(cost)
        Q_ind_al = Q_ind[:, col_perm]
        Q_pop_al = Q_pop[:, col_perm]
        ind_acum += Q_ind_al
        pop_acum += Q_pop_al
        sim = 1.0 - float(np.mean(np.abs(Q_ref_ind - Q_ind_al)))
        correlaciones.append(sim)

    n_usadas = 1 + len(correlaciones)
    Q_ind_mean = ind_acum / n_usadas
    Q_ind_mean = Q_ind_mean / np.clip(Q_ind_mean.sum(axis=1, keepdims=True), 1e-12, None)
    Q_pop_mean = pop_acum / n_usadas
    Q_pop_mean = Q_pop_mean / np.clip(Q_pop_mean.sum(axis=1, keepdims=True), 1e-12, None)

    df_ind_out = ref["df_ind"].copy()
    df_pop_out = ref["df_pop"].copy()
    for i, col in enumerate(k_cols):
        df_ind_out[col] = np.round(Q_ind_mean[:, i], 4)
        df_pop_out[col] = np.round(Q_pop_mean[:, i], 4)

    sim_media = float(np.mean(correlaciones)) if correlaciones else 1.0
    return df_pop_out, df_ind_out, sim_media, ref

# Estado global del módulo
STRUCTURE_LISTO = False
df_evanno = None
gdf_struct_pop_ea = None
gdf_struct_pop_wgs84 = None
df_struct_ind = None
K_ACTIVO = None
K_COLS_ACTIVOS = []
COLORES_ACTIVOS = []

if not ACTIVAR_MODULO_STRUCTURE:
    print("Módulo 11 desactivado (ACTIVAR_MODULO_STRUCTURE = False).")
    print("Para activarlo: marca ACTIVAR_MODULO_STRUCTURE, sube tu .zip de Results/ de STRUCTURE (o archivo *_f) e indica su nombre en ARCHIVO_STRUCTURE.")
else:
    ruta_str = ARCHIVO_STRUCTURE.strip()
    if not ruta_str:
        zips_cand = [f for f in os.listdir(".") if f.lower().endswith(".zip") and "gadm" not in f.lower() and "ne_10m" not in f.lower()]
        if zips_cand:
            ruta_str = zips_cand[0]
            print(f"Auto-detectado archivo de resultados: {ruta_str}")
        else:
            raise FileNotFoundError("Indica el nombre del archivo .zip (o *_f / .csv) en ARCHIVO_STRUCTURE.")
    ruta_str = resolver_archivo_entrada(ruta_str)

    runs_parseados = []
    es_csv_manual = ruta_str.lower().endswith(".csv")

    if es_csv_manual:
        df_csv_q = cargar_csv_localidades(ruta_str)
        k_cols_csv = [c for c in df_csv_q.columns if re.match(r"^[Kk]\d+$", str(c).strip())]
        if not k_cols_csv or "Pop" not in df_csv_q.columns:
            raise ValueError("El CSV debe contener la columna 'Pop' y columnas de ancestría K1, K2, ..., Kk.")
        k_cols_csv = sorted(k_cols_csv, key=lambda c: int(c[1:]))
        K_ACTIVO = len(k_cols_csv)
        K_COLS_ACTIVOS = [f"K{i+1}" for i in range(K_ACTIVO)]
        renombre = {old: new for old, new in zip(k_cols_csv, K_COLS_ACTIVOS)}
        df_struct_ind = df_csv_q.rename(columns=renombre).copy()
        for c in K_COLS_ACTIVOS:
            df_struct_ind[c] = pd.to_numeric(df_struct_ind[c], errors="coerce")
        df_struct_ind["Pop"] = pd.to_numeric(df_struct_ind["Pop"], errors="coerce").astype(int)
        if "ind_id" not in df_struct_ind.columns:
            df_struct_ind["ind_id"] = range(1, len(df_struct_ind) + 1)
        df_pop_cons = df_struct_ind.groupby("Pop", as_index=False)[K_COLS_ACTIVOS].mean()
        df_pop_cons["n_ind"] = df_struct_ind.groupby("Pop").size().values
        sim_consenso = 1.0
        print(f"CSV de STRUCTURE cargado: {len(df_struct_ind)} individuos, {len(df_pop_cons)} poblaciones, K = {K_ACTIVO}.")
    else:
        if zipfile.is_zipfile(ruta_str):
            with zipfile.ZipFile(ruta_str, "r") as zf:
                for member in zf.namelist():
                    base = os.path.basename(member)
                    if not base or "__MACOSX" in member or member.endswith("/"):
                        continue
                    if base.endswith("_f") or "_run_" in base:
                        try:
                            raw_txt = zf.read(member).decode("utf-8", errors="ignore")
                            parsed = parsear_archivo_structure_f(raw_txt, nombre_archivo=base)
                            if parsed is not None:
                                runs_parseados.append(parsed)
                        except Exception:
                            continue
        else:
            raw_txt = open(ruta_str, "r", encoding="utf-8", errors="ignore").read()
            parsed = parsear_archivo_structure_f(raw_txt, nombre_archivo=os.path.basename(ruta_str))
            if parsed is not None:
                runs_parseados.append(parsed)

        if not runs_parseados:
            raise ValueError(f"No se encontraron archivos válidos '*_f' de STRUCTURE dentro de '{ruta_str}'.")

        ks_disponibles = sorted(set(r["K"] for r in runs_parseados))
        print(f"Archivos STRUCTURE parseados: {len(runs_parseados)} corridas | Valores de K disponibles: {ks_disponibles}")

        df_evanno = calcular_evanno(runs_parseados)
        k_evanno_opt = None
        if df_evanno["delta_K"].notna().any():
            idx_max_dk = df_evanno["delta_K"].idxmax()
            k_evanno_opt = int(df_evanno.loc[idx_max_dk, "K"])
            print(f"K óptimo sugerido por Evanno (Delta K máximo): K = {k_evanno_opt} (Delta K = {df_evanno.loc[idx_max_dk, 'delta_K']:.2f})")
            display(df_evanno.round(3))

        if K_SELECCIONADO > 0:
            if K_SELECCIONADO not in ks_disponibles:
                raise ValueError(f"K_SELECCIONADO={K_SELECCIONADO} no existe en el archivo. K disponibles: {ks_disponibles}")
            K_ACTIVO = K_SELECCIONADO
            print(f"Usando K elegido manualmente: K = {K_ACTIVO}")
        else:
            K_ACTIVO = k_evanno_opt if k_evanno_opt is not None else max(ks_disponibles)
            print(f"Usando K seleccionado automáticamente: K = {K_ACTIVO}")

        runs_k = [r for r in runs_parseados if r["K"] == K_ACTIVO]
        K_COLS_ACTIVOS = [f"K{i+1}" for i in range(K_ACTIVO)]
        df_pop_cons, df_struct_ind, sim_consenso, ref_run = alinear_consenso_clumpp(
            runs_k, K_ACTIVO, modo=MODO_REPLICAS
        )
        print(f"Modo réplicas: {MODO_REPLICAS} ({len(runs_k)} réplicas para K={K_ACTIVO} | similitud media alineada = {sim_consenso:.4f})")

    # Cruce secuencial 1..P con localidades de Coordenadas_codigo.csv (inmune a IDs duplicados como en Ruil)
    df_sit_map = df_sitios.reset_index(drop=True).copy()
    df_sit_map["Pop"] = range(1, len(df_sit_map) + 1)
    if "Codigo" not in df_sit_map.columns:
        df_sit_map["Codigo"] = df_sit_map["Referencia"].astype(str).str.slice(0, 3).str.upper()

    df_pop_join = df_sit_map[["Pop", "ID", "Codigo", "Referencia", "Lat", "Lon"]].merge(
        df_pop_cons, on="Pop", how="inner"
    )
    if df_pop_join.empty:
        raise ValueError("No coincidieron los números de población de STRUCTURE con las localidades del CSV.")

    # Reordenar clusters K1..Kk de Norte a Sur según latitud promedio ponderada por ancestría
    if K_ACTIVO > 1:
        lat_ponderada = []
        for col in K_COLS_ACTIVOS:
            pesos = df_pop_join[col].to_numpy(dtype=float)
            lats = df_pop_join["Lat"].to_numpy(dtype=float)
            lp = float(np.sum(pesos * lats) / np.sum(pesos)) if np.sum(pesos) > 0 else -90.0
            lat_ponderada.append((col, lp))
        # Mayor latitud (menos negativa en hemisferio sur) = más al Norte
        orden_cols = [c for c, _ in sorted(lat_ponderada, key=lambda x: x[1], reverse=True)]
        Q_pop_reord = df_pop_join[orden_cols].to_numpy(dtype=float)
        Q_ind_reord = df_struct_ind[orden_cols].to_numpy(dtype=float)
        for i, col in enumerate(K_COLS_ACTIVOS):
            df_pop_join[col] = np.round(Q_pop_reord[:, i], 4)
            df_struct_ind[col] = np.round(Q_ind_reord[:, i], 4)

    # Métricas poblacionales de ancestría y mezcla
    Q_mat = df_pop_join[K_COLS_ACTIVOS].to_numpy(dtype=float)
    dom_idx = np.argmax(Q_mat, axis=1)
    q_max = np.max(Q_mat, axis=1)
    q_safe = np.clip(Q_mat, 1e-12, 1.0)
    entropia_q = -np.sum(Q_mat * np.log(q_safe), axis=1)

    df_pop_join["cluster_dom"] = [f"K{i+1}" for i in dom_idx]
    df_pop_join["q_max"] = np.round(q_max, 3)
    df_pop_join["entropia_q"] = np.round(entropia_q, 3)
    df_pop_join["estado_mezcla"] = np.where(q_max >= 0.80, "Pura (Qmax ≥ 0.80)", "Mezcla (Qmax < 0.80)")

    # Vincular metadatos de localidad a la tabla individual
    df_struct_ind = df_struct_ind.merge(
        df_pop_join[["Pop", "Codigo", "Referencia", "Lat", "Lon"]], on="Pop", how="left"
    )
    Q_ind_mat = df_struct_ind[K_COLS_ACTIVOS].to_numpy(dtype=float)
    df_struct_ind["cluster_dom"] = [f"K{i+1}" for i in np.argmax(Q_ind_mat, axis=1)]
    df_struct_ind["q_max"] = np.round(np.max(Q_ind_mat, axis=1), 3)

    gdf_struct_pop_wgs84 = gpd.GeoDataFrame(
        df_pop_join,
        geometry=gpd.points_from_xy(df_pop_join["Lon"], df_pop_join["Lat"]),
        crs="EPSG:4326"
    )
    gdf_struct_pop_ea = gdf_struct_pop_wgs84.to_crs(CRS_EQUAL_AREA)

    gdf_struct_ind_ea = gpd.GeoDataFrame(
        df_struct_ind,
        geometry=gpd.points_from_xy(df_struct_ind["Lon"], df_struct_ind["Lat"]),
        crs="EPSG:4326"
    ).to_crs(CRS_EQUAL_AREA)

    # Exportar capas genéticas al mismo GeoPackage (UTM Chile) y tabla Evanno
    gdf_struct_pop_ea.to_crs(CRS_EXPORT_UTM).to_file(ARCHIVO_SALIDA, layer="structure_poblaciones_q", driver="GPKG")
    gdf_struct_ind_ea.to_crs(CRS_EXPORT_UTM).to_file(ARCHIVO_SALIDA, layer="structure_individuos_q", driver="GPKG")
    if df_evanno is not None:
        archivo_evanno = f"evanno_resumen_{ESPECIE_SELECCIONADA.lower()}.csv"
        df_evanno.to_csv(archivo_evanno, index=False, sep=";")
        print(f"Tabla Evanno guardada en: {archivo_evanno}")

    COLORES_ACTIVOS = [PALETA_CLUSTERS[i % len(PALETA_CLUSTERS)] for i in range(K_ACTIVO)]

    # Precalcular superficie IDW exploratoria si está activada (así queda lista tanto para 11.2 como para 11.3)
    IDW_RGBA_WGS84 = None
    IDW_BOUNDS_WGS84 = None
    if MOSTRAR_IDW_ANCESTRIA:
        if es_disyunta and gdf_eoo_subpob_ea is not None:
            _dom_idw = gdf_eoo_subpob_ea.geometry.union_all().union(gdf_struct_pop_ea.geometry.buffer(5000).union_all())
        else:
            _dom_idw = eoo_mcp_tierra_geom.union(gdf_struct_pop_ea.geometry.buffer(5000).union_all())
        _m_rad = gdf_struct_pop_ea.geometry.buffer(RADIO_MASCARA_IDW_KM * 1000.0).union_all()
        _m_idw = shapely.make_valid(_dom_idw.intersection(_m_rad))
        if not _m_idw.is_empty:
            _m_wgs = gpd.GeoSeries([_m_idw], crs=CRS_EQUAL_AREA).to_crs("EPSG:4326").iloc[0]
            _lon0, _lat0, _lon1, _lat1 = _m_wgs.bounds
            _ng = 240
            _GLON, _GLAT = np.meshgrid(np.linspace(_lon0, _lon1, _ng), np.linspace(_lat0, _lat1, _ng))
            _pts_g = gpd.GeoSeries(gpd.points_from_xy(_GLON.ravel(), _GLAT.ravel()), crs="EPSG:4326").to_crs(CRS_EQUAL_AREA)
            _gx = _pts_g.x.to_numpy().reshape(_GLON.shape)
            _gy = _pts_g.y.to_numpy().reshape(_GLAT.shape)
            _in_m = shapely.contains_xy(_m_idw, _gx, _gy)
            if np.any(_in_m):
                _px = gdf_struct_pop_ea.geometry.x.to_numpy()
                _py = gdf_struct_pop_ea.geometry.y.to_numpy()
                _Qm = gdf_struct_pop_ea[K_COLS_ACTIVOS].to_numpy(dtype=float)
                _rgb_p = np.array([mcolors.to_rgb(c) for c in COLORES_ACTIVOS], dtype=np.float32)
                _dist = np.hypot(_gx[_in_m][:, None] - _px[None, :], _gy[_in_m][:, None] - _py[None, :])
                _w = 1.0 / np.power(_dist + 250.0, POTENCIA_IDW)
                _w /= np.sum(_w, axis=1, keepdims=True)
                _Qi = _w @ _Qm
                _dom = np.argmax(_Qi, axis=1)
                _qmax = np.max(_Qi, axis=1)
                _cert = np.clip((_qmax - 1.0 / K_ACTIVO) / max(1.0 - 1.0 / K_ACTIVO, 1e-6), 0.0, 1.0) if K_ACTIVO > 1 else np.ones_like(_qmax)
                _rgba = np.zeros((_ng, _ng, 4), dtype=np.float32)
                _rgba[_in_m, :3] = _rgb_p[_dom]
                _rgba[_in_m, 3] = 0.15 + 0.47 * _cert
                IDW_RGBA_WGS84 = _rgba
                IDW_BOUNDS_WGS84 = [[_lat0, _lon0], [_lat1, _lon1]]
                print(f"Superficie IDW exploratoria lista (radio máximo {RADIO_MASCARA_IDW_KM} km).")

    STRUCTURE_LISTO = True

    print(f"Capas 'structure_poblaciones_q' y 'structure_individuos_q' añadidas a '{ARCHIVO_SALIDA}'.")
    display(df_pop_join[["Pop", "Codigo", "Referencia", "n_ind"] + K_COLS_ACTIVOS + ["cluster_dom", "q_max", "estado_mezcla"]])

### 11.2. Figura Compuesta de Publicación (Panel A: Mapa con Tortas y Líneas Guía | Panel B: Evanno $\Delta K$ | Panel C: Barplot Vertical)

Esta celda genera una figura compuesta lista para informes y presentaciones (`300 dpi` en formato `.png` y vectorial `.svg`):
* **Panel A (izquierda):** Mapa con límites regionales de Chile (GADM v4.1), EOO/núcleos en verde suave, *inset* de ubicación de Chile continental, barra de escala métrica y **diagramas de torta sin solapamiento** conectados con líneas guía punteadas y el `Codigo` de cada localidad *(más superficie IDW enmascarada si activaste `MOSTRAR_IDW_ANCESTRIA = True`)*.
* **Panel B (arriba derecha):** Curva de **Evanno ($\Delta K$)** junto a $\text{Mean } \ln P(D) \pm \text{SD}$, destacando el $K$ seleccionado.
* **Panel C (abajo derecha):** **Barplot vertical de ancestría individual** ordenado de Norte a Sur (en espejo con la latitud del mapa), etiquetado con los códigos cortos y con la misma paleta de colores del mapa.

> **Referencia del diseño de visualización:** La composición multipanel A + B + C (mapa con diagramas de torta conectados por líneas guía, curva de selección de $K$ y barplot vertical sincronizado) está inspirada en Pereira et al. (2022), *Heredity* 129: 161–168 ([doi:10.1038/s41437-022-00549-7](https://doi.org/10.1038/s41437-022-00549-7)).

In [ ]:
def cargar_contexto_vecino_arg():
    '''Descarga y cachea el contorno simplificado de Argentina (GADM 4.1 nivel 0) como contexto terrestre al Este.'''
    cache_dir = "cache_limites_geo"
    os.makedirs(cache_dir, exist_ok=True)
    zip_arg = os.path.join(cache_dir, "gadm41_ARG_0.json.zip")
    url_arg = "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_ARG_0.json.zip"
    try:
        if not os.path.exists(zip_arg):
            r = SESSION_HTTP.get(url_arg, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
            r.raise_for_status()
            with open(zip_arg, "wb") as f:
                f.write(r.content)
        with zipfile.ZipFile(zip_arg, "r") as zf:
            jname = [n for n in zf.namelist() if n.endswith(".json")][0]
            jpath = os.path.join(cache_dir, jname)
            if not os.path.exists(jpath):
                zf.extract(jname, cache_dir)
        gdf_arg = gpd.read_file(jpath).to_crs("EPSG:4326")
        gdf_arg["geometry"] = gdf_arg.geometry.make_valid()
        return gdf_arg
    except Exception:
        return None

def cargar_regiones_chile_gadm():
    '''Descarga y cachea los límites regionales de Chile (GADM 4.1 nivel 1) para el mapa de publicación.'''
    cache_dir = "cache_limites_geo"
    os.makedirs(cache_dir, exist_ok=True)
    zip_reg = os.path.join(cache_dir, "gadm41_CHL_1.json.zip")
    url_reg = "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_CHL_1.json.zip"
    try:
        if not os.path.exists(zip_reg):
            r = SESSION_HTTP.get(url_reg, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
            r.raise_for_status()
            with open(zip_reg, "wb") as f:
                f.write(r.content)
        with zipfile.ZipFile(zip_reg, "r") as zf:
            jname = [n for n in zf.namelist() if n.endswith(".json")][0]
            jpath = os.path.join(cache_dir, jname)
            if not os.path.exists(jpath):
                zf.extract(jname, cache_dir)
        gdf_reg = gpd.read_file(jpath).to_crs("EPSG:4326")
        gdf_reg["geometry"] = gdf_reg.geometry.make_valid()
        return gdf_reg
    except Exception as e:
        print(f"Aviso: No se pudo cargar GADM nivel 1 ({e}); se usará la máscara base.")
        return None

def calcular_superficie_idw_rgba(gdf_pop_ea, k_cols, colores_hex, mascara_dom_ea, radio_km=60, potencia=2.0, n_grid=240):
    '''
    Calcula superficie raster RGBA de ancestría interpolada por IDW (tipo tess3r) sobre malla WGS84,
    con distancias exactas en LAEA Equal-Area, doble enmascaramiento (EOO/núcleos + radio muestreado)
    y opacidad modulada por certeza Qmax.
    '''
    mascara_radio = gdf_pop_ea.geometry.buffer(radio_km * 1000.0).union_all()
    mascara_idw_ea = shapely.make_valid(mascara_dom_ea.intersection(mascara_radio))
    if mascara_idw_ea.is_empty:
        return None, None, None

    # Límites en WGS84 para alineación exacta en Folium ImageOverlay y en Matplotlib
    mascara_wgs84 = gpd.GeoSeries([mascara_idw_ea], crs=CRS_EQUAL_AREA).to_crs("EPSG:4326").iloc[0]
    min_lon, min_lat, max_lon, max_lat = mascara_wgs84.bounds

    lons = np.linspace(min_lon, max_lon, n_grid)
    lats = np.linspace(min_lat, max_lat, n_grid)
    GLON, GLAT = np.meshgrid(lons, lats)

    # Proyectar grilla a LAEA Equal-Area para máscara y distancias euclidianas en metros
    pts_grid_ea = gpd.GeoSeries(
        gpd.points_from_xy(GLON.ravel(), GLAT.ravel()), crs="EPSG:4326"
    ).to_crs(CRS_EQUAL_AREA)
    gx_ea = pts_grid_ea.x.to_numpy().reshape(GLON.shape)
    gy_ea = pts_grid_ea.y.to_numpy().reshape(GLAT.shape)

    dentro = shapely.contains_xy(mascara_idw_ea, gx_ea, gy_ea)
    rgba = np.zeros((n_grid, n_grid, 4), dtype=np.float32)

    if not np.any(dentro):
        return None, None, None

    px = gdf_pop_ea.geometry.x.to_numpy()
    py = gdf_pop_ea.geometry.y.to_numpy()
    Q = gdf_pop_ea[k_cols].to_numpy(dtype=float)
    K = len(k_cols)
    rgb_pal = np.array([mcolors.to_rgb(c) for c in colores_hex], dtype=np.float32)

    x_in = gx_ea[dentro][:, None]
    y_in = gy_ea[dentro][:, None]
    dist = np.hypot(x_in - px[None, :], y_in - py[None, :])
    w = 1.0 / np.power(dist + 250.0, potencia)
    w = w / np.sum(w, axis=1, keepdims=True)
    Q_interp = w @ Q  # (n_in, K)

    dom = np.argmax(Q_interp, axis=1)
    q_max_in = np.max(Q_interp, axis=1)
    # Certeza normalizada [0, 1] -> opacidad [0.15, 0.62]
    certeza = np.clip((q_max_in - (1.0 / K)) / max(1.0 - (1.0 / K), 1e-6), 0.0, 1.0) if K > 1 else np.ones_like(q_max_in)
    alpha_in = 0.15 + 0.47 * certeza

    rgba_in = np.zeros((len(dom), 4), dtype=np.float32)
    rgba_in[:, :3] = rgb_pal[dom]
    rgba_in[:, 3] = alpha_in
    rgba[dentro] = rgba_in

    # También devolver bounds en EA para graficar limpiamente en Panel A (que está en LAEA)
    bounds_wgs84 = [[min_lat, min_lon], [max_lat, max_lon]]
    return rgba, bounds_wgs84, mascara_idw_ea

def distribuir_posiciones_y(ys_objetivo, y_min, y_max, sep_min):
    '''Ajusta coordenadas Y ordenadas (Norte->Sur) garantizando separación mínima sin solapamiento.'''
    n = len(ys_objetivo)
    if n == 0:
        return []
    if n == 1:
        return [float(np.clip(ys_objetivo[0], y_min, y_max))]

    # Si no cabe con el espaciado ideal, distribuir uniformemente en el rango disponible
    if (n - 1) * sep_min >= (y_max - y_min):
        return list(np.linspace(y_max, y_min, n))

    # Relajación iterativa preservando orden Norte->Sur y cercanía a la latitud real
    ys = np.clip(np.array(ys_objetivo, dtype=float), y_min, y_max)
    for _ in range(120):
        # Empujar hacia abajo si están muy cerca (orden decreciente: ys[i] >= ys[i+1] + sep_min)
        for i in range(n - 1):
            gap = ys[i] - ys[i + 1]
            if gap < sep_min:
                dif = (sep_min - gap) / 2.0
                ys[i] += dif
                ys[i + 1] -= dif
        # Re-clamping a los límites
        if ys[0] > y_max:
            ys -= (ys[0] - y_max)
        if ys[-1] < y_min:
            ys += (y_min - ys[-1])
        ys = np.clip(ys, y_min, y_max)
    return ys.tolist()

if not globals().get("STRUCTURE_LISTO", False):
    print("Aún no se ha ejecutado la celda 11.1 (o el entorno de Colab se reinició). Ejecuta las celdas previas y la 11.1 con ACTIVAR_MODULO_STRUCTURE = True.")
else:
    # 1. Calcular IDW si está activado y aún no fue precalculado en 11.1
    if MOSTRAR_IDW_ANCESTRIA and globals().get("IDW_RGBA_WGS84") is None:
        if es_disyunta and gdf_eoo_subpob_ea is not None:
            dom_idw = gdf_eoo_subpob_ea.geometry.union_all().union(gdf_struct_pop_ea.geometry.buffer(5000).union_all())
        else:
            dom_idw = eoo_mcp_tierra_geom.union(gdf_struct_pop_ea.geometry.buffer(5000).union_all())
        IDW_RGBA_WGS84, IDW_BOUNDS_WGS84, _ = calcular_superficie_idw_rgba(
            gdf_struct_pop_ea, K_COLS_ACTIVOS, COLORES_ACTIVOS,
            mascara_dom_ea=dom_idw, radio_km=RADIO_MASCARA_IDW_KM, potencia=POTENCIA_IDW
        )

    # 2. Cargar regiones de Chile (GADM 4.1 nivel 1) + contexto terrestre de Argentina
    gdf_regiones_wgs84 = cargar_regiones_chile_gadm()
    gdf_arg_wgs84 = cargar_contexto_vecino_arg()

    # Definir caja espacial en LAEA Equal-Area con pasillos laterales amplios para las tortas
    pts_union_bounds = gdf_struct_pop_ea.total_bounds
    eoo_bounds = gdf_eoo_oficial_ea.total_bounds
    x_min_d = min(pts_union_bounds[0], eoo_bounds[0])
    y_min_d = min(pts_union_bounds[1], eoo_bounds[1])
    x_max_d = max(pts_union_bounds[2], eoo_bounds[2])
    y_max_d = max(pts_union_bounds[3], eoo_bounds[3])

    span_x = max(x_max_d - x_min_d, 120000.0)
    span_y = max(y_max_d - y_min_d, 180000.0)

    # Corredores izquierdo y derecho para ubicar las tortas sin tapar el territorio ni el inset
    margen_y_top = 0.14 * span_y
    margen_y_bot = 0.16 * span_y
    ylim_bot = y_min_d - margen_y_bot
    ylim_top = y_max_d + margen_y_top
    altura_total = ylim_top - ylim_bot

    # Radio de torta equilibrado para garantizar holgura vertical entre todas las tortas
    n_pops = len(gdf_struct_pop_ea)
    n_lado = math.ceil(n_pops / 2)
    r_pie = min(altura_total / (4.15 * max(n_lado, 4)), 0.048 * altura_total)

    x_corredor_izq = x_min_d - max(0.32 * span_x, 2.5 * r_pie)
    x_corredor_der = x_max_d + max(0.32 * span_x, 2.5 * r_pie)
    xlim_izq = x_corredor_izq - 2.60 * r_pie
    xlim_der = x_corredor_der + 2.55 * r_pie

    # 3. Configurar figura compuesta (Panel A izquierda | Panel B arriba-der | Panel C abajo-der)
    fig = plt.figure(figsize=(14.8, 10.8), dpi=150, facecolor="white")
    gs = gridspec.GridSpec(
        2, 2,
        width_ratios=[2.25, 1.0],
        height_ratios=[0.27, 0.73],
        wspace=0.14, hspace=0.24
    )

    ax_map = fig.add_subplot(gs[:, 0])
    ax_ev  = fig.add_subplot(gs[0, 1])
    ax_bar = fig.add_subplot(gs[1, 1])

    # ================= PANEL A: MAPA + TORTAS SIN SOLAPAMIENTO =================
    ax_map.set_facecolor("#E5F0F9")  # Océano Pacífico suave
    bbox_plot_ea = box(xlim_izq - 90000, ylim_bot - 90000, xlim_der + 90000, ylim_top + 90000)
    bbox_plot_wgs84 = gpd.GeoSeries([bbox_plot_ea], crs=CRS_EQUAL_AREA).to_crs("EPSG:4326").iloc[0]

    # Contexto continental terrestre (Argentina en gris cálido muy tenue para no confundirlo con océano)
    if gdf_arg_wgs84 is not None:
        arg_clip = gpd.clip(gdf_arg_wgs84, bbox_plot_wgs84)
        if not arg_clip.empty:
            arg_clip.to_crs(CRS_EQUAL_AREA).plot(
                ax=ax_map, facecolor="#ECECE8", edgecolor="#B5B5B0", linewidth=0.6, zorder=1
            )

    # Regiones de Chile (GADM 4.1 nivel 1)
    if gdf_regiones_wgs84 is not None:
        reg_clip = gpd.clip(gdf_regiones_wgs84, bbox_plot_wgs84).to_crs(CRS_EQUAL_AREA)
        reg_clip.plot(ax=ax_map, facecolor="#FAF9F5", edgecolor="#888888", linewidth=0.65, zorder=2)
        # Etiquetas discretas de regiones
        for _, reg_row in reg_clip.iterrows():
            if reg_row.geometry.area / 1e6 > 1500:
                pt_r = reg_row.geometry.representative_point()
                if (x_min_d - 0.15 * span_x <= pt_r.x <= x_max_d + 0.15 * span_x) and (y_min_d <= pt_r.y <= y_max_d):
                    nom_reg = str(reg_row.get("NAME_1", "")).replace("Región de ", "").replace("Región del ", "")
                    ax_map.text(
                        pt_r.x, pt_r.y, nom_reg, fontsize=6.5, color="#777777",
                        ha="center", va="center", style="italic", zorder=3,
                        path_effects=[withStroke(linewidth=1.8, foreground="#FAF9F5")]
                    )
    else:
        gpd.GeoSeries([mascara_tierra_ea], crs=CRS_EQUAL_AREA).plot(
            ax=ax_map, facecolor="#FAF9F5", edgecolor="#666666", linewidth=0.8, zorder=2
        )

    # EOO / núcleos biogeográficos en verde salvia suave (referencia visual del rango)
    capa_rango = gdf_eoo_subpob_ea if (es_disyunta and gdf_eoo_subpob_ea is not None) else gdf_eoo_oficial_ea
    capa_rango.plot(ax=ax_map, facecolor="#C7E2C2", edgecolor="#4F8A4B", linewidth=1.1, alpha=0.58, zorder=4)

    # Superficie IDW opcional (si el usuario la activó)
    if MOSTRAR_IDW_ANCESTRIA and IDW_RGBA_WGS84 is not None:
        mascara_radio_ea = gdf_struct_pop_ea.geometry.buffer(RADIO_MASCARA_IDW_KM * 1000.0).union_all()
        dom_ea = (gdf_eoo_subpob_ea.geometry.union_all() if (es_disyunta and gdf_eoo_subpob_ea is not None) else eoo_mcp_tierra_geom)
        dom_ea = dom_ea.union(gdf_struct_pop_ea.geometry.buffer(5000).union_all())
        m_idw = shapely.make_valid(dom_ea.intersection(mascara_radio_ea))
        if not m_idw.is_empty:
            mx0, my0, mx1, my1 = m_idw.bounds
            gx1 = np.linspace(mx0, mx1, 220)
            gy1 = np.linspace(my0, my1, 220)
            GXX, GYY = np.meshgrid(gx1, gy1)
            mask_in = shapely.contains_xy(m_idw, GXX, GYY)
            rgba_ea = np.zeros((220, 220, 4), dtype=np.float32)
            if np.any(mask_in):
                px = gdf_struct_pop_ea.geometry.x.to_numpy()
                py = gdf_struct_pop_ea.geometry.y.to_numpy()
                Q_m = gdf_struct_pop_ea[K_COLS_ACTIVOS].to_numpy(dtype=float)
                rgb_p = np.array([mcolors.to_rgb(c) for c in COLORES_ACTIVOS], dtype=np.float32)
                d_ea = np.hypot(GXX[mask_in][:, None] - px[None, :], GYY[mask_in][:, None] - py[None, :])
                w_ea = 1.0 / np.power(d_ea + 250.0, POTENCIA_IDW)
                w_ea /= np.sum(w_ea, axis=1, keepdims=True)
                Qi = w_ea @ Q_m
                dom_i = np.argmax(Qi, axis=1)
                qm_i = np.max(Qi, axis=1)
                cert_i = np.clip((qm_i - 1.0 / K_ACTIVO) / max(1.0 - 1.0 / K_ACTIVO, 1e-6), 0.0, 1.0) if K_ACTIVO > 1 else np.ones_like(qm_i)
                rgba_ea[mask_in, :3] = rgb_p[dom_i]
                rgba_ea[mask_in, 3] = 0.14 + 0.44 * cert_i
                ax_map.imshow(rgba_ea, origin="lower", extent=[mx0, mx1, my0, my1], zorder=5)

    # Asignación balanceada Izquierda / Derecha ordenada de Norte a Sur (0 cruces de líneas guía)
    df_p_ord = gdf_struct_pop_ea.sort_values("Lat", ascending=False).reset_index(drop=True)
    izq_rows, der_rows = [], []
    i = 0
    while i < len(df_p_ord):
        if i + 1 < len(df_p_ord):
            r1, r2 = df_p_ord.iloc[i], df_p_ord.iloc[i + 1]
            if r1.geometry.x <= r2.geometry.x:
                izq_rows.append(r1)
                der_rows.append(r2)
            else:
                izq_rows.append(r2)
                der_rows.append(r1)
            i += 2
        else:
            r1 = df_p_ord.iloc[i]
            if len(izq_rows) <= len(der_rows):
                izq_rows.append(r1)
            else:
                der_rows.append(r1)
            i += 1

    izq_rows = sorted(izq_rows, key=lambda r: r.geometry.y, reverse=True)
    der_rows = sorted(der_rows, key=lambda r: r.geometry.y, reverse=True)

    # Reservar espacio despejado para inset (superior-izq) y barra de escala (inferior-izq)
    y_izq_max = ylim_top - 0.34 * altura_total
    y_izq_min = ylim_bot + 0.13 * altura_total
    y_der_max = ylim_top - 0.08 * altura_total
    y_der_min = ylim_bot + 0.08 * altura_total

    ys_izq = distribuir_posiciones_y([r.geometry.y for r in izq_rows], y_izq_min, y_izq_max, sep_min=2.65 * r_pie)
    ys_der = distribuir_posiciones_y([r.geometry.y for r in der_rows], y_der_min, y_der_max, sep_min=2.65 * r_pie)

    def dibujar_torta_con_guia(ax, row, cx, cy, radio, lado):
        px, py = row.geometry.x, row.geometry.y
        ax.scatter([px], [py], color="black", s=24, zorder=8, edgecolors="white", linewidths=0.6)
        ax.plot([px, cx], [py, cy], color="#2F2F2F", linestyle=":", linewidth=1.05, zorder=6)
        ang_ini = 90.0
        for idx_k, col_k in enumerate(K_COLS_ACTIVOS):
            frac = float(row[col_k])
            if frac <= 0:
                continue
            ang_fin = ang_ini - frac * 360.0
            wedge = mpatches.Wedge(
                (cx, cy), radio, ang_fin, ang_ini,
                facecolor=COLORES_ACTIVOS[idx_k], edgecolor="#111111", linewidth=0.75, zorder=9
            )
            ax.add_patch(wedge)
            ang_ini = ang_fin
        circ = mpatches.Circle((cx, cy), radio, facecolor="none", edgecolor="#111111", linewidth=0.9, zorder=10)
        ax.add_patch(circ)
        if lado == "izq":
            tx, ha = cx - 1.22 * radio, "right"
        else:
            tx, ha = cx + 1.22 * radio, "left"
        ax.text(
            tx, cy, str(row["Codigo"]), fontsize=8.5, fontweight="bold", color="#111111",
            ha=ha, va="center", zorder=11,
            path_effects=[withStroke(linewidth=2.4, foreground="white")]
        )

    for row, cy in zip(izq_rows, ys_izq):
        dibujar_torta_con_guia(ax_map, row, x_corredor_izq, cy, r_pie, lado="izq")
    for row, cy in zip(der_rows, ys_der):
        dibujar_torta_con_guia(ax_map, row, x_corredor_der, cy, r_pie, lado="der")

    ax_map.set_xlim(xlim_izq, xlim_der)
    ax_map.set_ylim(ylim_bot, ylim_top)
    ax_map.set_aspect("equal")
    ax_map.set_xticks([])
    ax_map.set_yticks([])
    ax_map.set_xlabel("")
    ax_map.set_ylabel("")
    for spine in ax_map.spines.values():
        spine.set_edgecolor("#333333")
        spine.set_linewidth(1.0)

    titulo_idw = " + IDW exploratorio" if MOSTRAR_IDW_ANCESTRIA else ""
    ax_map.set_title(
        f"A   Estructura genética ({ESPECIE_SELECCIONADA}, K = {K_ACTIVO}){titulo_idw}",
        loc="left", fontsize=11.5, fontweight="bold", pad=8
    )

    # Barra de escala métrica exacta en km (esquina inferior izquierda)
    ancho_mapa_km = (xlim_der - xlim_izq) / 1000.0
    largo_esc_km = next((v for v in [20, 50, 100, 150, 200, 300] if v >= ancho_mapa_km * 0.16), 100)
    esc_x0 = xlim_izq + 0.04 * (xlim_der - xlim_izq)
    esc_y0 = ylim_bot + 0.038 * altura_total
    esc_h = 0.011 * altura_total
    half_m = (largo_esc_km * 1000.0) / 2.0
    ax_map.add_patch(mpatches.Rectangle((esc_x0, esc_y0), half_m, esc_h, facecolor="black", edgecolor="black", zorder=12))
    ax_map.add_patch(mpatches.Rectangle((esc_x0 + half_m, esc_y0), half_m, esc_h, facecolor="white", edgecolor="black", zorder=12))
    for val_km, pos_x in [(0, esc_x0), (largo_esc_km // 2, esc_x0 + half_m), (largo_esc_km, esc_x0 + 2 * half_m)]:
        lbl = f"{val_km} km" if val_km == largo_esc_km else f"{val_km}"
        ax_map.text(pos_x, esc_y0 + esc_h * 1.45, lbl, fontsize=7.5, ha="center", va="bottom", fontweight="bold", zorder=12)

    # Leyenda horizontal limpia DEBAJO del Panel A (cero solapamiento con cualquier torta o K)
    handles_leg = [
        mpatches.Patch(facecolor="#C7E2C2", edgecolor="#4F8A4B", label="Rango EOO / Núcleos")
    ]
    for idx_k in range(K_ACTIVO):
        suf_pos = " (Norte)" if idx_k == 0 and K_ACTIVO > 1 else (" (Sur)" if idx_k == K_ACTIVO - 1 and K_ACTIVO > 1 else "")
        handles_leg.append(
            mpatches.Patch(facecolor=COLORES_ACTIVOS[idx_k], edgecolor="#111111", label=f"Cluster K{idx_k+1}{suf_pos}")
        )
    ax_map.legend(
        handles=handles_leg, loc="upper center", bbox_to_anchor=(0.5, -0.018),
        ncol=min(len(handles_leg), 4), frameon=True, facecolor="white",
        edgecolor="#666666", fontsize=8.2, handlelength=1.4, columnspacing=1.2
    )

    # Inset de ubicación de Chile continental (esquina superior izquierda, sin etiquetas de ejes)
    if gdf_regiones_wgs84 is not None:
        ax_ins = ax_map.inset_axes([0.02, 0.72, 0.16, 0.25])
        ax_ins.set_facecolor("#E5F0F9")
        chile_simple = gdf_regiones_wgs84.dissolve().simplify(0.08)
        chile_simple.plot(ax=ax_ins, facecolor="#D6D6D2", edgecolor="#555555", linewidth=0.5)
        b_est = gdf_struct_pop_wgs84.total_bounds
        rec = mpatches.Rectangle(
            (b_est[0] - 0.8, b_est[1] - 0.8),
            (b_est[2] - b_est[0]) + 1.6,
            (b_est[3] - b_est[1]) + 1.6,
            facecolor="none", edgecolor="#C0392B", linewidth=1.6, zorder=5
        )
        ax_ins.add_patch(rec)
        ax_ins.set_xlim(-76.5, -66.0)
        ax_ins.set_ylim(-56.5, -17.0)
        ax_ins.set_xticks([])
        ax_ins.set_yticks([])
        ax_ins.set_xlabel("")
        ax_ins.set_ylabel("")
        ax_ins.set_title("Chile", fontsize=7.5, fontweight="bold", pad=2)
        for sp in ax_ins.spines.values():
            sp.set_edgecolor("#333333")
            sp.set_linewidth(0.8)

    # ================= PANEL B: EVANNO (DELTA K + LN P(D)) =================
    if df_evanno is not None and len(df_evanno) >= 2:
        df_dk = df_evanno.dropna(subset=["delta_K"])
        if not df_dk.empty:
            l1, = ax_ev.plot(df_dk["K"], df_dk["delta_K"], color="#111111", marker="o", linewidth=1.8, markersize=5, label="ΔK (Evanno)")
            l2 = ax_ev.axvline(K_ACTIVO, color="#C0392B", linestyle="--", linewidth=1.3, label=f"K elegido = {K_ACTIVO}")
            ax_ev.set_ylabel("ΔK (Evanno)", fontsize=8.5, fontweight="bold", color="#111111")
            ax_ev.set_xticks(df_evanno["K"].astype(int).tolist())
            max_dk_val = float(df_dk["delta_K"].max())
            ax_ev.set_ylim(0, max(max_dk_val * 1.28, 1.0))

            ax_ev2 = ax_ev.twinx()
            l3 = ax_ev2.errorbar(
                df_evanno["K"], df_evanno["mean_lnpd"], yerr=df_evanno["sd_lnpd"].fillna(0),
                color="#7F8C8D", linestyle=":", marker="s", markersize=3.5, linewidth=1.0, capsize=2, alpha=0.85,
                label="Mean Ln P(D)"
            )
            ax_ev2.set_ylabel("Mean Ln P(D)", fontsize=7.5, color="#7F8C8D")
            ax_ev2.tick_params(axis="y", labelcolor="#7F8C8D", labelsize=7)
            ax_ev2.spines["top"].set_visible(False)
            ln_min = float((df_evanno["mean_lnpd"] - df_evanno["sd_lnpd"].fillna(0)).min())
            ln_max = float((df_evanno["mean_lnpd"] + df_evanno["sd_lnpd"].fillna(0)).max())
            ln_pad = max((ln_max - ln_min) * 0.25, 10.0)
            ax_ev2.set_ylim(ln_min - ln_pad * 0.4, ln_max + ln_pad)

            ax_ev.legend(
                handles=[l1, l2, l3], labels=["ΔK (Evanno)", f"K = {K_ACTIVO}", "Mean Ln P(D)"],
                loc="upper center", bbox_to_anchor=(0.5, 1.03), ncol=3,
                fontsize=6.8, frameon=False, handlelength=1.4, columnspacing=0.8
            )
        else:
            ax_ev.errorbar(
                df_evanno["K"], df_evanno["mean_lnpd"], yerr=df_evanno["sd_lnpd"].fillna(0),
                color="#111111", marker="o", linewidth=1.6, capsize=3
            )
            ax_ev.axvline(K_ACTIVO, color="#C0392B", linestyle="--", linewidth=1.3)
            ax_ev.set_ylabel("Mean Ln P(D)", fontsize=8.5, fontweight="bold")

        ax_ev.set_xlabel("Número de grupos genéticos (K)", fontsize=8.5)
        ax_ev.tick_params(labelsize=7.5)
        ax_ev.spines["top"].set_visible(False)
        ax_ev.set_title("B   Selección de K (Evanno / Ln P(D))", loc="left", fontsize=10, fontweight="bold", pad=12)
    else:
        ax_ev.axis("off")
        ax_ev.text(
            0.5, 0.5, f"Corrida única / CSV\nK seleccionado = {K_ACTIVO}\n(Sube el .zip de Results/\npara calcular Evanno ΔK)",
            ha="center", va="center", fontsize=9, color="#444444",
            bbox=dict(boxstyle="round,pad=0.5", facecolor="#F4F6F6", edgecolor="#BBBBBB")
        )
        ax_ev.set_title("B   Resumen de K", loc="left", fontsize=10, fontweight="bold", pad=6)

    # ================= PANEL C: BARPLOT VERTICAL SINCRONIZADO =================
    if ORDEN_BARPLOT == "NORTE_A_SUR":
        orden_pops = df_pop_join.sort_values("Lat", ascending=False)["Pop"].tolist()
    else:
        orden_pops = df_pop_join.sort_values("Pop", ascending=True)["Pop"].tolist()

    bloques = []
    etiquetas_y = []
    separadores = []
    pos_act = 0
    for pid in orden_pops:
        sub_ind = df_struct_ind[df_struct_ind["Pop"] == pid].copy()
        bloques.append(sub_ind)
        n_b = len(sub_ind)
        cod = str(df_pop_join.loc[df_pop_join["Pop"] == pid, "Codigo"].iloc[0])
        etiquetas_y.append((pos_act + n_b / 2.0, cod))
        pos_act += n_b
        separadores.append(pos_act)

    df_bar = pd.concat(bloques, ignore_index=True)
    y_idx = np.arange(len(df_bar))
    left_acum = np.zeros(len(df_bar), dtype=float)

    for idx_k, col_k in enumerate(K_COLS_ACTIVOS):
        vals = df_bar[col_k].to_numpy(dtype=float)
        ax_bar.barh(
            y_idx, vals, left=left_acum, height=1.0,
            color=COLORES_ACTIVOS[idx_k], edgecolor="none"
        )
        left_acum += vals

    for sep_y in separadores[:-1]:
        ax_bar.axhline(sep_y - 0.5, color="white", linewidth=1.1)

    ax_bar.set_ylim(len(df_bar) - 0.5, -0.5)  # Norte arriba, Sur abajo
    ax_bar.set_xlim(0.0, 1.0)
    ax_bar.set_yticks([p for p, _ in etiquetas_y])
    ax_bar.set_yticklabels([c for _, c in etiquetas_y], fontsize=8.5, fontweight="bold")
    ax_bar.set_xlabel("Proporción de ancestría (Q)", fontsize=8.5)
    ax_bar.set_xticks([0.0, 0.5, 1.0])
    ax_bar.tick_params(axis="x", labelsize=7.5)
    for sp in ax_bar.spines.values():
        sp.set_edgecolor("#222222")
        sp.set_linewidth(0.8)
    ax_bar.set_title(f"C   Ancestría individual ({'Norte → Sur' if ORDEN_BARPLOT == 'NORTE_A_SUR' else 'Orden ID'})", loc="left", fontsize=10, fontweight="bold", pad=6)

    fig_png = f"figura_structure_{ESPECIE_SELECCIONADA.lower()}_K{K_ACTIVO}.png"
    fig_svg = f"figura_structure_{ESPECIE_SELECCIONADA.lower()}_K{K_ACTIVO}.svg"
    fig.savefig(fig_png, dpi=300, bbox_inches="tight", facecolor="white")
    fig.savefig(fig_svg, bbox_inches="tight", facecolor="white")
    print(f"Figura compuesta guardada en alta resolución: '{fig_png}' (300 dpi) y '{fig_svg}' (vectorial).")
    plt.show()

### 11.3. Mapa Interactivo Integrado (`Folium`): Distribución, Biodiversidad y Tortas SVG de STRUCTURE

Integra en un único visor interactivo todas las capas anteriores junto con:
* **Diagramas de torta SVG vectoriales** por localidad (haciendo clic se despliega el detalle de porcentajes $K_1 \dots K_K$, $Q_{\max}$, entropía $H_Q$ y estado *Pura* vs. *Mezcla*).
* **Superficie IDW de ancestría** *(solo si activaste `MOSTRAR_IDW_ANCESTRIA = True`, recortada a los núcleos/EOO y al radio muestreado)*.

In [ ]:
def construir_svg_torta(valores_q, colores, tamano=42):
    '''Genera un SVG limpio de gráfico de torta para incrustar como DivIcon en Folium.'''
    r = tamano / 2.0 - 2.0
    cx = cy = tamano / 2.0
    total = sum(valores_q)
    if total <= 0:
        return f'<svg width="{tamano}" height="{tamano}"><circle cx="{cx}" cy="{cy}" r="{r}" fill="#999"/></svg>'

    partes = [f'<svg width="{tamano}" height="{tamano}" viewBox="0 0 {tamano} {tamano}" xmlns="http://www.w3.org/2000/svg">']
    # Si un único cluster domina ~100%, dibujar círculo sólido limpio
    if max(valores_q) / total >= 0.999:
        idx_dom = int(np.argmax(valores_q))
        partes.append(f'<circle cx="{cx}" cy="{cy}" r="{r}" fill="{colores[idx_dom]}" stroke="#111" stroke-width="1.4"/>')
    else:
        ang = -math.pi / 2.0
        for val, col in zip(valores_q, colores):
            frac = val / total
            if frac <= 1e-4:
                continue
            ang_sig = ang + 2.0 * math.pi * frac
            x1, y1 = cx + r * math.cos(ang), cy + r * math.sin(ang)
            x2, y2 = cx + r * math.cos(ang_sig), cy + r * math.sin(ang_sig)
            large_arc = 1 if frac > 0.5 else 0
            d = f"M {cx:.2f} {cy:.2f} L {x1:.2f} {y1:.2f} A {r:.2f} {r:.2f} 0 {large_arc} 1 {x2:.2f} {y2:.2f} Z"
            partes.append(f'<path d="{d}" fill="{col}" stroke="#111" stroke-width="0.9"/>')
            ang = ang_sig
        partes.append(f'<circle cx="{cx}" cy="{cy}" r="{r}" fill="none" stroke="#111" stroke-width="1.3"/>')
    partes.append("</svg>")
    return "".join(partes)

if not globals().get("STRUCTURE_LISTO", False):
    print("Aún no se ha ejecutado la celda 11.1 (o el entorno de Colab se reinició). Ejecuta 'Entorno de ejecución -> Ejecutar todo' o corre desde la celda 11.1 con ACTIVAR_MODULO_STRUCTURE = True.")
else:
    m_gen = folium.Map(location=[lat_centro, lon_centro], zoom_start=8, tiles=None)

    folium.TileLayer(
        tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
        attr="&copy; OpenStreetMap contributors &copy; CARTO",
        name="Mapa Claro (CartoDB Positron)",
        show=True,
    ).add_to(m_gen)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}",
        attr="Tiles &copy; Esri",
        name="Topográfico (Esri World Topo)",
        show=False,
    ).add_to(m_gen)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Tiles &copy; Esri",
        name="Satélite (Esri World Imagery)",
        show=False,
    ).add_to(m_gen)

    # Capas base de conservación (apagadas las más densas para dar protagonismo a STRUCTURE)
    gdf_hex_final_ea.to_crs("EPSG:4326").explore(
        m=m_gen,
        name="Biodiversidad Flora Vascular (Shannon H')",
        column="shannon_h",
        cmap="viridis",
        tooltip=["hex_id", "n_registros", "riqueza_s", "shannon_h", "pielou_j", "simpson_1_d", "cobertura_c", "sp_dominante"],
        legend=False,
        style_kwds={"fillOpacity": 0.45, "weight": 0.7},
        show=False,
    )
    gdf_eoo_oficial_ea.to_crs("EPSG:4326").explore(
        m=m_gen,
        name=f"EOO Oficial B1 ({eoo_oficial_km2:,.0f} km²)",
        color="darkorange",
        style_kwds={"fill": True, "fillOpacity": 0.10, "weight": 2.2},
        tooltip=["especie", "escala", "eoo_km2", "umbral_preliminar_b1"],
        show=True,
    )
    if gdf_eoo_subpob_ea is not None:
        gdf_eoo_subpob_ea.to_crs("EPSG:4326").explore(
            m=m_gen,
            name="Núcleos Biogeográficos",
            color="royalblue",
            style_kwds={"fill": True, "fillOpacity": 0.10, "weight": 1.8, "dashArray": "5, 5"},
            tooltip=["subpoblacion", "n_registros", "area_nucleo_km2"],
            show=False,
        )
    gdf_aoo_uicn_ea.to_crs("EPSG:4326").explore(
        m=m_gen,
        name=f"AOO Oficial 2×2 km ({stats_aoo_uicn['aoo_km2']:,.0f} km²)",
        color="crimson",
        style_kwds={"fillOpacity": 0.35, "weight": 0.8},
        tooltip=["celda_id", "aoo_total_km2", "umbral_preliminar_b2"],
        show=False,
    )

    # Capa IDW exploratoria opcional (ImageOverlay con inversión vertical para origen norte de PNG)
    _idw_rgba = globals().get("IDW_RGBA_WGS84")
    _idw_bounds = globals().get("IDW_BOUNDS_WGS84")
    if MOSTRAR_IDW_ANCESTRIA and _idw_rgba is not None and _idw_bounds is not None:
        buf_png = io.BytesIO()
        plt.imsave(buf_png, np.flipud(_idw_rgba), format="png")
        b64_png = base64.b64encode(buf_png.getvalue()).decode("ascii")
        folium.raster_layers.ImageOverlay(
            image=f"data:image/png;base64,{b64_png}",
            bounds=_idw_bounds,
            opacity=0.9,
            name=f"Ancestría IDW Exploratoria (radio {RADIO_MASCARA_IDW_KM} km)",
            interactive=False,
            show=True,
        ).add_to(m_gen)

    # Capa de tortas SVG vectoriales de STRUCTURE por localidad
    fg_pies = folium.FeatureGroup(name=f"STRUCTURE Tortas por Localidad (K = {K_ACTIVO})", show=True)
    for _, row in gdf_struct_pop_wgs84.iterrows():
        q_vals = [float(row[c]) for c in K_COLS_ACTIVOS]
        svg_html = construir_svg_torta(q_vals, COLORES_ACTIVOS, tamano=40)
        icon_html = (
            f'<div style="display:flex;flex-direction:column;align-items:center;">'
            f'{svg_html}'
            f'<span style="font-size:10px;font-weight:700;color:#111;background:rgba(255,255,255,0.85);'
            f'padding:0px 3px;border-radius:3px;border:1px solid #999;margin-top:-3px;">{row["Codigo"]}</span>'
            f'</div>'
        )
        filas_q = "".join(
            f'<tr><td><span style="display:inline-block;width:10px;height:10px;background:{COLORES_ACTIVOS[i]};'
            f'border:1px solid #222;margin-right:5px;"></span><b>K{i+1}</b></td>'
            f'<td style="text-align:right;padding-left:10px;">{q_vals[i]*100:.1f}%</td></tr>'
            for i in range(K_ACTIVO)
        )
        popup_html = (
            f'<div style="font-family:sans-serif;font-size:12px;min-width:190px;">'
            f'<b>{row["Codigo"]} — {row["Referencia"]}</b><br>'
            f'Pop #{int(row["Pop"])} | n = {int(row["n_ind"])} ind.<br>'
            f'<table style="margin-top:5px;width:100%;">{filas_q}</table>'
            f'<hr style="margin:5px 0;">'
            f'<b>Dominante:</b> {row["cluster_dom"]} (Q<sub>max</sub>={row["q_max"]:.2f})<br>'
            f'<b>Estado:</b> {row["estado_mezcla"]}<br>'
            f'<b>Entropía mezcla (H<sub>Q</sub>):</b> {row["entropia_q"]:.3f}'
            f'</div>'
        )
        folium.Marker(
            location=[float(row["Lat"]), float(row["Lon"])],
            icon=folium.DivIcon(html=icon_html, icon_size=(46, 54), icon_anchor=(23, 20)),
            popup=folium.Popup(popup_html, max_width=260),
            tooltip=f"{row['Codigo']} ({row['Referencia']}) — {row['cluster_dom']} ({row['q_max']*100:.0f}%)",
        ).add_to(fg_pies)
    fg_pies.add_to(m_gen)

    # Leyenda flotante compacta de clusters genéticos
    items_ley = "".join(
        f'<div style="margin:2px 0;"><span style="display:inline-block;width:12px;height:12px;'
        f'background:{COLORES_ACTIVOS[i]};border:1px solid #222;vertical-align:middle;margin-right:6px;"></span>'
        f'Cluster K{i+1}</div>'
        for i in range(K_ACTIVO)
    )
    leyenda_html = (
        f'<div style="position:fixed;bottom:24px;left:24px;z-index:9999;background:rgba(255,255,255,0.94);'
        f'padding:8px 12px;border:1px solid #555;border-radius:5px;font-family:sans-serif;font-size:12px;'
        f'box-shadow:0 2px 6px rgba(0,0,0,0.2);">'
        f'<b>STRUCTURE ({ESPECIE_SELECCIONADA}, K={K_ACTIVO})</b>{items_ley}</div>'
    )
    m_gen.get_root().html.add_child(folium.Element(leyenda_html))
    folium.LayerControl(collapsed=False).add_to(m_gen)

    mapa_gen_html = f"mapa_interactivo_structure_{ESPECIE_SELECCIONADA.lower()}_K{K_ACTIVO}.html"
    m_gen.save(mapa_gen_html)
    print(f"Mapa interactivo integrado guardado como archivo independiente: '{mapa_gen_html}'")
    display(m_gen)

---

## 12. Módulo MapBiomas Chile: Diagnóstico de Amenazas de Uso del Suelo, Incendios y Vulnerabilidad Genética (Sin GEE)

Este módulo evalúa cuantitativamente las **amenazas espaciales que rodean cada localidad de muestreo y las celdas AOO ($2 \times 2\text{ km}$)** empleando los datos abiertos de **MapBiomas Chile** ([chile.mapbiomas.org](https://chile.mapbiomas.org/), licencia CC BY 4.0) **sin necesidad de utilizar Google Earth Engine (GEE)**.

### 12.1. Modos de Operación (100 % sin GEE) y Leyenda Oficial

El módulo funciona en **dos modos complementarios y automáticos**:
1. **Modo Comunal Automático (por defecto — cero descargas manuales):** Cruza espacialmente las localidades y celdas AOO con las comunas de Chile (`GADM v4.1 Nivel 3`) y con las estadísticas oficiales de **MapBiomas Chile Colección 2.0 (1999–2024)** (cobertura anual de 26 años y matriz exacta de transiciones `1999 → 2024`). Si se sube además una tabla `.csv`/`.xlsx` de **MapBiomas Fuego** exportada desde [plataforma.mapbiomas.org](https://plataforma.mapbiomas.org/projects/mapbiomas/chile), se integra automáticamente.
2. **Modo Ráster de Alta Resolución (30 m) en Buffers Multiescala + Fuego Col. 1 (opcional *plug-and-play*):** Si subes recortes GeoTIFF (`.tif` o `.zip`) descargados directamente desde el portal de MapBiomas Chile ([App de descarga LULC Col. 2.0](https://mapbiomas-chile.projects.earthengine.app/view/chile-lulc-descargas) y [MapBiomas Fuego Col. 1](https://chile.mapbiomas.org/iniciativas-y-productos/fuego/)), `rasterio` extrae los píxeles exactos de $30\text{ m}$ dentro de:
   * **Buffer local inmediato (`2 km`)** alrededor de cada localidad,
   * **Buffer de paisaje (`7,5 km`)** (misma escala espacial que la grilla hexagonal de biodiversidad), y
   * **Celdas oficiales AOO ($2 \times 2\text{ km}$)**, calculando composición, transiciones píxel a píxel y métricas de fuego (superficie quemada acumulada, recurrencia 2013–2025, año del último incendio y cobertura quemada).

#### Agrupación Oficial de Clases — MapBiomas Chile Colección 2.0
| Grupo analítico | Códigos ID oficiales Col. 2.0 | Color hex oficial | Rol en evaluación de amenazas |
| :--- | :--- | :---: | :--- |
| **Bosque Nativo (BN)** | `3` (Bosque), `59` (Primario), `60` (Secundario), `67` (Achaparrado) | `#1F8D49` | Hábitat natural remanente; su pérdida alimenta el Subcriterio B(b)(iii). |
| **Silvicultura / Plantaciones (PF)** | `9` (Silvicultura / Plantación forestal) | `#7A5900` | Principal matriz productiva exótica circundante y vector de fragmentación/fuego. |
| **Matorral (MAT)** | `66` (Matorral) | `#A89358` | Formación natural no boscosa; transiciones BN → Matorral suelen reflejar degradación o incendios. |
| **Agricultura y Pasturas (AG)** | `14`, `15` (Pastura), `18` (Agricultura) | `#E974ED` | Presión agropecuaria y cambio de uso del suelo. |
| **Infraestructura / Sin Vegetación (INF)** | `22`, `24` (Infraestructura), `23`, `25`, `61` | `#D4271E` | Expansión urbana, vial y pérdida irreversible de hábitat. |
| **Otras Naturales y Agua (OTR)** | `10`, `11` (Humedal), `12`, `29`, `63`, `26`, `33`, `34` | `#519799` | Humedales, pastizales, afloramientos rocosos, cuerpos de agua, hielo y nieve. |

### 12.2. Fórmulas y Criterios de Evaluación

1. **Cambio neto y tasa anual compuesta de deforestación (Puyravaud, 2003):**  
   Para el área de Bosque Nativo en el año inicial ($A_{t_1}$) y final ($A_{t_2}$):
   $$\Delta\%_{\text{BN}} = \frac{A_{t_2} - A_{t_1}}{A_{t_1}} \times 100, \qquad r_{\text{BN}} = \frac{1}{t_2 - t_1} \ln\!\left(\frac{A_{t_2}}{A_{t_1}}\right) \times 100 \quad (\%\cdot\text{año}^{-1})$$
2. **Conexión directa con UICN / RCE — Subcriterio B(b)(iii):**  
   Evalúa cuantitativamente la *disminución continua observada o inferida en el área, extensión y/o calidad del hábitat* a partir de $\Delta\%_{\text{BN}} < 0$, el reemplazo directo de Bosque Nativo hacia matriz antrópica (`PF`, `AG`, `INF`) y la exposición a incendios (`Fuego Col. 1`).
3. **Índice Compuesto de Prioridad de Amenaza por Localidad ($I_{\text{amenaza}} \in [0, 100]$):**  
   * **Con LULC Col. 2.0:** combina pérdida relativa de Bosque Nativo ($45\%$, saturada en $50\%$ de pérdida), presión de matriz antrópica actual (`PF + AG + INF`, $35\%$) y escasez de remanente boscoso ($100 - \%_{\text{BN}}$, $20\%$).
   * **Al incorporar MapBiomas Fuego Col. 1:** redistribuye los pesos ($30\%$ pérdida BN, $25\%$ matriz antrópica, $15\%$ escasez de remanente y **$30\%$ exposición y recurrencia de incendios**).
   * Categorías operacionales: **Crítica** ($\ge 70$), **Alta** ($50\text{–}70$), **Moderada** ($30\text{–}50$) y **Baja** ($< 30$).
4. **Síntesis `STRUCTURE` × MapBiomas (Vulnerabilidad por Unidad Genética):**  
   Cuando el Módulo 11 está activo, la exposición media de cada clúster genético $k \in \{1, \dots, K\}$ a cualquier métrica de amenaza $X_p$ se pondera por su proporción de ancestría poblacional $\bar{Q}_{pk}$:
   $$\overline{X}_k = \frac{\sum_{p=1}^{P} \bar{Q}_{pk}\, X_p}{\sum_{p=1}^{P} \bar{Q}_{pk}}$$

In [ ]:
#@title Parámetros del Módulo 12: MapBiomas Chile (Uso del Suelo y Fuego) { display-mode: "form" }
ACTIVAR_MODULO_MAPBIOMAS = True  #@param {type:"boolean"}
ANIO_INICIO_MB = 1999  #@param {type:"integer"}
ANIO_FIN_MB = 2024  #@param {type:"integer"}
RADIO_BUFFER_CORTO_M = 2000  #@param {type:"integer"}
RADIO_BUFFER_PAISAJE_M = 7500  #@param {type:"integer"}
ARCHIVOS_RASTER_LULC = ""  #@param {type:"string"}
ARCHIVO_RASTER_FUEGO_FREQ = ""  #@param {type:"string"}
ARCHIVO_RASTER_FUEGO_ULT = ""  #@param {type:"string"}
ARCHIVO_RASTER_FUEGO_COB = ""  #@param {type:"string"}
ARCHIVO_TABLA_FUEGO_COMUNAL = ""  #@param {type:"string"}

# Leyenda oficial MapBiomas Chile Colección 2.0 (Leyenda2025-LEYENDA-CODIGOS.pdf)
MB_GRUPOS_CLS = {
    "Bosque Nativo":            {3, 59, 60, 67},
    "Plantaciones Forestales":  {9},
    "Matorral":                 {66},
    "Agricultura y Pasturas":   {14, 15, 18},
    "Infraestructura / S.Veg.": {22, 23, 24, 25, 61},
    "Otras Naturales y Agua":   {10, 11, 12, 26, 29, 33, 34, 63},
}
MB_ORDEN_GRUPOS = [
    "Bosque Nativo",
    "Plantaciones Forestales",
    "Matorral",
    "Agricultura y Pasturas",
    "Infraestructura / S.Veg.",
    "Otras Naturales y Agua",
]
MB_COLORES_OFICIALES = {
    "Bosque Nativo":            "#1F8D49",
    "Plantaciones Forestales":  "#7A5900",
    "Matorral":                 "#A89358",
    "Agricultura y Pasturas":   "#E974ED",
    "Infraestructura / S.Veg.": "#D4271E",
    "Otras Naturales y Agua":   "#519799",
}
MB_CLS_A_GRUPO = {c: g for g, cs in MB_GRUPOS_CLS.items() for c in cs}
MB_BOSQUE_IDS = MB_GRUPOS_CLS["Bosque Nativo"]
MB_ALIAS_COMUNAS = {"aysen": "aisen", "paihuano": "paiguano", "marchigue": "marchihue"}

URL_MB_STATS_XLSX = "https://chile.mapbiomas.org/wp-content/uploads/sites/7/2026/08/statistics_lulc_chile_col2_political_level_234.xlsx"
URL_GADM_CHL_3 = "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_CHL_3.json.zip"

def normalizar_comuna(texto):
    '''Normaliza nombres de comunas (sin tildes, minúsculas, sin espacios) para emparejar GADM y MapBiomas.'''
    if pd.isna(texto):
        return ""
    s = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[^a-z0-9]", "", s.lower())
    return MB_ALIAS_COMUNAS.get(s, s)

def tasa_puyravaud(a1, a2, dt):
    '''Tasa anual compuesta de cambio de cobertura según Puyravaud (2003), en %/año.'''
    if dt <= 0 or a1 is None or a2 is None or a1 <= 0 or a2 <= 0:
        return np.nan
    return float((1.0 / dt) * math.log(a2 / a1) * 100.0)

def clasificar_prioridad_amenaza(score):
    if pd.isna(score):
        return "Sin dato"
    if score >= 70.0:
        return "Crítica"
    if score >= 50.0:
        return "Alta"
    if score >= 30.0:
        return "Moderada"
    return "Baja"

def calcular_indice_amenaza(perdida_bn_pct, pct_antropico, pct_bn, fuego_pct=np.nan, fuego_freq_max=np.nan, fuego_ult=np.nan, usar_fuego=False):
    '''Calcula el Índice Compuesto de Prioridad de Amenaza I_amenaza en escala [0, 100].'''
    comp_perd = float(np.clip(max(0.0, perdida_bn_pct) / 50.0 * 100.0, 0.0, 100.0)) if pd.notna(perdida_bn_pct) else 0.0
    comp_antr = float(np.clip(pct_antropico, 0.0, 100.0)) if pd.notna(pct_antropico) else 0.0
    comp_esca = float(np.clip(100.0 - pct_bn, 0.0, 100.0)) if pd.notna(pct_bn) else 50.0
    if usar_fuego and pd.notna(fuego_pct):
        # Escala de exposición a fuego: 40% del buffer/comuna quemada satura el componente base + bono por recurrencia/recencia
        base_f = float(np.clip(fuego_pct / 40.0 * 80.0, 0.0, 80.0))
        bono_rec = 10.0 if (pd.notna(fuego_freq_max) and fuego_freq_max >= 2) else 0.0
        bono_ano = 10.0 if (pd.notna(fuego_ult) and fuego_ult >= 2017) else 0.0
        comp_fuego = float(np.clip(base_f + bono_rec + bono_ano, 0.0, 100.0))
        return round(0.30 * comp_perd + 0.25 * comp_antr + 0.15 * comp_esca + 0.30 * comp_fuego, 2)
    return round(0.45 * comp_perd + 0.35 * comp_antr + 0.20 * comp_esca, 2)

def cargar_comunas_chile_gadm():
    '''Descarga y cachea límites comunales de Chile GADM v4.1 (nivel 3, 1.3 MB).'''
    local_zip = "gadm41_CHL_3.json.zip"
    if not os.path.exists(local_zip):
        print("Descargando límites comunales de Chile (GADM v4.1 Nivel 3)...")
        r = SESSION_HTTP.get(URL_GADM_CHL_3, timeout=45)
        r.raise_for_status()
        with open(local_zip, "wb") as f:
            f.write(r.content)
    gdf_c = gpd.read_file(f"zip://{local_zip}")
    gdf_c["com_key"] = gdf_c["NAME_3"].map(normalizar_comuna)
    return gdf_c[["NAME_1", "NAME_2", "NAME_3", "com_key", "geometry"]].copy()

def cargar_estadisticas_mapbiomas_col2():
    '''Carga cobertura y transiciones comunales MapBiomas Chile Col. 2.0 (1999-2024) desde repo o portal oficial.'''
    arch_cov = "mapbiomas_chile_col2_cobertura_comunal_1999_2024.csv.gz"
    arch_tr  = "mapbiomas_chile_col2_transicion_comunal_1999_2024.csv.gz"
    try:
        p_cov = resolver_archivo_entrada(arch_cov)
        p_tr  = resolver_archivo_entrada(arch_tr)
        return pd.read_csv(p_cov), pd.read_csv(p_tr)
    except FileNotFoundError:
        print("Descargando tabla oficial MapBiomas Chile Col. 2.0 directamente desde chile.mapbiomas.org (~48 MB)...")
        import openpyxl
        xlsx_local = "statistics_lulc_chile_col2_political_level_234.xlsx"
        if not os.path.exists(xlsx_local):
            with SESSION_HTTP.get(URL_MB_STATS_XLSX, stream=True, timeout=120) as r:
                r.raise_for_status()
                with open(xlsx_local, "wb") as f:
                    for chunk in r.iter_content(1024 * 1024):
                        f.write(chunk)
        wb = openpyxl.load_workbook(xlsx_local, read_only=True)
        ws = wb["COVERAGE"]
        it = ws.iter_rows(values_only=True)
        hdr = list(next(it))
        vidx = [i for i, h in enumerate(hdr) if h is not None]
        cols = [hdr[i] for i in vidx]
        rows = [[r[i] for i in vidx] for r in it if r[4] is not None and r[5] is not None]
        df_c = pd.DataFrame(rows, columns=cols)
        ycols = [c for c in df_c.columns if str(c).startswith("y")]
        for c in ycols:
            df_c[c] = pd.to_numeric(df_c[c], errors="coerce").fillna(0.0).round(2)
        df_c = df_c[["territory_level_2", "territory_level_3", "territory_level_4", "class"] + ycols]
        df_c.to_csv(arch_cov, index=False, compression="gzip")

        ws_t = wb["TRANSITION"]
        it_t = ws_t.iter_rows(values_only=True)
        hdr_t = list(next(it_t))
        keep_t = ["territory_level_2", "territory_level_4", "class_from", "class_to",
                  "p1999_2024", "p2000_2005", "p2005_2010", "p2010_2015", "p2015_2020", "p2020_2024"]
        idx_t = [hdr_t.index(c) for c in keep_t]
        rows_t = []
        for r in it_t:
            if r[4] is None or r[5] is None or r[6] is None:
                continue
            vals = [r[i] for i in idx_t]
            if any((v or 0) > 0.05 for v in vals[4:]):
                rows_t.append(vals)
        wb.close()
        df_t = pd.DataFrame(rows_t, columns=keep_t)
        for c in keep_t[4:]:
            df_t[c] = pd.to_numeric(df_t[c], errors="coerce").fillna(0.0).round(2)
        df_t.to_csv(arch_tr, index=False, compression="gzip")
        return df_c, df_t

def resolver_lista_tifs(especificacion):
    '''Acepta un .tif, varios .tif separados por coma, una carpeta o un .zip con .tif.'''
    espec = str(especificacion or "").strip()
    if not espec:
        return []
    rutas = []
    for item in [x.strip() for x in espec.split(",") if x.strip()]:
        if os.path.isdir(item):
            for fn in sorted(os.listdir(item)):
                if fn.lower().endswith((".tif", ".tiff")):
                    rutas.append(os.path.join(item, fn))
        elif os.path.isfile(item) and zipfile.is_zipfile(item):
            carp = os.path.join("mapbiomas_rasters_tmp", os.path.splitext(os.path.basename(item))[0])
            os.makedirs(carp, exist_ok=True)
            with zipfile.ZipFile(item, "r") as zf:
                for m in zf.namelist():
                    if m.lower().endswith((".tif", ".tiff")) and "__MACOSX" not in m:
                        out_f = os.path.join(carp, os.path.basename(m))
                        if not os.path.exists(out_f):
                            with open(out_f, "wb") as f_out:
                                f_out.write(zf.read(m))
                        rutas.append(out_f)
        elif os.path.isfile(item):
            rutas.append(item)
        else:
            print(f"Aviso: no se encontró el archivo o carpeta ráster '{item}'.")
    return sorted(set(rutas))

def extraer_composicion_raster_poligonos(rutas_tif, gdf_pols_ea, anio_inicio_def=1999):
    '''Extrae composición LULC MapBiomas Col. 2 (30 m) y transiciones píxel a píxel dentro de polígonos.'''
    if not RASTERIO_DISPONIBLE or not rutas_tif:
        return None, None, []
    # Indexar (anio -> (ruta, indice_banda))
    fuentes_anio = {}
    for rt in rutas_tif:
        with rasterio.open(rt) as src:
            if src.count == 1:
                m = re.search(r"(19[89]\d|20[0-2]\d)", os.path.basename(rt))
                anio = int(m.group(1)) if m else anio_inicio_def
                fuentes_anio[anio] = (rt, 1)
            else:
                descs = src.descriptions or ()
                for b in range(1, src.count + 1):
                    d = descs[b - 1] if b - 1 < len(descs) and descs[b - 1] else ""
                    m = re.search(r"(19[89]\d|20[0-2]\d)", str(d))
                    anio = int(m.group(1)) if m else (anio_inicio_def + b - 1)
                    fuentes_anio[anio] = (rt, b)
    anios_disp = sorted(fuentes_anio.keys())
    if not anios_disp:
        return None, None, []

    registros = []
    transiciones = []
    a_ini, a_fin = anios_disp[0], anios_disp[-1]

    for idx_pol, row in gdf_pols_ea.iterrows():
        geom_ea = row.geometry
        area_total_ha = float(geom_ea.area) / 10000.0
        rec = {"_idx": idx_pol, "area_eval_ha": round(area_total_ha, 2)}
        arr_ini = None
        arr_fin = None

        for anio in anios_disp:
            rt, band_idx = fuentes_anio[anio]
            with rasterio.open(rt) as src:
                geom_src = gpd.GeoSeries([geom_ea], crs=CRS_EQUAL_AREA).to_crs(src.crs).iloc[0]
                try:
                    ma, _ = rio_mask(src, [geom_src.__geo_interface__], indexes=band_idx, crop=True, filled=False)
                except ValueError:
                    continue
                valid_px = ma.compressed()
                valid_px = valid_px[(valid_px != 0) & (valid_px != 27)]
                tot_px = int(len(valid_px))
                if tot_px == 0:
                    continue
                if anio == a_ini:
                    arr_ini = ma
                if anio == a_fin:
                    arr_fin = ma
                vals, cnts = np.unique(valid_px, return_counts=True)
                grp_cnt = {g: 0 for g in MB_ORDEN_GRUPOS}
                for v, c in zip(vals, cnts):
                    g = MB_CLS_A_GRUPO.get(int(v), "Otras Naturales y Agua")
                    grp_cnt[g] += int(c)
                for g in MB_ORDEN_GRUPOS:
                    rec[f"pct_{g}_{anio}"] = round(grp_cnt[g] / tot_px * 100.0, 2)
                    rec[f"ha_{g}_{anio}"] = round(grp_cnt[g] / tot_px * area_total_ha, 2)

        # Matriz de transición píxel a píxel (si coinciden dimensiones de a_ini y a_fin)
        if arr_ini is not None and arr_fin is not None and arr_ini.shape == arr_fin.shape:
            m_ok = (~np.ma.getmaskarray(arr_ini)) & (~np.ma.getmaskarray(arr_fin))
            d1 = np.asarray(arr_ini.data)
            d2 = np.asarray(arr_fin.data)
            val_m = m_ok & (d1 != 0) & (d1 != 27) & (d2 != 0) & (d2 != 27)
            tot_tr = int(val_m.sum())
            if tot_tr > 0:
                f_vals = d1[val_m].astype(int)
                t_vals = d2[val_m].astype(int)
                pairs, p_cnts = np.unique(np.stack([f_vals, t_vals], axis=1), axis=0, return_counts=True)
                for (cf, ct), cnt in zip(pairs, p_cnts):
                    transiciones.append({
                        "_idx": idx_pol,
                        "class_from": int(cf),
                        "class_to": int(ct),
                        "ha": round(int(cnt) / tot_tr * area_total_ha, 2),
                    })
        registros.append(rec)

    return pd.DataFrame(registros), (pd.DataFrame(transiciones) if transiciones else None), anios_disp

def extraer_fuego_raster_poligonos(gdf_pols_ea, rt_freq="", rt_ult="", rt_cob=""):
    '''Extrae métricas de MapBiomas Fuego Col. 1 (2013-2025) dentro de polígonos (buffers o celdas AOO).'''
    if not RASTERIO_DISPONIBLE:
        return None
    rt_freq = rt_freq.strip() if rt_freq and os.path.isfile(rt_freq.strip()) else ""
    rt_ult  = rt_ult.strip()  if rt_ult  and os.path.isfile(rt_ult.strip())  else ""
    rt_cob  = rt_cob.strip()  if rt_cob  and os.path.isfile(rt_cob.strip())  else ""
    if not any([rt_freq, rt_ult, rt_cob]):
        return None

    filas = []
    for idx_pol, row in gdf_pols_ea.iterrows():
        geom_ea = row.geometry
        area_ha = float(geom_ea.area) / 10000.0
        rec = {
            "_idx": idx_pol,
            "fuego_pct_quemado": np.nan,
            "fuego_ha_quemada": np.nan,
            "fuego_freq_media": np.nan,
            "fuego_freq_max": np.nan,
            "fuego_ultimo_anio": np.nan,
            "fuego_pct_bn_en_quemado": np.nan,
            "fuego_pct_pf_en_quemado": np.nan,
        }
        if rt_freq:
            with rasterio.open(rt_freq) as src:
                g_s = gpd.GeoSeries([geom_ea], crs=CRS_EQUAL_AREA).to_crs(src.crs).iloc[0]
                try:
                    ma, _ = rio_mask(src, [g_s.__geo_interface__], indexes=1, crop=True, filled=False)
                    dentro = np.asarray(ma.data)[~np.ma.getmaskarray(ma)]
                    tot_in = int(len(dentro))
                    quem = dentro[dentro > 0]
                    if tot_in > 0:
                        frac = len(quem) / tot_in
                        rec["fuego_pct_quemado"] = round(frac * 100.0, 2)
                        rec["fuego_ha_quemada"] = round(frac * area_ha, 2)
                    if len(quem) > 0:
                        rec["fuego_freq_media"] = round(float(np.mean(quem)), 2)
                        rec["fuego_freq_max"] = int(np.max(quem))
                    else:
                        rec["fuego_freq_media"] = 0.0
                        rec["fuego_freq_max"] = 0
                except ValueError:
                    pass
        if rt_ult:
            with rasterio.open(rt_ult) as src:
                g_s = gpd.GeoSeries([geom_ea], crs=CRS_EQUAL_AREA).to_crs(src.crs).iloc[0]
                try:
                    ma, _ = rio_mask(src, [g_s.__geo_interface__], indexes=1, crop=True, filled=False)
                    dentro = np.asarray(ma.data)[~np.ma.getmaskarray(ma)]
                    tot_in = int(len(dentro))
                    valid = dentro[(dentro >= 1985) & (dentro <= 2030)]
                    if len(valid) > 0:
                        rec["fuego_ultimo_anio"] = int(np.max(valid))
                        if pd.isna(rec["fuego_pct_quemado"]) and tot_in > 0:
                            rec["fuego_pct_quemado"] = round(len(valid) / tot_in * 100.0, 2)
                            rec["fuego_ha_quemada"] = round(len(valid) / tot_in * area_ha, 2)
                except ValueError:
                    pass
        if rt_cob:
            with rasterio.open(rt_cob) as src:
                g_s = gpd.GeoSeries([geom_ea], crs=CRS_EQUAL_AREA).to_crs(src.crs).iloc[0]
                try:
                    ma, _ = rio_mask(src, [g_s.__geo_interface__], indexes=1, crop=True, filled=False)
                    dentro = np.asarray(ma.data)[~np.ma.getmaskarray(ma)]
                    quem_c = dentro[(dentro > 0) & (dentro != 27)]
                    if len(quem_c) > 0:
                        n_bn = int(np.isin(quem_c, list(MB_BOSQUE_IDS)).sum())
                        n_pf = int((quem_c == 9).sum())
                        rec["fuego_pct_bn_en_quemado"] = round(n_bn / len(quem_c) * 100.0, 2)
                        rec["fuego_pct_pf_en_quemado"] = round(n_pf / len(quem_c) * 100.0, 2)
                except ValueError:
                    pass
        filas.append(rec)
    return pd.DataFrame(filas)

# Estado global del Módulo 12
MAPBIOMAS_LISTO = False
MODO_MAPBIOMAS_ACTIVO = None
FUEGO_DISPONIBLE = False
gdf_mb_loc_ea = None
gdf_mb_loc_2km_ea = None
gdf_mb_comunas_ea = None
gdf_mb_aoo_ea = None
df_mb_serie_especie = None
df_mb_trans_resumen = None
df_mb_clusters_gen = None

if not ACTIVAR_MODULO_MAPBIOMAS:
    print("Módulo 12 desactivado (ACTIVAR_MODULO_MAPBIOMAS = False).")
else:
    # 1. Cargar límites comunales GADM 4.1 y datos oficiales MapBiomas Chile Col. 2.0
    gdf_comunas_wgs84 = cargar_comunas_chile_gadm()
    df_mb_cov, df_mb_tr = cargar_estadisticas_mapbiomas_col2()
    df_mb_cov["com_key"] = df_mb_cov["territory_level_4"].map(normalizar_comuna)
    df_mb_tr["com_key"]  = df_mb_tr["territory_level_4"].map(normalizar_comuna)

    # Validar rango de años disponibles en la tabla oficial
    anios_col2 = sorted([int(c[1:]) for c in df_mb_cov.columns if re.match(r"^y\d{4}$", str(c))])
    a1_tbl = max(anios_col2[0], min(ANIO_INICIO_MB, anios_col2[-1]))
    a2_tbl = max(anios_col2[0], min(ANIO_FIN_MB, anios_col2[-1]))
    if a1_tbl >= a2_tbl:
        a1_tbl, a2_tbl = anios_col2[0], anios_col2[-1]
    dt_tbl = a2_tbl - a1_tbl

    # Mapa de nombre oficial MapBiomas por com_key
    map_nombre_com = df_mb_cov.drop_duplicates("com_key").set_index("com_key")["territory_level_4"].to_dict()
    map_region_com = df_mb_cov.drop_duplicates("com_key").set_index("com_key")["territory_level_2"].to_dict()
    gdf_comunas_wgs84["mb_comuna"] = gdf_comunas_wgs84["com_key"].map(map_nombre_com)
    gdf_comunas_ea = gdf_comunas_wgs84.to_crs(CRS_EQUAL_AREA)

    # 2. Emparejar cada localidad de muestreo con su comuna chilena (con respaldo nearest para sitios costeros/fronterizos)
    loc_base = gdf_sitios_ea.copy().reset_index(drop=True)
    if "Codigo" not in loc_base.columns:
        loc_base["Codigo"] = [f"L{i+1}" for i in range(len(loc_base))]
    if "Referencia" not in loc_base.columns:
        loc_base["Referencia"] = loc_base["Codigo"]

    j_loc = gpd.sjoin(loc_base, gdf_comunas_ea[["com_key", "mb_comuna", "NAME_1", "geometry"]], how="left", predicate="intersects")
    j_loc = j_loc[~j_loc.index.duplicated(keep="first")].copy()
    miss_l = j_loc[j_loc["mb_comuna"].isna()].index
    if len(miss_l) > 0:
        j_near = gpd.sjoin_nearest(
            loc_base.loc[miss_l],
            gdf_comunas_ea[gdf_comunas_ea["mb_comuna"].notna()][["com_key", "mb_comuna", "NAME_1", "geometry"]],
            how="left",
        )
        j_near = j_near[~j_near.index.duplicated(keep="first")]
        for ix, rr in j_near.iterrows():
            j_loc.loc[ix, "com_key"] = rr["com_key"]
            j_loc.loc[ix, "mb_comuna"] = rr["mb_comuna"]
            j_loc.loc[ix, "NAME_1"] = rr["NAME_1"]
    j_loc["region_mb"] = j_loc["com_key"].map(map_region_com).fillna(j_loc["NAME_1"])

    # 3. Calcular métricas comunales MapBiomas Chile Col. 2.0 (1999-2024) para las comunas de la especie
    com_keys_sp = sorted(j_loc["com_key"].dropna().unique().tolist())
    df_cov_sp = df_mb_cov[df_mb_cov["com_key"].isin(com_keys_sp)].copy()
    df_cov_sp["grupo"] = df_cov_sp["class"].map(MB_CLS_A_GRUPO).fillna("Otras Naturales y Agua")

    # Serie temporal histórica consolidada de las comunas del área de estudio
    ycols_serie = [f"y{y}" for y in range(a1_tbl, a2_tbl + 1) if f"y{y}" in df_cov_sp.columns]
    df_mb_serie_especie = df_cov_sp.groupby("grupo")[ycols_serie].sum().reindex(MB_ORDEN_GRUPOS).fillna(0.0)

    # Métricas por comuna
    rows_com = []
    for ck in com_keys_sp:
        sub_c = df_cov_sp[df_cov_sp["com_key"] == ck]
        tot1 = float(sub_c[f"y{a1_tbl}"].sum())
        tot2 = float(sub_c[f"y{a2_tbl}"].sum())
        if tot2 <= 0:
            continue
        grp1 = sub_c.groupby("grupo")[f"y{a1_tbl}"].sum().to_dict()
        grp2 = sub_c.groupby("grupo")[f"y{a2_tbl}"].sum().to_dict()

        bn1 = float(grp1.get("Bosque Nativo", 0.0))
        bn2 = float(grp2.get("Bosque Nativo", 0.0))
        pf1 = float(grp1.get("Plantaciones Forestales", 0.0))
        pf2 = float(grp2.get("Plantaciones Forestales", 0.0))
        mat2 = float(grp2.get("Matorral", 0.0))
        ag2  = float(grp2.get("Agricultura y Pasturas", 0.0))
        inf2 = float(grp2.get("Infraestructura / S.Veg.", 0.0))
        otr2 = float(grp2.get("Otras Naturales y Agua", 0.0))

        delta_bn_pct = ((bn2 - bn1) / bn1 * 100.0) if bn1 > 0 else 0.0
        perdida_bn_pct = max(0.0, -delta_bn_pct)
        r_bn = tasa_puyravaud(bn1, bn2, dt_tbl)
        delta_pf_pct = ((pf2 - pf1) / pf1 * 100.0) if pf1 > 0 else (100.0 if pf2 > 0 else 0.0)

        # Transiciones exactas 1999 -> 2024 desde Bosque Nativo en esta comuna
        sub_tr = df_mb_tr[(df_mb_tr["com_key"] == ck) & (df_mb_tr["class_from"].isin(MB_BOSQUE_IDS)) & (~df_mb_tr["class_to"].isin(MB_BOSQUE_IDS))].copy()
        sub_tr["grupo_to"] = sub_tr["class_to"].map(MB_CLS_A_GRUPO).fillna("Otras Naturales y Agua")
        tr_by_g = sub_tr.groupby("grupo_to")["p1999_2024"].sum().to_dict()
        bn_a_pf  = float(tr_by_g.get("Plantaciones Forestales", 0.0))
        bn_a_mat = float(tr_by_g.get("Matorral", 0.0))
        bn_a_ag  = float(tr_by_g.get("Agricultura y Pasturas", 0.0))
        bn_a_inf = float(tr_by_g.get("Infraestructura / S.Veg.", 0.0))
        bn_a_otr = float(tr_by_g.get("Otras Naturales y Agua", 0.0))
        bn_perd_bruta = bn_a_pf + bn_a_mat + bn_a_ag + bn_a_inf + bn_a_otr
        vector_dom = max(
            [("Plantaciones Forestales", bn_a_pf), ("Matorral / Degradación", bn_a_mat),
             ("Agricultura / Pasturas", bn_a_ag), ("Infraestructura / S.Veg.", bn_a_inf)],
            key=lambda kv: kv[1]
        )[0] if bn_perd_bruta > 0 else "Estable"

        pct_bn2  = bn2 / tot2 * 100.0
        pct_pf2  = pf2 / tot2 * 100.0
        pct_mat2 = mat2 / tot2 * 100.0
        pct_ag2  = ag2 / tot2 * 100.0
        pct_inf2 = inf2 / tot2 * 100.0
        pct_otr2 = otr2 / tot2 * 100.0
        pct_ant2 = pct_pf2 + pct_ag2 + pct_inf2

        rows_com.append({
            "com_key": ck,
            "comuna": map_nombre_com.get(ck, ck),
            "region": map_region_com.get(ck, ""),
            "area_eval_ha": round(tot2, 1),
            "bn_ini_ha": round(bn1, 1),
            "bn_fin_ha": round(bn2, 1),
            "pct_bn_ini": round(bn1 / max(tot1, 1e-6) * 100.0, 2),
            "pct_bn_fin": round(pct_bn2, 2),
            "delta_bn_pct": round(delta_bn_pct, 2),
            "perdida_bn_pct": round(perdida_bn_pct, 2),
            "tasa_puyravaud_r": round(r_bn, 3) if pd.notna(r_bn) else np.nan,
            "pf_ini_ha": round(pf1, 1),
            "pf_fin_ha": round(pf2, 1),
            "pct_pf_ini": round(pf1 / max(tot1, 1e-6) * 100.0, 2),
            "pct_pf_fin": round(pct_pf2, 2),
            "delta_pf_pct": round(delta_pf_pct, 2),
            "pct_matorral_fin": round(pct_mat2, 2),
            "pct_agro_fin": round(pct_ag2, 2),
            "pct_infra_fin": round(pct_inf2, 2),
            "pct_otras_fin": round(pct_otr2, 2),
            "pct_antropico_fin": round(pct_ant2, 2),
            "tr_bn_a_pf_ha": round(bn_a_pf, 1),
            "tr_bn_a_mat_ha": round(bn_a_mat, 1),
            "tr_bn_a_ag_ha": round(bn_a_ag, 1),
            "tr_bn_a_inf_ha": round(bn_a_inf, 1),
            "tr_bn_a_otr_ha": round(bn_a_otr, 1),
            "tr_bn_perd_bruta_ha": round(bn_perd_bruta, 1),
            "vector_perdida_dom": vector_dom,
        })
    df_com_stats = pd.DataFrame(rows_com)

    # Resumen de transiciones consolidadas de la especie (comunal)
    df_mb_trans_resumen = pd.DataFrame([
        {"Destino": "Plantaciones Forestales",  "ha": df_com_stats["tr_bn_a_pf_ha"].sum(),  "color": MB_COLORES_OFICIALES["Plantaciones Forestales"]},
        {"Destino": "Matorral",                 "ha": df_com_stats["tr_bn_a_mat_ha"].sum(), "color": MB_COLORES_OFICIALES["Matorral"]},
        {"Destino": "Agricultura y Pasturas",   "ha": df_com_stats["tr_bn_a_ag_ha"].sum(),  "color": MB_COLORES_OFICIALES["Agricultura y Pasturas"]},
        {"Destino": "Infraestructura / S.Veg.", "ha": df_com_stats["tr_bn_a_inf_ha"].sum(), "color": MB_COLORES_OFICIALES["Infraestructura / S.Veg."]},
        {"Destino": "Otras Naturales y Agua",   "ha": df_com_stats["tr_bn_a_otr_ha"].sum(), "color": MB_COLORES_OFICIALES["Otras Naturales y Agua"]},
    ]).sort_values("ha", ascending=False).reset_index(drop=True)

    # 4. Construir buffers multiescala (2 km y 7,5 km) alrededor de cada localidad
    gdf_buf_paisaje_ea = j_loc.drop(columns=["index_right"], errors="ignore").copy()
    gdf_buf_paisaje_ea["geometry"] = gdf_buf_paisaje_ea.geometry.buffer(RADIO_BUFFER_PAISAJE_M)
    gdf_buf_corto_ea = j_loc.drop(columns=["index_right"], errors="ignore").copy()
    gdf_buf_corto_ea["geometry"] = gdf_buf_corto_ea.geometry.buffer(RADIO_BUFFER_CORTO_M)

    # Unir estadísticas comunales como base (siempre disponibles)
    gdf_mb_loc_ea = gdf_buf_paisaje_ea.merge(df_com_stats, on="com_key", how="left")
    gdf_mb_loc_ea["escala_lulc"] = f"Comunal MapBiomas Col.2 ({a1_tbl}-{a2_tbl})"
    MODO_MAPBIOMAS_ACTIVO = "COMUNAL_COL2"

    # 5. MODO RASTER 30 m (opcional, si el usuario subió GeoTIFFs de MapBiomas Col. 2.0)
    tifs_lulc = resolver_lista_tifs(ARCHIVOS_RASTER_LULC)
    if tifs_lulc:
        print(f"Modo Ráster 30 m detectado ({len(tifs_lulc)} GeoTIFF LULC). Extrayendo buffers de {RADIO_BUFFER_CORTO_M/1000:.1f} km, {RADIO_BUFFER_PAISAJE_M/1000:.1f} km y celdas AOO...")
        res_p75 = extraer_composicion_raster_poligonos(tifs_lulc, gdf_buf_paisaje_ea, ANIO_INICIO_MB)
        res_p20 = extraer_composicion_raster_poligonos(tifs_lulc, gdf_buf_corto_ea, ANIO_INICIO_MB)
        if res_p75 is not None and res_p75[0] is not None and not res_p75[0].empty:
            df_r75, df_tr_r75, anios_r = res_p75
            a_r1, a_r2 = anios_r[0], anios_r[-1]
            dt_r = max(1, a_r2 - a_r1)
            # Reemplazar métricas primarias con valores exactos de 30 m en buffer 7,5 km
            for _, rr in df_r75.iterrows():
                ix = int(rr["_idx"])
                if f"pct_Bosque Nativo_{a_r2}" not in rr or pd.isna(rr.get(f"pct_Bosque Nativo_{a_r2}")):
                    continue
                bn1_r = float(rr.get(f"ha_Bosque Nativo_{a_r1}", 0.0))
                bn2_r = float(rr.get(f"ha_Bosque Nativo_{a_r2}", 0.0))
                pf1_r = float(rr.get(f"ha_Plantaciones Forestales_{a_r1}", 0.0))
                pf2_r = float(rr.get(f"ha_Plantaciones Forestales_{a_r2}", 0.0))
                pct_bn2_r  = float(rr.get(f"pct_Bosque Nativo_{a_r2}", 0.0))
                pct_pf2_r  = float(rr.get(f"pct_Plantaciones Forestales_{a_r2}", 0.0))
                pct_mat2_r = float(rr.get(f"pct_Matorral_{a_r2}", 0.0))
                pct_ag2_r  = float(rr.get(f"pct_Agricultura y Pasturas_{a_r2}", 0.0))
                pct_inf2_r = float(rr.get(f"pct_Infraestructura / S.Veg._{a_r2}", 0.0))
                pct_otr2_r = float(rr.get(f"pct_Otras Naturales y Agua_{a_r2}", 0.0))

                d_bn = ((bn2_r - bn1_r) / bn1_r * 100.0) if (a_r2 > a_r1 and bn1_r > 0) else 0.0
                d_pf = ((pf2_r - pf1_r) / pf1_r * 100.0) if (a_r2 > a_r1 and pf1_r > 0) else 0.0
                gdf_mb_loc_ea.loc[ix, "area_eval_ha"] = rr["area_eval_ha"]
                gdf_mb_loc_ea.loc[ix, "bn_ini_ha"] = round(bn1_r, 1)
                gdf_mb_loc_ea.loc[ix, "bn_fin_ha"] = round(bn2_r, 1)
                gdf_mb_loc_ea.loc[ix, "pct_bn_ini"] = round(float(rr.get(f"pct_Bosque Nativo_{a_r1}", pct_bn2_r)), 2)
                gdf_mb_loc_ea.loc[ix, "pct_bn_fin"] = round(pct_bn2_r, 2)
                gdf_mb_loc_ea.loc[ix, "delta_bn_pct"] = round(d_bn, 2)
                gdf_mb_loc_ea.loc[ix, "perdida_bn_pct"] = round(max(0.0, -d_bn), 2)
                gdf_mb_loc_ea.loc[ix, "tasa_puyravaud_r"] = round(tasa_puyravaud(bn1_r, bn2_r, dt_r), 3) if a_r2 > a_r1 else np.nan
                gdf_mb_loc_ea.loc[ix, "pf_ini_ha"] = round(pf1_r, 1)
                gdf_mb_loc_ea.loc[ix, "pf_fin_ha"] = round(pf2_r, 1)
                gdf_mb_loc_ea.loc[ix, "pct_pf_ini"] = round(float(rr.get(f"pct_Plantaciones Forestales_{a_r1}", pct_pf2_r)), 2)
                gdf_mb_loc_ea.loc[ix, "pct_pf_fin"] = round(pct_pf2_r, 2)
                gdf_mb_loc_ea.loc[ix, "delta_pf_pct"] = round(d_pf, 2)
                gdf_mb_loc_ea.loc[ix, "pct_matorral_fin"] = round(pct_mat2_r, 2)
                gdf_mb_loc_ea.loc[ix, "pct_agro_fin"] = round(pct_ag2_r, 2)
                gdf_mb_loc_ea.loc[ix, "pct_infra_fin"] = round(pct_inf2_r, 2)
                gdf_mb_loc_ea.loc[ix, "pct_otras_fin"] = round(pct_otr2_r, 2)
                gdf_mb_loc_ea.loc[ix, "pct_antropico_fin"] = round(pct_pf2_r + pct_ag2_r + pct_inf2_r, 2)
                gdf_mb_loc_ea.loc[ix, "escala_lulc"] = f"Ráster 30 m buffer {RADIO_BUFFER_PAISAJE_M/1000:.1f} km ({a_r1}-{a_r2})"

            # Transiciones píxel-a-píxel desde Bosque Nativo en buffers 7,5 km
            if df_tr_r75 is not None and not df_tr_r75.empty:
                tr_bn = df_tr_r75[df_tr_r75["class_from"].isin(MB_BOSQUE_IDS) & (~df_tr_r75["class_to"].isin(MB_BOSQUE_IDS))].copy()
                tr_bn["grupo_to"] = tr_bn["class_to"].map(MB_CLS_A_GRUPO).fillna("Otras Naturales y Agua")
                for ix in gdf_mb_loc_ea.index:
                    sub_ix = tr_bn[tr_bn["_idx"] == ix].groupby("grupo_to")["ha"].sum().to_dict()
                    gdf_mb_loc_ea.loc[ix, "tr_bn_a_pf_ha"]  = round(float(sub_ix.get("Plantaciones Forestales", 0.0)), 1)
                    gdf_mb_loc_ea.loc[ix, "tr_bn_a_mat_ha"] = round(float(sub_ix.get("Matorral", 0.0)), 1)
                    gdf_mb_loc_ea.loc[ix, "tr_bn_a_ag_ha"]  = round(float(sub_ix.get("Agricultura y Pasturas", 0.0)), 1)
                    gdf_mb_loc_ea.loc[ix, "tr_bn_a_inf_ha"] = round(float(sub_ix.get("Infraestructura / S.Veg.", 0.0)), 1)
                    gdf_mb_loc_ea.loc[ix, "tr_bn_a_otr_ha"] = round(float(sub_ix.get("Otras Naturales y Agua", 0.0)), 1)
                df_mb_trans_resumen = pd.DataFrame([
                    {"Destino": g, "ha": float(tr_bn[tr_bn["grupo_to"] == g]["ha"].sum()), "color": MB_COLORES_OFICIALES[g]}
                    for g in MB_ORDEN_GRUPOS if g != "Bosque Nativo"
                ]).sort_values("ha", ascending=False).reset_index(drop=True)

            # Guardar también resumen de buffer corto (2 km)
            if res_p20 is not None and res_p20[0] is not None and not res_p20[0].empty:
                df_r20 = res_p20[0]
                gdf_mb_loc_2km_ea = gdf_buf_corto_ea.copy()
                for col_s, col_d in [
                    (f"pct_Bosque Nativo_{a_r2}", "pct_bn_2km"),
                    (f"pct_Plantaciones Forestales_{a_r2}", "pct_pf_2km"),
                    (f"pct_Matorral_{a_r2}", "pct_matorral_2km"),
                    (f"pct_Agricultura y Pasturas_{a_r2}", "pct_agro_2km"),
                    (f"pct_Infraestructura / S.Veg._{a_r2}", "pct_infra_2km"),
                ]:
                    if col_s in df_r20.columns:
                        mapa_v = df_r20.set_index("_idx")[col_s].to_dict()
                        gdf_mb_loc_2km_ea[col_d] = gdf_mb_loc_2km_ea.index.map(mapa_v)
                        gdf_mb_loc_ea[col_d] = gdf_mb_loc_ea.index.map(mapa_v)

            MODO_MAPBIOMAS_ACTIVO = f"RASTER_30M ({a_r1}-{a_r2})"
            a1_tbl, a2_tbl = a_r1, a_r2

    # 6. MAPBIOMAS FUEGO COLECCION 1 (2013-2025) — Ráster 30 m y/o tabla comunal exportada
    for col_f in ["fuego_pct_quemado", "fuego_ha_quemada", "fuego_freq_media", "fuego_freq_max",
                  "fuego_ultimo_anio", "fuego_pct_bn_en_quemado", "fuego_pct_pf_en_quemado"]:
        gdf_mb_loc_ea[col_f] = np.nan

    df_fuego_r = extraer_fuego_raster_poligonos(
        gdf_buf_paisaje_ea, ARCHIVO_RASTER_FUEGO_FREQ, ARCHIVO_RASTER_FUEGO_ULT, ARCHIVO_RASTER_FUEGO_COB
    )
    if df_fuego_r is not None and df_fuego_r["fuego_pct_quemado"].notna().any():
        FUEGO_DISPONIBLE = True
        for col_f in ["fuego_pct_quemado", "fuego_ha_quemada", "fuego_freq_media", "fuego_freq_max",
                      "fuego_ultimo_anio", "fuego_pct_bn_en_quemado", "fuego_pct_pf_en_quemado"]:
            mapa_f = df_fuego_r.set_index("_idx")[col_f].to_dict()
            gdf_mb_loc_ea[col_f] = gdf_mb_loc_ea.index.map(mapa_f)
        print("MapBiomas Fuego Colección 1 (ráster 30 m) integrado en buffers de localidades.")
    elif ARCHIVO_TABLA_FUEGO_COMUNAL.strip() and os.path.isfile(ARCHIVO_TABLA_FUEGO_COMUNAL.strip()):
        ruta_ft = ARCHIVO_TABLA_FUEGO_COMUNAL.strip()
        df_ft = pd.read_excel(ruta_ft) if ruta_ft.lower().endswith((".xlsx", ".xls")) else cargar_csv_localidades(ruta_ft)
        col_com_f = next((c for c in df_ft.columns if any(k in str(c).lower() for k in ("territory_level_4", "comuna", "municip", "name"))), None)
        col_val_f = next((c for c in df_ft.columns if any(k in str(c).lower() for k in ("pct", "porc", "burned", "quem", "area", "ha"))), None)
        if col_com_f and col_val_f:
            df_ft["_ck"] = df_ft[col_com_f].map(normalizar_comuna)
            df_ft["_val"] = pd.to_numeric(df_ft[col_val_f], errors="coerce").fillna(0.0)
            agg_f = df_ft.groupby("_ck")["_val"].sum().to_dict()
            es_pct = any(k in str(col_val_f).lower() for k in ("pct", "porc", "%"))
            for ix, row in gdf_mb_loc_ea.iterrows():
                val_f = agg_f.get(row["com_key"], np.nan)
                if pd.notna(val_f):
                    if es_pct:
                        gdf_mb_loc_ea.loc[ix, "fuego_pct_quemado"] = round(val_f, 2)
                    else:
                        gdf_mb_loc_ea.loc[ix, "fuego_ha_quemada"] = round(val_f, 1)
                        gdf_mb_loc_ea.loc[ix, "fuego_pct_quemado"] = round(val_f / max(float(row["area_eval_ha"]), 1.0) * 100.0, 2)
            FUEGO_DISPONIBLE = gdf_mb_loc_ea["fuego_pct_quemado"].notna().any()
            if FUEGO_DISPONIBLE:
                print(f"Tabla de MapBiomas Fuego '{ruta_ft}' integrada por comuna.")

    # 7. Calcular Índice Compuesto de Prioridad de Amenaza I_amenaza (0-100)
    gdf_mb_loc_ea["indice_amenaza"] = [
        calcular_indice_amenaza(
            r["perdida_bn_pct"], r["pct_antropico_fin"], r["pct_bn_fin"],
            r["fuego_pct_quemado"], r["fuego_freq_max"], r["fuego_ultimo_anio"],
            usar_fuego=FUEGO_DISPONIBLE,
        )
        for _, r in gdf_mb_loc_ea.iterrows()
    ]
    gdf_mb_loc_ea["prioridad_conservacion"] = gdf_mb_loc_ea["indice_amenaza"].map(clasificar_prioridad_amenaza)

    # 8. Enriquecer capa comunal y celdas oficiales AOO (2x2 km) con amenazas MapBiomas
    gdf_mb_comunas_ea = gdf_comunas_ea[gdf_comunas_ea["com_key"].isin(com_keys_sp)].merge(df_com_stats, on="com_key", how="inner")
    gdf_mb_comunas_ea["indice_amenaza"] = [
        calcular_indice_amenaza(r["perdida_bn_pct"], r["pct_antropico_fin"], r["pct_bn_fin"], usar_fuego=False)
        for _, r in gdf_mb_comunas_ea.iterrows()
    ]
    gdf_mb_comunas_ea["prioridad_conservacion"] = gdf_mb_comunas_ea["indice_amenaza"].map(clasificar_prioridad_amenaza)

    aoo_cent = gdf_aoo_uicn_ea.copy()
    aoo_cent["geometry"] = aoo_cent.geometry.centroid
    j_aoo = gpd.sjoin(aoo_cent[["celda_id", "geometry"]], gdf_comunas_ea[["com_key", "mb_comuna", "geometry"]], how="left", predicate="intersects")
    j_aoo = j_aoo[~j_aoo.index.duplicated(keep="first")]
    gdf_mb_aoo_ea = gdf_aoo_uicn_ea.copy()
    gdf_mb_aoo_ea["com_key"] = gdf_mb_aoo_ea.index.map(j_aoo["com_key"])
    gdf_mb_aoo_ea = gdf_mb_aoo_ea.merge(
        df_com_stats[["com_key", "comuna", "pct_bn_fin", "pct_pf_fin", "delta_bn_pct", "perdida_bn_pct", "pct_antropico_fin", "vector_perdida_dom"]],
        on="com_key", how="left"
    )
    if tifs_lulc:
        res_aoo_r = extraer_composicion_raster_poligonos(tifs_lulc, gdf_aoo_uicn_ea.reset_index(drop=True), ANIO_INICIO_MB)
        if res_aoo_r is not None and res_aoo_r[0] is not None and not res_aoo_r[0].empty:
            df_ra, _, anios_ra = res_aoo_r
            ar1, ar2 = anios_ra[0], anios_ra[-1]
            for ix, rr in df_ra.iterrows():
                p_idx = int(rr["_idx"])
                if f"pct_Bosque Nativo_{ar2}" in rr and pd.notna(rr[f"pct_Bosque Nativo_{ar2}"]):
                    bn1_a = float(rr.get(f"ha_Bosque Nativo_{ar1}", 0.0))
                    bn2_a = float(rr.get(f"ha_Bosque Nativo_{ar2}", 0.0))
                    d_a = ((bn2_a - bn1_a) / bn1_a * 100.0) if (ar2 > ar1 and bn1_a > 0) else 0.0
                    gdf_mb_aoo_ea.loc[p_idx, "pct_bn_fin"] = round(float(rr[f"pct_Bosque Nativo_{ar2}"]), 2)
                    gdf_mb_aoo_ea.loc[p_idx, "pct_pf_fin"] = round(float(rr.get(f"pct_Plantaciones Forestales_{ar2}", 0.0)), 2)
                    gdf_mb_aoo_ea.loc[p_idx, "delta_bn_pct"] = round(d_a, 2)
                    gdf_mb_aoo_ea.loc[p_idx, "perdida_bn_pct"] = round(max(0.0, -d_a), 2)
    if FUEGO_DISPONIBLE and any([ARCHIVO_RASTER_FUEGO_FREQ, ARCHIVO_RASTER_FUEGO_ULT, ARCHIVO_RASTER_FUEGO_COB]):
        df_fa = extraer_fuego_raster_poligonos(
            gdf_aoo_uicn_ea.reset_index(drop=True), ARCHIVO_RASTER_FUEGO_FREQ, ARCHIVO_RASTER_FUEGO_ULT, ARCHIVO_RASTER_FUEGO_COB
        )
        if df_fa is not None:
            for cf in ["fuego_pct_quemado", "fuego_freq_max", "fuego_ultimo_anio"]:
                gdf_mb_aoo_ea[cf] = gdf_mb_aoo_ea.index.map(df_fa.set_index("_idx")[cf].to_dict())

    # 9. SINTESIS STRUCTURE x MAPBIOMAS (si Módulo 11 está activo)
    if globals().get("STRUCTURE_LISTO", False) and globals().get("gdf_struct_pop_ea") is not None:
        cols_st = ["Codigo", "Pop", "n_ind", "cluster_dom", "q_max", "estado_mezcla", "entropia_q"] + K_COLS_ACTIVOS
        cols_st = [c for c in cols_st if c in gdf_struct_pop_ea.columns]
        gdf_mb_loc_ea = gdf_mb_loc_ea.merge(
            gdf_struct_pop_ea[cols_st].drop_duplicates("Codigo"), on="Codigo", how="left"
        )
        rows_k = []
        for i_k, k_col in enumerate(K_COLS_ACTIVOS):
            w = pd.to_numeric(gdf_mb_loc_ea[k_col], errors="coerce").fillna(0.0).to_numpy()
            w_sum = float(w.sum())
            if w_sum <= 1e-9:
                continue
            def wmean(col_name):
                vals = pd.to_numeric(gdf_mb_loc_ea[col_name], errors="coerce").to_numpy()
                ok = ~np.isnan(vals)
                return round(float(np.sum(w[ok] * vals[ok]) / np.sum(w[ok])), 2) if np.sum(w[ok]) > 0 else np.nan

            n_dom = int((gdf_mb_loc_ea["cluster_dom"] == k_col).sum())
            locs_dom = ", ".join(gdf_mb_loc_ea[gdf_mb_loc_ea["cluster_dom"] == k_col]["Codigo"].astype(str).tolist()) or "-"
            row_k = {
                "Cluster": k_col,
                "Color": COLORES_ACTIVOS[i_k],
                "n_loc_dom": n_dom,
                "localidades_dom": locs_dom,
                "pct_bn_2024_pond": wmean("pct_bn_fin"),
                "perdida_bn_pct_pond": wmean("perdida_bn_pct"),
                "pct_plantaciones_pond": wmean("pct_pf_fin"),
                "pct_antropico_pond": wmean("pct_antropico_fin"),
                "indice_amenaza_pond": wmean("indice_amenaza"),
            }
            if FUEGO_DISPONIBLE:
                row_k["fuego_pct_quemado_pond"] = wmean("fuego_pct_quemado")
            row_k["prioridad_cluster"] = clasificar_prioridad_amenaza(row_k["indice_amenaza_pond"])
            rows_k.append(row_k)
        df_mb_clusters_gen = pd.DataFrame(rows_k).sort_values("indice_amenaza_pond", ascending=False).reset_index(drop=True)

    # 10. Evaluación cuantitativa del Subcriterio UICN / RCE B(b)(iii)
    bn_tot_1 = float(df_mb_serie_especie.loc["Bosque Nativo", ycols_serie[0]])
    bn_tot_2 = float(df_mb_serie_especie.loc["Bosque Nativo", ycols_serie[-1]])
    pf_tot_1 = float(df_mb_serie_especie.loc["Plantaciones Forestales", ycols_serie[0]])
    pf_tot_2 = float(df_mb_serie_especie.loc["Plantaciones Forestales", ycols_serie[-1]])
    delta_bn_sp = (bn_tot_2 - bn_tot_1) / max(bn_tot_1, 1e-6) * 100.0
    r_bn_sp = tasa_puyravaud(bn_tot_1, bn_tot_2, a2_tbl - a1_tbl)
    n_loc_perdida = int((gdf_mb_loc_ea["delta_bn_pct"] < -1.0).sum())
    pct_pf_medio = float(gdf_mb_loc_ea["pct_pf_fin"].mean())

    cumple_b_iii = (delta_bn_sp < -1.0) or (n_loc_perdida >= max(1, len(gdf_mb_loc_ea) // 3)) or (pct_pf_medio >= 30.0)
    veredicto_b_iii = (
        "EVIDENCIA CUANTITATIVA DE DISMINUCIÓN CONTINUA DEL HÁBITAT — Subcriterio B(b)(iii) RESPALDADO"
        if cumple_b_iii else
        "Pérdida neta moderada/estable a escala comunal — verificar degradación fina en rodales (modo ráster 30 m)"
    )

    # 11. Exportar capas MapBiomas al GeoPackage (.gpkg) del curso y CSV resumen
    gdf_mb_loc_ea.to_crs(CRS_EXPORT_UTM).to_file(ARCHIVO_SALIDA, layer="mapbiomas_amenazas_localidades", driver="GPKG")
    gdf_mb_comunas_ea.to_crs(CRS_EXPORT_UTM).to_file(ARCHIVO_SALIDA, layer="mapbiomas_amenazas_comunas", driver="GPKG")
    gdf_mb_aoo_ea.to_crs(CRS_EXPORT_UTM).to_file(ARCHIVO_SALIDA, layer="mapbiomas_amenazas_aoo", driver="GPKG")
    if gdf_mb_loc_2km_ea is not None:
        gdf_mb_loc_2km_ea.to_crs(CRS_EXPORT_UTM).to_file(ARCHIVO_SALIDA, layer="mapbiomas_amenazas_localidades_2km", driver="GPKG")

    csv_mb_loc = f"resumen_amenazas_mapbiomas_{ESPECIE_SELECCIONADA.lower()}_{sufijo_escala}.csv"
    cols_csv = [c for c in gdf_mb_loc_ea.columns if c != "geometry"]
    gdf_mb_loc_ea[cols_csv].to_csv(csv_mb_loc, index=False, encoding="utf-8-sig")
    if df_mb_clusters_gen is not None:
        csv_mb_cl = f"vulnerabilidad_clusters_mapbiomas_{ESPECIE_SELECCIONADA.lower()}_K{K_ACTIVO}.csv"
        df_mb_clusters_gen.to_csv(csv_mb_cl, index=False, encoding="utf-8-sig")

    MAPBIOMAS_LISTO = True

    print(f"================ DIAGNÓSTICO MAPBIOMAS CHILE ({ESPECIE_SELECCIONADA}) ================")
    print(f"  Modo de análisis activo              : {MODO_MAPBIOMAS_ACTIVO} | Fuego Col. 1: {'ACTIVO' if FUEGO_DISPONIBLE else 'No cargado (opcional)'}")
    print(f"  Comunas analizadas (GADM v4.1)       : {len(com_keys_sp)} ({', '.join(sorted(df_com_stats['comuna'].tolist()))})")
    print(f"  Bosque Nativo ({a1_tbl} -> {a2_tbl})       : {bn_tot_1:,.0f} -> {bn_tot_2:,.0f} ha ({delta_bn_sp:+.1f}% | r = {r_bn_sp:+.3f} %/año)")
    print(f"  Plantaciones ({a1_tbl} -> {a2_tbl})        : {pf_tot_1:,.0f} -> {pf_tot_2:,.0f} ha ({(pf_tot_2-pf_tot_1)/max(pf_tot_1,1)*100:+.1f}%)")
    print(f"  Localidades con pérdida BN > 1%      : {n_loc_perdida} / {len(gdf_mb_loc_ea)}")
    print(f"  Subcriterio UICN/RCE B(b)(iii)       : {veredicto_b_iii}")
    print(f"  Capas añadidas a QGIS (.gpkg)        : mapbiomas_amenazas_localidades, mapbiomas_amenazas_comunas, mapbiomas_amenazas_aoo")
    print(f"  Tabla exportada                      : '{csv_mb_loc}'")
    if not tifs_lulc:
        print("  [Tip 30 m sin GEE] Para activar análisis ráster en buffers de 2 km y 7,5 km + Fuego Col. 1, sube tus .tif descargados")
        print("  desde https://mapbiomas-chile.projects.earthengine.app/view/chile-lulc-descargas e indícalos en los parámetros arriba.")

    cols_ver = [c for c in ["Codigo", "Referencia", "comuna", "pct_bn_fin", "delta_bn_pct", "tasa_puyravaud_r",
                            "pct_pf_fin", "pct_antropico_fin", "fuego_pct_quemado", "cluster_dom",
                            "indice_amenaza", "prioridad_conservacion"] if c in gdf_mb_loc_ea.columns]
    if not FUEGO_DISPONIBLE and "fuego_pct_quemado" in cols_ver:
        cols_ver.remove("fuego_pct_quemado")
    print("\n--- Resumen de amenazas por localidad (ordenado por prioridad) ---")
    display(gdf_mb_loc_ea[cols_ver].sort_values("indice_amenaza", ascending=False).reset_index(drop=True))
    if df_mb_clusters_gen is not None:
        print(f"\n--- Síntesis STRUCTURE × MapBiomas: Vulnerabilidad ponderada por Clúster Genético (K = {K_ACTIVO}) ---")
        display(df_mb_clusters_gen.drop(columns=["Color"]))

### 12.2. Figuras de Análisis de Paisaje, Trayectoria Histórica y Vulnerabilidad Genética (300 dpi)

Se generan dos láminas listas para informes y presentaciones (exportadas en **PNG a 300 dpi** y **SVG vectorial**):
1. **Figura 12A — Paisaje y Cambio de Uso del Suelo (MapBiomas Chile Col. 2.0):**
   * **Panel A:** Composición actual del paisaje (`2024`) por localidad ordenada de Norte a Sur (paleta oficial MapBiomas Chile).
   * **Panel B:** Trayectoria histórica `1999–2024` (26 años) de Bosque Nativo vs. Plantaciones Forestales y otros usos.
   * **Panel C:** Destino de la pérdida de Bosque Nativo (`1999 → 2024`: reemplazo hacia plantaciones, degradación a matorral, agricultura o infraestructura).
2. **Figura 12B — Cuadrante de Riesgo y Vulnerabilidad por Unidad Genética (`STRUCTURE` × MapBiomas):**
   * **Panel A:** Cuadrante de riesgo por localidad (presión de plantaciones o incendios vs. pérdida de Bosque Nativo), coloreado por el clúster genético dominante de `STRUCTURE` (cuando el Módulo 11 está activo) y con tamaño proporcional a $I_{\text{amenaza}}$.
   * **Panel B:** Perfil de vulnerabilidad promedio ponderado por ancestría ($\overline{X}_k$) para cada clúster genético ($K_1 \dots K_K$), o ranking de prioridad de conservación por localidad si el Módulo 11 no está activo.

In [ ]:
if not globals().get("MAPBIOMAS_LISTO", False):
    print("Ejecuta primero la celda 12.1 con ACTIVAR_MODULO_MAPBIOMAS = True.")
else:
    # ==========================================================================
    # FIGURA 12A: COMPOSICION 2024 + TRAYECTORIA 1999-2024 + TRANSICIONES BN
    # ==========================================================================
    df_ord = gdf_mb_loc_ea.sort_values("Lat", ascending=False).reset_index(drop=True)

    fig1 = plt.figure(figsize=(15.5, 6.4), dpi=140, facecolor="white")
    gs1 = gridspec.GridSpec(2, 2, width_ratios=[1.25, 1.0], height_ratios=[1.15, 1.0], wspace=0.28, hspace=0.42)
    ax_a = fig1.add_subplot(gs1[:, 0])
    ax_b = fig1.add_subplot(gs1[0, 1])
    ax_c = fig1.add_subplot(gs1[1, 1])

    # --- Panel A: Barras 100% apiladas por localidad (Norte -> Sur) ---
    col_map_pct = [
        ("Bosque Nativo",            "pct_bn_fin"),
        ("Plantaciones Forestales",  "pct_pf_fin"),
        ("Matorral",                 "pct_matorral_fin"),
        ("Agricultura y Pasturas",   "pct_agro_fin"),
        ("Infraestructura / S.Veg.", "pct_infra_fin"),
        ("Otras Naturales y Agua",   "pct_otras_fin"),
    ]
    y_pos = np.arange(len(df_ord))
    left_arr = np.zeros(len(df_ord), dtype=float)
    for grp_name, col_pct in col_map_pct:
        vals = pd.to_numeric(df_ord[col_pct], errors="coerce").fillna(0.0).to_numpy()
        ax_a.barh(y_pos, vals, left=left_arr, height=0.72, color=MB_COLORES_OFICIALES[grp_name],
                  edgecolor="white", linewidth=0.5, label=grp_name)
        left_arr += vals

    etiquetas_loc = [f"{r['Codigo']}  ({r['comuna']})" for _, r in df_ord.iterrows()]
    ax_a.set_yticks(y_pos)
    ax_a.set_yticklabels(etiquetas_loc, fontsize=8.5, fontweight="bold")
    ax_a.invert_yaxis()
    ax_a.set_xlim(0, 100)
    ax_a.set_xlabel(f"Cobertura y uso del suelo {a2_tbl} (%)", fontsize=9)
    ax_a.set_title(f"A   Composición del paisaje por localidad (Norte → Sur | {MODO_MAPBIOMAS_ACTIVO})",
                   loc="left", fontsize=10, fontweight="bold", pad=8)
    ax_a.grid(axis="x", linestyle=":", alpha=0.45)
    ax_a.spines["top"].set_visible(False)
    ax_a.spines["right"].set_visible(False)
    ax_a.legend(loc="upper center", bbox_to_anchor=(0.5, -0.11), ncol=3, fontsize=8, frameon=True, edgecolor="#CCCCCC")

    # --- Panel B: Trayectoria histórica 1999-2024 ---
    anios_x = [int(c[1:]) for c in df_mb_serie_especie.columns]
    y_max_b = 0.0
    for grp_plot, lw in [("Bosque Nativo", 2.4), ("Plantaciones Forestales", 2.2), ("Matorral", 1.5), ("Agricultura y Pasturas", 1.5)]:
        serie_kha = df_mb_serie_especie.loc[grp_plot].to_numpy(dtype=float) / 1000.0
        y_max_b = max(y_max_b, float(np.nanmax(serie_kha)) if len(serie_kha) else 0.0)
        ax_b.plot(anios_x, serie_kha, color=MB_COLORES_OFICIALES[grp_plot], linewidth=lw, label=grp_plot)
    ax_b.set_ylim(0, max(1.0, y_max_b * 1.28))
    ax_b.set_ylabel("Superficie (miles ha)", fontsize=8.5)
    ax_b.set_title(f"B   Trayectoria {anios_x[0]}–{anios_x[-1]} ({ESPECIE_SELECCIONADA}: BN {delta_bn_sp:+.1f}%, r={r_bn_sp:+.2f}%/año)",
                   loc="left", fontsize=9.5, fontweight="bold", pad=6)
    ax_b.grid(True, linestyle=":", alpha=0.45)
    ax_b.spines["top"].set_visible(False)
    ax_b.spines["right"].set_visible(False)
    ax_b.tick_params(labelsize=8)
    ax_b.legend(loc="upper center", fontsize=7.6, ncol=2, frameon=True, framealpha=0.95, edgecolor="#CCCCCC")

    # --- Panel C: Destino de la pérdida de Bosque Nativo (1999 -> 2024) ---
    tr_plot = df_mb_trans_resumen[df_mb_trans_resumen["ha"] > 0].copy()
    if not tr_plot.empty:
        tot_tr_ha = tr_plot["ha"].sum()
        yp_c = np.arange(len(tr_plot))
        ax_c.barh(yp_c, tr_plot["ha"] / 1000.0, color=tr_plot["color"], edgecolor="#333333", linewidth=0.6, height=0.65)
        for i_t, r_t in tr_plot.iterrows():
            pct_t = r_t["ha"] / max(tot_tr_ha, 1e-6) * 100.0
            ax_c.text(r_t["ha"] / 1000.0 + (tr_plot["ha"].max() / 1000.0) * 0.02, i_t,
                      f"{r_t['ha']:,.0f} ha ({pct_t:.1f}%)", va="center", fontsize=8, color="#222")
        ax_c.set_yticks(yp_c)
        ax_c.set_yticklabels(tr_plot["Destino"], fontsize=8.5)
        ax_c.invert_yaxis()
        ax_c.set_xlim(0, (tr_plot["ha"].max() / 1000.0) * 1.38)
    ax_c.set_xlabel("Bosque Nativo reemplazado (miles ha)", fontsize=8.5)
    ax_c.set_title(f"C   Destino de la pérdida de Bosque Nativo ({a1_tbl} → {a2_tbl})",
                   loc="left", fontsize=9.5, fontweight="bold", pad=6)
    ax_c.grid(axis="x", linestyle=":", alpha=0.45)
    ax_c.spines["top"].set_visible(False)
    ax_c.spines["right"].set_visible(False)
    ax_c.tick_params(labelsize=8)

    fig1.suptitle(
        f"MapBiomas Chile Colección 2.0 — Diagnóstico de Paisaje y Uso del Suelo: {ESPECIE_SELECCIONADA} ({nombre_cientifico})",
        fontsize=11.5, fontweight="bold", y=0.99
    )
    f1_png = f"figura_mapbiomas_paisaje_{ESPECIE_SELECCIONADA.lower()}_{sufijo_escala}.png"
    f1_svg = f"figura_mapbiomas_paisaje_{ESPECIE_SELECCIONADA.lower()}_{sufijo_escala}.svg"
    fig1.savefig(f1_png, dpi=300, bbox_inches="tight", facecolor="white")
    fig1.savefig(f1_svg, bbox_inches="tight", facecolor="white")
    print(f"Figura 12A guardada: '{f1_png}' (300 dpi) y '{f1_svg}' (vectorial).")
    plt.show()

    # ==========================================================================
    # FIGURA 12B: CUADRANTE DE RIESGO + VULNERABILIDAD STRUCTURE x MAPBIOMAS
    # ==========================================================================
    fig2 = plt.figure(figsize=(14.2, 5.8), dpi=140, facecolor="white")
    gs2 = gridspec.GridSpec(1, 2, width_ratios=[1.15, 1.0], wspace=0.28)
    ax_q = fig2.add_subplot(gs2[0, 0])
    ax_v = fig2.add_subplot(gs2[0, 1])

    # Eje X del cuadrante: Fuego (%) si está disponible, o Plantaciones Forestales (%)
    usar_eje_fuego = FUEGO_DISPONIBLE and gdf_mb_loc_ea["fuego_pct_quemado"].notna().any()
    col_x = "fuego_pct_quemado" if usar_eje_fuego else "pct_pf_fin"
    etiq_x = "Superficie quemada acumulada 2013–2025 (%)" if usar_eje_fuego else f"Plantaciones forestales {a2_tbl} (%)"

    col_prio_map = {"Crítica": "#B2182B", "Alta": "#EF6548", "Moderada": "#FDBB84", "Baja": "#2B8CBE", "Sin dato": "#999999"}
    tiene_struct_plot = (df_mb_clusters_gen is not None) and ("cluster_dom" in gdf_mb_loc_ea.columns) and gdf_mb_loc_ea["cluster_dom"].notna().any()
    color_k_map = {f"K{i+1}": COLORES_ACTIVOS[i] for i in range(len(COLORES_ACTIVOS))} if tiene_struct_plot else {}

    # Agrupar localidades con coordenadas (x, y) idénticas para etiquetado limpio sin solapes
    df_q = gdf_mb_loc_ea.copy()
    df_q["_x"] = pd.to_numeric(df_q[col_x], errors="coerce").fillna(0.0)
    df_q["_y"] = pd.to_numeric(df_q["perdida_bn_pct"], errors="coerce").fillna(0.0)
    df_q["_s"] = np.clip(df_q["indice_amenaza"].fillna(20.0), 15.0, 100.0) * 3.4

    # Desplazamiento mínimo determinístico solo si 2+ localidades comparten el mismo punto (modo comunal)
    df_q["_x_plot"] = df_q["_x"].copy()
    df_q["_y_plot"] = df_q["_y"].copy()
    span_x = max(10.0, float(df_q["_x"].max() - df_q["_x"].min()))
    span_y = max(5.0, float(df_q["_y"].max() - df_q["_y"].min()))
    for (_, _), grp_dup in df_q.groupby(["_x", "_y"]):
        if len(grp_dup) > 1:
            angulos = np.linspace(0, 2 * math.pi, len(grp_dup), endpoint=False)
            for ang, idx_d in zip(angulos, grp_dup.index):
                df_q.loc[idx_d, "_x_plot"] += math.cos(ang) * span_x * 0.024
                df_q.loc[idx_d, "_y_plot"] += math.sin(ang) * span_y * 0.032

    colores_pts = [
        color_k_map.get(r["cluster_dom"], "#555555") if tiene_struct_plot else col_prio_map.get(r["prioridad_conservacion"], "#777")
        for _, r in df_q.iterrows()
    ]
    ax_q.scatter(df_q["_x_plot"], df_q["_y_plot"], s=df_q["_s"], c=colores_pts, alpha=0.88,
                 edgecolors="#111111", linewidth=0.9, zorder=4)

    # Límites amplios del cuadrante
    x_min_q = max(-4.0, float(df_q["_x_plot"].min()) - span_x * 0.14)
    x_max_q = float(df_q["_x_plot"].max()) + span_x * 0.22
    y_min_q = max(-2.5, float(df_q["_y_plot"].min()) - span_y * 0.18)
    y_max_q = float(df_q["_y_plot"].max()) + span_y * 0.24
    ax_q.set_xlim(x_min_q, x_max_q)
    ax_q.set_ylim(y_min_q, y_max_q)

    # Líneas de referencia de cuadrante
    x_ref = 15.0 if usar_eje_fuego else 25.0
    y_ref = 10.0
    ax_q.axvline(x_ref, color="#888888", linestyle="--", linewidth=0.9, alpha=0.7)
    ax_q.axhline(y_ref, color="#888888", linestyle="--", linewidth=0.9, alpha=0.7)
    ax_q.text(0.98, 0.97, "Mayor presión\ncompuesta", transform=ax_q.transAxes,
              ha="right", va="top", fontsize=8, color="#8B0000", fontweight="bold",
              bbox=dict(boxstyle="round,pad=0.25", facecolor="#FFF0F0", edgecolor="#D9534F", alpha=0.85))

    # Posicionamiento iterativo antisolapamiento para etiquetas de localidades
    pts_x = df_q["_x_plot"].to_numpy(dtype=float)
    pts_y = df_q["_y_plot"].to_numpy(dtype=float)
    lbl_x = pts_x + (x_max_q - x_min_q) * 0.025
    lbl_y = pts_y + (y_max_q - y_min_q) * 0.030
    dx_min = (x_max_q - x_min_q) * 0.105
    dy_min = (y_max_q - y_min_q) * 0.075
    for _it in range(60):
        for i1 in range(len(lbl_x)):
            for i2 in range(i1 + 1, len(lbl_x)):
                ddx = lbl_x[i2] - lbl_x[i1]
                ddy = lbl_y[i2] - lbl_y[i1]
                if abs(ddx) < dx_min and abs(ddy) < dy_min:
                    push_y = (dy_min - abs(ddy)) * 0.35 * (1.0 if ddy >= 0 else -1.0)
                    push_x = (dx_min - abs(ddx)) * 0.18 * (1.0 if ddx >= 0 else -1.0)
                    lbl_y[i1] -= push_y; lbl_y[i2] += push_y
                    lbl_x[i1] -= push_x; lbl_x[i2] += push_x
            # Mantener dentro del cuadro
            lbl_x[i1] = float(np.clip(lbl_x[i1], x_min_q + (x_max_q - x_min_q) * 0.04, x_max_q - (x_max_q - x_min_q) * 0.08))
            lbl_y[i1] = float(np.clip(lbl_y[i1], y_min_q + (y_max_q - y_min_q) * 0.05, y_max_q - (y_max_q - y_min_q) * 0.07))

    for i_p, (_, r_p) in enumerate(df_q.iterrows()):
        ax_q.annotate(
            str(r_p["Codigo"]),
            xy=(pts_x[i_p], pts_y[i_p]),
            xytext=(lbl_x[i_p], lbl_y[i_p]),
            fontsize=8.3, fontweight="bold", color="#111111", ha="center", va="center",
            path_effects=[withStroke(linewidth=2.4, foreground="white")],
            arrowprops=dict(arrowstyle="-", color="#777777", lw=0.7, shrinkA=2, shrinkB=4),
            zorder=5,
        )

    ax_q.set_xlabel(etiq_x, fontsize=9)
    ax_q.set_ylabel(f"Pérdida de Bosque Nativo {a1_tbl}–{a2_tbl} (%)", fontsize=9)
    ax_q.grid(True, linestyle=":", alpha=0.45)
    ax_q.spines["top"].set_visible(False)
    ax_q.spines["right"].set_visible(False)

    # Leyenda horizontal debajo del eje X (nunca cubre puntos del cuadrante)
    if tiene_struct_plot:
        handles_q = [mpatches.Patch(facecolor=COLORES_ACTIVOS[i], edgecolor="#222", label=f"K{i+1}") for i in range(K_ACTIVO)]
        ax_q.legend(handles=handles_q, title="Grupo STRUCTURE", fontsize=8, title_fontsize=8.2,
                    loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=min(5, K_ACTIVO), frameon=True, edgecolor="#CCCCCC")
        ax_q.set_title(f"A   Cuadrante de riesgo por localidad y clúster genético (K = {K_ACTIVO})", loc="left", fontsize=9.8, fontweight="bold", pad=8)
    else:
        handles_q = [mpatches.Patch(facecolor=c, edgecolor="#222", label=k) for k, c in col_prio_map.items() if k != "Sin dato"]
        ax_q.legend(handles=handles_q, title="Prioridad de conservación", fontsize=8, title_fontsize=8.2,
                    loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=4, frameon=True, edgecolor="#CCCCCC")
        ax_q.set_title("A   Cuadrante de riesgo de amenazas por localidad", loc="left", fontsize=9.8, fontweight="bold", pad=8)

    # --- Panel B: Vulnerabilidad por Clúster Genético (si STRUCTURE activo) o Ranking por Localidad ---
    if tiene_struct_plot:
        df_kc = df_mb_clusters_gen.copy()
        xk = np.arange(len(df_kc))
        w_bar = 0.26
        b1 = ax_v.bar(xk - w_bar, df_kc["indice_amenaza_pond"], width=w_bar, color=df_kc["Color"], edgecolor="#111", linewidth=0.7)
        b2 = ax_v.bar(xk,         df_kc["pct_plantaciones_pond"], width=w_bar, color=MB_COLORES_OFICIALES["Plantaciones Forestales"], edgecolor="#111", linewidth=0.6, alpha=0.85)
        if usar_eje_fuego:
            b3 = ax_v.bar(xk + w_bar, df_kc["fuego_pct_quemado_pond"], width=w_bar, color="#D95F02", edgecolor="#111", linewidth=0.6, alpha=0.9)
            etiq_b3, col_b3 = "Área quemada (%)", "#D95F02"
        else:
            b3 = ax_v.bar(xk + w_bar, df_kc["perdida_bn_pct_pond"], width=w_bar, color="#B2182B", edgecolor="#111", linewidth=0.6, alpha=0.85)
            etiq_b3, col_b3 = "Pérdida BN (%)", "#B2182B"
        for bar in b1:
            h = bar.get_height()
            ax_v.text(bar.get_x() + bar.get_width() / 2.0, h + 1.2, f"{h:.0f}", ha="center", va="bottom", fontsize=8.2, fontweight="bold")
        # Etiquetas con salto de línea cada 2 códigos para evitar choques horizontales
        etiquetas_k = []
        for _, rk in df_kc.iterrows():
            cods = [c.strip() for c in str(rk["localidades_dom"]).split(",") if c.strip()]
            lineas_c = [", ".join(cods[j:j+2]) for j in range(0, len(cods), 2)]
            etiquetas_k.append(f"{rk['Cluster']}\n(" + ",\n".join(lineas_c) + ")")
        ax_v.set_xticks(xk)
        ax_v.set_xticklabels(etiquetas_k, fontsize=7.8, fontweight="bold")
        ax_v.set_ylabel("Valor ponderado por ancestría Q (% / 0–100)", fontsize=8.5)
        y_top_v = max(60.0, float(df_kc[["indice_amenaza_pond", "pct_plantaciones_pond"]].max().max()) + 18.0)
        ax_v.set_ylim(0, min(115.0, max(y_top_v, 55.0)))
        ax_v.set_title("B   Vulnerabilidad ponderada por unidad genética (STRUCTURE × MapBiomas)", loc="left", fontsize=9.8, fontweight="bold", pad=8)
        handles_v = [
            mpatches.Patch(facecolor="#555555", edgecolor="#111", label="Índice Amenaza 0–100 (color del clúster)"),
            mpatches.Patch(facecolor=MB_COLORES_OFICIALES["Plantaciones Forestales"], edgecolor="#111", label="Plantaciones forestales (%)"),
            mpatches.Patch(facecolor=col_b3, edgecolor="#111", label=etiq_b3),
        ]
        ax_v.legend(handles=handles_v, loc="upper right", fontsize=7.6, frameon=True, framealpha=0.95, edgecolor="#CCCCCC")
    else:
        df_rk = gdf_mb_loc_ea.sort_values("indice_amenaza", ascending=True).reset_index(drop=True)
        yr = np.arange(len(df_rk))
        cols_rk = [col_prio_map.get(p, "#777") for p in df_rk["prioridad_conservacion"]]
        ax_v.barh(yr, df_rk["indice_amenaza"], color=cols_rk, edgecolor="#222", linewidth=0.6, height=0.68)
        for i_r, r_r in df_rk.iterrows():
            ax_v.text(r_r["indice_amenaza"] + 1.2, i_r, f"{r_r['indice_amenaza']:.1f} ({r_r['prioridad_conservacion']})",
                      va="center", fontsize=8, fontweight="bold")
        ax_v.set_yticks(yr)
        ax_v.set_yticklabels([f"{r['Codigo']} — {r['Referencia']}" for _, r in df_rk.iterrows()], fontsize=8.2)
        ax_v.set_xlim(0, 118)
        ax_v.set_xlabel("Índice Compuesto de Prioridad de Amenaza (0–100)", fontsize=8.5)
        ax_v.set_title("B   Ranking de prioridad de conservación por localidad", loc="left", fontsize=9.8, fontweight="bold", pad=8)

    ax_v.grid(axis="y" if tiene_struct_plot else "x", linestyle=":", alpha=0.45)
    ax_v.spines["top"].set_visible(False)
    ax_v.spines["right"].set_visible(False)

    f2_png = f"figura_mapbiomas_riesgo_genetica_{ESPECIE_SELECCIONADA.lower()}_{sufijo_escala}.png"
    f2_svg = f"figura_mapbiomas_riesgo_genetica_{ESPECIE_SELECCIONADA.lower()}_{sufijo_escala}.svg"
    fig2.savefig(f2_png, dpi=300, bbox_inches="tight", facecolor="white")
    fig2.savefig(f2_svg, bbox_inches="tight", facecolor="white")
    print(f"Figura 12B guardada: '{f2_png}' (300 dpi) y '{f2_svg}' (vectorial).")
    plt.show()

### 12.3. Mapa Interactivo de Amenazas (`Folium`): MapBiomas Chile, AOO y Estructura Genética

Explora espacialmente la exposición de cada localidad y comuna haciendo clic sobre los buffers de paisaje (`7,5 km`) o los polígonos comunales para inspeccionar el desglose histórico `1999` vs. `2024`, la tasa anual de cambio de Bosque Nativo ($r$), las métricas de incendios (cuando están cargadas) y el clúster genético dominante de `STRUCTURE`.

In [ ]:
if not globals().get("MAPBIOMAS_LISTO", False):
    print("Ejecuta primero la celda 12.1 con ACTIVAR_MODULO_MAPBIOMAS = True.")
else:
    m_mb = folium.Map(location=[lat_centro, lon_centro], zoom_start=8, tiles=None)

    folium.TileLayer(
        tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
        attr="&copy; OpenStreetMap contributors &copy; CARTO",
        name="Mapa Claro (CartoDB Positron)",
        show=True,
    ).add_to(m_mb)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}",
        attr="Tiles &copy; Esri",
        name="Topográfico (Esri World Topo)",
        show=False,
    ).add_to(m_mb)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Tiles &copy; Esri",
        name="Satélite (Esri World Imagery)",
        show=False,
    ).add_to(m_mb)

    # 1. Coropleta comunal de pérdida de Bosque Nativo (1999-2024)
    gdf_mb_comunas_ea.to_crs("EPSG:4326").explore(
        m=m_mb,
        name=f"Comunas — Pérdida Bosque Nativo {a1_tbl}–{a2_tbl} (%)",
        column="perdida_bn_pct",
        cmap="YlOrRd",
        tooltip=["comuna", "region", "pct_bn_fin", "delta_bn_pct", "pct_pf_fin", "vector_perdida_dom", "prioridad_conservacion"],
        legend=True,
        style_kwds={"fillOpacity": 0.35, "weight": 1.1, "color": "#444444"},
        show=True,
    )

    # 2. EOO Oficial y Celdas AOO enriquecidas con amenazas MapBiomas
    gdf_eoo_oficial_ea.to_crs("EPSG:4326").explore(
        m=m_mb,
        name=f"EOO Oficial B1 ({eoo_oficial_km2:,.0f} km²)",
        color="darkorange",
        style_kwds={"fill": False, "weight": 2.2},
        tooltip=["especie", "escala", "eoo_km2"],
        show=True,
    )
    aoo_tips = [c for c in ["celda_id", "comuna", "pct_bn_fin", "pct_pf_fin", "delta_bn_pct", "fuego_pct_quemado"] if c in gdf_mb_aoo_ea.columns]
    gdf_mb_aoo_ea.to_crs("EPSG:4326").explore(
        m=m_mb,
        name=f"AOO 2×2 km + Amenazas MapBiomas",
        column="pct_pf_fin",
        cmap="Oranges",
        tooltip=aoo_tips,
        legend=False,
        style_kwds={"fillOpacity": 0.55, "weight": 0.8, "color": "#662506"},
        show=False,
    )

    # 3. Buffers de paisaje (7,5 km) por localidad con fichas emergentes enriquecidas
    col_prio_hex = {"Crítica": "#B2182B", "Alta": "#EF6548", "Moderada": "#FDBB84", "Baja": "#2B8CBE", "Sin dato": "#888888"}
    fg_buf = folium.FeatureGroup(name=f"Amenazas por Localidad (buffers {RADIO_BUFFER_PAISAJE_M/1000:.1f} km)", show=True)
    gdf_buf_wgs = gdf_mb_loc_ea.to_crs("EPSG:4326")

    for _, row in gdf_buf_wgs.iterrows():
        c_p = col_prio_hex.get(row["prioridad_conservacion"], "#666666")
        fuego_html = ""
        if FUEGO_DISPONIBLE and pd.notna(row.get("fuego_pct_quemado")):
            ult_txt = f"{int(row['fuego_ultimo_anio'])}" if pd.notna(row.get("fuego_ultimo_anio")) else "s/d"
            fmax_txt = f"{int(row['fuego_freq_max'])}×" if pd.notna(row.get("fuego_freq_max")) else "s/d"
            fuego_html = (
                f'<hr style="margin:4px 0;">'
                f'<b>MapBiomas Fuego (2013–2025):</b><br>'
                f'Área quemada: <b>{row["fuego_pct_quemado"]:.1f}%</b> | Máx. recurrencia: <b>{fmax_txt}</b> | Último año: <b>{ult_txt}</b>'
            )
        gen_html = ""
        if pd.notna(row.get("cluster_dom")):
            gen_html = (
                f'<hr style="margin:4px 0;">'
                f'<b>STRUCTURE (K={K_ACTIVO}):</b> Cluster dominante <b>{row["cluster_dom"]}</b> '
                f'(Q<sub>max</sub>={row["q_max"]:.2f}, {row["estado_mezcla"]})'
            )
        popup_mb = (
            f'<div style="font-family:sans-serif;font-size:12px;min-width:235px;">'
            f'<div style="display:flex;justify-content:space-between;align-items:center;">'
            f'<b>{row["Codigo"]} — {row["Referencia"]}</b>'
            f'<span style="background:{c_p};color:#fff;padding:1px 6px;border-radius:3px;font-size:10px;font-weight:700;">'
            f'{row["prioridad_conservacion"].upper()} ({row["indice_amenaza"]:.0f}/100)</span></div>'
            f'<div style="color:#555;font-size:11px;margin-bottom:4px;">Comuna: {row["comuna"]} | {row["escala_lulc"]}</div>'
            f'<table style="width:100%;border-collapse:collapse;font-size:11.5px;">'
            f'<tr><td><span style="display:inline-block;width:9px;height:9px;background:#1F8D49;margin-right:4px;"></span>Bosque Nativo</td>'
            f'<td style="text-align:right;">{row["pct_bn_ini"]:.1f}% → <b>{row["pct_bn_fin"]:.1f}%</b> ({row["delta_bn_pct"]:+.1f}%)</td></tr>'
            f'<tr><td><span style="display:inline-block;width:9px;height:9px;background:#7A5900;margin-right:4px;"></span>Plantaciones</td>'
            f'<td style="text-align:right;">{row["pct_pf_ini"]:.1f}% → <b>{row["pct_pf_fin"]:.1f}%</b></td></tr>'
            f'<tr><td><span style="display:inline-block;width:9px;height:9px;background:#A89358;margin-right:4px;"></span>Matorral</td>'
            f'<td style="text-align:right;"><b>{row["pct_matorral_fin"]:.1f}%</b></td></tr>'
            f'<tr><td><span style="display:inline-block;width:9px;height:9px;background:#E974ED;margin-right:4px;"></span>Agro / Pasturas</td>'
            f'<td style="text-align:right;"><b>{row["pct_agro_fin"]:.1f}%</b></td></tr>'
            f'<tr><td><span style="display:inline-block;width:9px;height:9px;background:#D4271E;margin-right:4px;"></span>Infraestructura</td>'
            f'<td style="text-align:right;"><b>{row["pct_infra_fin"]:.1f}%</b></td></tr>'
            f'</table>'
            f'<div style="margin-top:4px;font-size:11px;">Tasa Puyravaud BN: <b>{row["tasa_puyravaud_r"]:+.3f}%/año</b> | Vector dom.: <b>{row["vector_perdida_dom"]}</b></div>'
            f'{fuego_html}{gen_html}'
            f'</div>'
        )
        folium.GeoJson(
            row.geometry.__geo_interface__,
            style_function=lambda feat, col=c_p: {"fillColor": col, "color": "#111111", "weight": 1.6, "fillOpacity": 0.42},
            tooltip=f"{row['Codigo']} ({row['comuna']}) — I_amenaza={row['indice_amenaza']:.0f}/100 ({row['prioridad_conservacion']})",
            popup=folium.Popup(popup_mb, max_width=310),
        ).add_to(fg_buf)
    fg_buf.add_to(m_mb)

    # 4. Superponer tortas SVG de STRUCTURE si el Módulo 11 está activo
    if globals().get("STRUCTURE_LISTO", False) and globals().get("gdf_struct_pop_wgs84") is not None:
        fg_pies_mb = folium.FeatureGroup(name=f"STRUCTURE Tortas por Localidad (K = {K_ACTIVO})", show=True)
        for _, row in gdf_struct_pop_wgs84.iterrows():
            q_vals = [float(row[c]) for c in K_COLS_ACTIVOS]
            svg_html = construir_svg_torta(q_vals, COLORES_ACTIVOS, tamano=34)
            icon_html = (
                f'<div style="display:flex;flex-direction:column;align-items:center;">{svg_html}'
                f'<span style="font-size:9.5px;font-weight:700;color:#111;background:rgba(255,255,255,0.88);'
                f'padding:0px 3px;border-radius:3px;border:1px solid #888;margin-top:-2px;">{row["Codigo"]}</span></div>'
            )
            folium.Marker(
                location=[float(row["Lat"]), float(row["Lon"])],
                icon=folium.DivIcon(html=icon_html, icon_size=(40, 48), icon_anchor=(20, 17)),
                tooltip=f"{row['Codigo']} — Cluster {row['cluster_dom']} ({row['q_max']*100:.0f}%)",
            ).add_to(fg_pies_mb)
        fg_pies_mb.add_to(m_mb)

    # Leyenda compacta de prioridad de amenaza
    ley_mb = (
        '<div style="position:fixed;bottom:24px;left:24px;z-index:9999;background:rgba(255,255,255,0.94);'
        'padding:8px 12px;border:1px solid #555;border-radius:5px;font-family:sans-serif;font-size:11.5px;'
        'box-shadow:0 2px 6px rgba(0,0,0,0.2);">'
        f'<b>Prioridad Amenaza MapBiomas</b><br>'
        '<span style="display:inline-block;width:11px;height:11px;background:#B2182B;margin-right:5px;"></span>Crítica (≥ 70)<br>'
        '<span style="display:inline-block;width:11px;height:11px;background:#EF6548;margin-right:5px;"></span>Alta (50–70)<br>'
        '<span style="display:inline-block;width:11px;height:11px;background:#FDBB84;margin-right:5px;"></span>Moderada (30–50)<br>'
        '<span style="display:inline-block;width:11px;height:11px;background:#2B8CBE;margin-right:5px;"></span>Baja (&lt; 30)'
        '<div style="margin-top:4px;font-size:10px;color:#555;">Fuente: MapBiomas Chile Col. 2.0 (CC BY 4.0)</div></div>'
    )
    m_mb.get_root().html.add_child(folium.Element(ley_mb))
    folium.LayerControl(collapsed=False).add_to(m_mb)

    mapa_mb_html = f"mapa_amenazas_mapbiomas_{ESPECIE_SELECCIONADA.lower()}_{sufijo_escala}.html"
    m_mb.save(mapa_mb_html)
    print(f"Mapa interactivo de amenazas MapBiomas guardado: '{mapa_mb_html}'")
    display(m_mb)

---

## Referencias Bibliográficas Principales

* **Chao, A., & Jost, L. (2012).** Coverage-based rarefaction and extrapolation: standardizing samples by completeness rather than size. *Ecology*, 93(12), 2533–2547. https://doi.org/10.1890/11-1952.1
* **Evanno, G., Regnaut, S., & Goudet, J. (2005).** Detecting the number of clusters of individuals using the software STRUCTURE: a simulation study. *Molecular Ecology*, 14(8), 2611–2620. https://doi.org/10.1111/j.1365-294X.2005.02553.x
* **IUCN Standards and Petitions Committee (2024).** *Guidelines for Using the IUCN Red List Categories and Criteria*. Version 16. https://www.iucnredlist.org/documents/RedListGuidelines.pdf
* **IUCN (2012).** *Guidelines for Application of IUCN Red List Criteria at Regional and National Levels*. Version 4.0. Gland, Switzerland and Cambridge, UK: IUCN.
* **MapBiomas Chile (2026).** *Proyecto MapBiomas Chile — Colección 2.0 de la Serie Anual de Mapas de Cobertura y Uso del Suelo de Chile (1999–2024) y Colección 1.0 de MapBiomas Fuego Chile (2013–2025)*. Datos de acceso abierto bajo licencia Creative Commons CC BY 4.0. https://chile.mapbiomas.org/
* **MMA — Ministerio del Medio Ambiente de Chile (2011).** Decreto Supremo N° 29/2011: *Reglamento para la Clasificación de Especies Silvestres según Estado de Conservación (RCE)*.
* **MMA — Ministerio del Medio Ambiente de Chile (2020).** Decreto Supremo N° 42/2020: *Plan de Recuperación, Conservación y Gestión del Ruil (Nothofagus alessandrii)*.
* **MMA – FAO – GEF (2022).** *Plan de Recuperación, Conservación y Gestión del Queule (Gomortega keule)*. Proyecto Iniciativa Conservación de Especies Amenazadas, Chile (pp. 5–7; LEP-UdeC, 2019).
* **Pereira, A. G., Ferreira, M. F. S., da Silveira, T. C., Soler-Guilhen, J. H., Canal, G. B., Alves, L. B., de Almeida, F. A. N., Gaiotto, F. A., & Ferreira, A. (2022).** Patterns of genetic diversity and structure of a threatened palm species (*Euterpe edulis* Arecaceae) from the Brazilian Atlantic Forest. *Heredity*, 129, 161–168. https://doi.org/10.1038/s41437-022-00549-7 *(referencia del diseño de figura compuesta A + B + C)*.
* **Pritchard, J. K., Stephens, M., & Donnelly, P. (2000).** Inference of population structure using multilocus genotype data. *Genetics*, 155(2), 945–959. https://doi.org/10.1093/genetics/155.2.945
* **Puyravaud, J.-P. (2003).** Standardizing the calculation of the annual rate of deforestation. *Forest Ecology and Management*, 177(1–3), 593–596. https://doi.org/10.1016/S0378-1127(02)00335-3